In [1]:
"""GPR Campaign Ploemeur — 06-06-2016.  Equivalent to seq06.m."""
from pathlib import Path
import sys
import importlib
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from gdp.data_io import load_mala
from gdp.preprocessing.filtering import filter_data, remove_mean
from gdp.preprocessing.gain import linear_gain
from gdp.preprocessing.image_processing import remove_svd
from gdp.preprocessing.trace_ops import align_traces

sys.path.insert(0, str(Path.cwd()))
import helper_functions.KirchhoffPylopsZeroOffset as KirchhoffPylopsZeroOffset
importlib.reload(KirchhoffPylopsZeroOffset)
from pylops.utils.wavelets import ricker
from helper_functions.migration import (
    gazdag_migration, write_backprop_files,
    lowpass_filter_excitation, dispersion_limited_cutoff,
    apply_3d_to_2d_correction,
)

from helper_functions.figures import save_fig   # standard thesis-figure export


# Loading in data from 6 June 2016

In [2]:
DATA = Path.cwd() / 'fielddata' / 'raw_data' / '060616'
OUT_DIR = Path.cwd() / 'fielddata' / 'output'

# Process Reference

In [3]:
runs = list(range(0, 6)) + list(range(7, 39))

# prof (run 0) is the pre-injection reference used for preprocessing.
# Its migration is zero (self-subtraction) so it is excluded from data_runs.
ref_run = runs[0]   # = 0 → loads prof.rd3 / prof.rad
data_runs = runs[1:]  # run 0 migration = zero (reference minus itself); exclude

_prof_name = lambda n: 'prof' if n == 0 else f'prof{n}'
ref_raw, info = load_mala(str(DATA / _prof_name(ref_run)), return_object=False)
sf = info['frequency (GHz)']
n_traces = ref_raw.shape[1]
samples = ref_raw.shape[0]

# Preprocess reference
ref_bp = filter_data(ref_raw, fq=(0.02, 0.2), sfreq=sf, btype='bandpass')
ref_dc, _ = remove_mean(ref_bp, 299, 517)

ref_aligned, _, _ = align_traces(ref_dc, ref_dc, upsample=5, normalize=False, align_reference=True)

ref_svd, _ = remove_svd(ref_aligned, low_s=0, high_s=1)

t = np.arange(1, samples + 1) / sf
v = 0.10 # [m/ns]
wavelength = v / 0.1 # [m] — wavelength at 100 MHz
dL = 0.05 # [m]
max_d = 85.0
sc = 1.2
rad_cut = 300
depth = np.linspace(max_d, max_d - n_traces * dL, n_traces)
radius = np.linspace(0, v * 450 / 2, samples)

OUT_DIR.mkdir(exist_ok=True)
(OUT_DIR / 'processed').mkdir(exist_ok=True)

# Migration setup
t_mig  = t[:rad_cut]
f0_mig = 0.1                                        # centre frequency [GHz] — adjust to antenna

# Ricker wavelet (matches PylopsKirchoffMigration convention)
_period = 1.0 / f0_mig
_n_wav  = int(np.ceil(6 * _period / (t_mig[1] - t_mig[0])))
if _n_wav % 2 == 0:
    _n_wav += 1
wav_mig, _, wcenter_mig = ricker(t_mig[:_n_wav], f0=f0_mig)

# Image axes: x = radial distance from borehole [m], z spans borehole depth range
x_img = np.linspace(0, v * t_mig[-1] / 2, rad_cut)

(OUT_DIR / 'migrated').mkdir(exist_ok=True)

# Back-propagation gprMax grid -- lambda/20 at f0_mig (~20 cells/wavelength)
bp_dx           = v / (f0_mig * 20)                      # grid spacing [m]
bp_pml          = 15                                      # PML cells
bp_src_y        = (bp_pml + 1) * bp_dx                   # source just inside PML on borehole side [m]
bp_domain_y     = float(x_img[-1]) + 2 * bp_pml * bp_dx  # radial extent + PML padding both sides [m]
bp_edge_exclude = 5                                       # zero outermost N sources each side before filtering

# ── Process and migrate reference profile (prof.rd3, absolute) ──────────────
n_ref       = n_traces
z_bh_ref    = depth[:n_ref]
x_bh_ref    = np.arange(n_ref) * dL

ref_gain, _ = linear_gain(ref_svd, t)
Dt_ref      = ref_gain[:rad_cut, :].T   # (n_ref, rad_cut)

limit_ref = max(sc * np.max(np.abs(Dt_ref)), 1.0)
fig_r, ax_r = plt.subplots(figsize=(8, 6))
ax_r.imshow(Dt_ref, aspect='auto', cmap='seismic',
            extent=[radius[0], radius[rad_cut - 1], depth[-1], depth[0]],
            vmin=-limit_ref, vmax=limit_ref)
ax_r.invert_yaxis()
ax_r.set_xlabel('Radial distance from B2 (m)')
ax_r.set_ylabel('Depth from top of B1 (m)')
fig_r.savefig(OUT_DIR / 'processed' / 'prof_0.png', dpi=150)
plt.close(fig_r)

# Kirchhoff reference migration
z_img_ref = np.linspace(z_bh_ref.max(), z_bh_ref.min(), n_ref)
recs_ref  = np.vstack((np.zeros(n_ref), z_bh_ref))
K_ref = KirchhoffPylopsZeroOffset.Kirchhoff(
    z=z_img_ref, x=x_img, t=t_mig,
    srcs=recs_ref, recs=recs_ref,
    vel=v, wav=wav_mig, wavcenter=wcenter_mig,
    mode='analytic', dynamic=False,
)
ref_m_kir = (K_ref.H @ Dt_ref.flatten()).reshape(len(x_img), len(z_img_ref)).T

lim_kir_ref = max(sc * np.max(np.abs(ref_m_kir)), 1.0)
fig_kr, ax_kr = plt.subplots(figsize=(8, 6))
ax_kr.imshow(ref_m_kir, aspect='equal', cmap='seismic',
             extent=[x_img[0], x_img[-1], z_img_ref[-1], z_img_ref[0]],
             vmin=-lim_kir_ref, vmax=lim_kir_ref)
ax_kr.invert_yaxis()
ax_kr.set_xlabel('Radial distance from borehole (m)')
ax_kr.set_ylabel('Depth (m)')
fig_kr.savefig(OUT_DIR / 'migrated' / 'kirchhoff_0.png', dpi=150)
plt.close(fig_kr)
np.save(OUT_DIR / 'migrated' / 'kirchhoff_0.npy', ref_m_kir)

# Kirchhoff-BP reference migration (delta wavelet)
wav_delta_ref = np.zeros_like(wav_mig); wav_delta_ref[wcenter_mig] = 1.0
K_bp_ref = KirchhoffPylopsZeroOffset.Kirchhoff(
    z=z_img_ref, x=x_img, t=t_mig,
    srcs=recs_ref, recs=recs_ref,
    vel=v, wav=wav_delta_ref, wavcenter=wcenter_mig,
    mode='analytic', dynamic=False,
)
ref_m_kir_bp = (K_bp_ref.H @ Dt_ref.flatten()).reshape(len(x_img), len(z_img_ref)).T
lim_kir_bp_ref = max(sc * np.max(np.abs(ref_m_kir_bp)), 1.0)
fig_kbpr, ax_kbpr = plt.subplots(figsize=(8, 6))
ax_kbpr.imshow(ref_m_kir_bp, aspect='equal', cmap='seismic',
               extent=[x_img[0], x_img[-1], z_img_ref[-1], z_img_ref[0]],
               vmin=-lim_kir_bp_ref, vmax=lim_kir_bp_ref)
ax_kbpr.invert_yaxis()
ax_kbpr.set_xlabel('Radial distance from borehole (m)')
ax_kbpr.set_ylabel('Depth (m)')
fig_kbpr.savefig(OUT_DIR / 'migrated' / 'kirchhoff_bp_0.png', dpi=150)
plt.close(fig_kbpr)
np.save(OUT_DIR / 'migrated' / 'kirchhoff_bp_0.npy', ref_m_kir_bp)

# Gazdag reference migration
ref_m_gaz = gazdag_migration(ref_gain[:rad_cut, :], x_bh_ref, t_mig, x_img, v)

lim_gaz_ref = max(sc * np.max(np.abs(ref_m_gaz)), 1.0)
fig_gr, ax_gr = plt.subplots(figsize=(8, 6))
ax_gr.imshow(ref_m_gaz.T, aspect='equal', cmap='seismic',
             extent=[x_img[0], x_img[-1], z_bh_ref[-1], z_bh_ref[0]],
             vmin=-lim_gaz_ref, vmax=lim_gaz_ref)
ax_gr.invert_yaxis()
ax_gr.set_xlabel('Radial distance from borehole (m)')
ax_gr.set_ylabel('Depth (m)')
fig_gr.savefig(OUT_DIR / 'migrated' / 'gazdag_0.png', dpi=150)
plt.close(fig_gr)
np.save(OUT_DIR / 'migrated' / 'gazdag_0.npy', ref_m_gaz.T)

print('Reference migrations saved.')

  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
Reference migrations saved.


# Process other profiles

In [ ]:
data_runs_shortened = [1,3,4,8,9,20,21,38]

In [ ]:
for run in data_runs_shortened: #data_runs:
    data, _ = load_mala(str(DATA / _prof_name(run)), return_object=False)
    n = min(data.shape[1], n_traces)

    d_bp = filter_data(data[:, :n], fq=(0.02, 0.2), sfreq=sf, btype='bandpass')
    d_dc, _ = remove_mean(d_bp, 299, 517)

    d_aligned, _, _ = align_traces(d_dc, ref_aligned[:, :n], upsample=5, normalize=True, align_reference=False)

    d_svd, _ = remove_svd(d_aligned, low_s=0, high_s=1)
    d_gain, _ = linear_gain(d_svd, t)

    # B-scan — Dt[0] = deepest trace (depth[0] = 85 m)
    Dt    = d_gain[:rad_cut, :].T        # (n, rad_cut) = (n_rec, n_t)
    z_bh  = depth[:n]                   # receiver depths [m]; z_bh[0]=85 (deep) → z_bh[-1]≈0
    x_bh  = np.arange(n) * dL          # relative along-borehole positions [m]
    dt_bh = 1.0 / sf                    # time step [ns]

    limit = max(sc * np.max(np.abs(Dt)), 1.0)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(Dt, aspect='auto', cmap='seismic',
              extent=[radius[0], radius[rad_cut - 1], depth[-1], depth[0]],
              vmin=-limit, vmax=limit)
    ax.invert_yaxis()
    ax.set_xlabel('Radial distance from B2 (m)')
    ax.set_ylabel('Depth from top of B1 (m)')
    fig.savefig(OUT_DIR / 'processed' / f'prof_{run}.png', dpi=150)
    plt.close(fig)

    # --- Kirchhoff migration ---
    # z_img decreasing (85→0) so row 0 of m_kir = 85 m
    z_img   = np.linspace(z_bh.max(), z_bh.min(), n)
    recs_bh = np.vstack((np.zeros(n), z_bh))       # (x=0, z=depth) — left-wall geometry

    K = KirchhoffPylopsZeroOffset.Kirchhoff(
        z=z_img, x=x_img, t=t_mig,
        srcs=recs_bh, recs=recs_bh,
        vel=v, wav=wav_mig, wavcenter=wcenter_mig,
        mode='analytic', dynamic=False,
    )
    m_kir = (K.H @ Dt.flatten()).reshape(len(x_img), len(z_img)).T   # (n_depth, n_radial)
    m_kir -= ref_m_kir[:len(z_img), :]          # subtract reference migration

    lim_kir = max(sc * np.max(np.abs(m_kir)), 1.0)
    fig_k, ax_k = plt.subplots(figsize=(8, 6))
    ax_k.imshow(m_kir, aspect='equal', cmap='seismic',
                extent=[x_img[0], x_img[-1], z_img[-1], z_img[0]],
                vmin=-lim_kir, vmax=lim_kir)
    ax_k.invert_yaxis()
    ax_k.set_xlabel('Radial distance from borehole (m)')
    ax_k.set_ylabel('Depth (m)')
    fig_k.savefig(OUT_DIR / 'migrated' / f'kirchhoff_{run}.png', dpi=150)
    plt.close(fig_k)
    np.save(OUT_DIR / 'migrated' / f'kirchhoff_{run}.npy', m_kir)

    # --- Kirchhoff migration (delta wavelet — pure back-projection) ---
    # Using a spike instead of wav_mig removes the matched-filter / autocorrelation
    # step from K.H, so the PSF collapses from wac-shaped (4–5 extrema) to
    # wav-shaped (3 extrema), matching Gazdag's t=0 imaging condition.
    
    wav_delta = np.zeros_like(wav_mig); wav_delta[wcenter_mig] = 1.0
    K_bp = KirchhoffPylopsZeroOffset.Kirchhoff(
        z=z_img, x=x_img, t=t_mig,
        srcs=recs_bh, recs=recs_bh,
        vel=v, wav=wav_delta, wavcenter=wcenter_mig,
        mode='analytic', dynamic=False,
    )
    m_kir_bp = (K_bp.H @ Dt.flatten()).reshape(len(x_img), len(z_img)).T
    m_kir_bp -= ref_m_kir_bp[:len(z_img), :]   # subtract reference migration
    lim_kir_bp = max(sc * np.max(np.abs(m_kir_bp)), 1.0)
    fig_kbp, ax_kbp = plt.subplots(figsize=(8, 6))
    ax_kbp.imshow(m_kir_bp, aspect='equal', cmap='seismic',
                  extent=[x_img[0], x_img[-1], z_img[-1], z_img[0]],
                  vmin=-lim_kir_bp, vmax=lim_kir_bp)
    ax_kbp.invert_yaxis()
    ax_kbp.set_xlabel('Radial distance from borehole (m)')
    ax_kbp.set_ylabel('Depth (m)')
    fig_kbp.savefig(OUT_DIR / 'migrated' / f'kirchhoff_bp_{run}.png', dpi=150)
    plt.close(fig_kbp)
    np.save(OUT_DIR / 'migrated' / f'kirchhoff_bp_{run}.npy', m_kir_bp)


    # --- Gazdag migration ---
    # Rotate: borehole depth → x (along-track), radial distance → z (continuation direction)
    # d_gain[:rad_cut,:] is (n_t, n_x) as required; v_mig = v/2 applied internally
    m_gaz = gazdag_migration(d_gain[:rad_cut, :], x_bh, t_mig, x_img, v)
    m_gaz -= ref_m_gaz[:, :n]                   # subtract reference migration
    # m_gaz: (n_radial, n_depth) → .T gives (n_depth, n_radial); row 0 = z_bh[0] = 85 m
    lim_gaz = max(sc * np.max(np.abs(m_gaz)), 1.0)
    fig_g, ax_g = plt.subplots(figsize=(8, 6))
    ax_g.imshow(m_gaz.T, aspect='equal', cmap='seismic',
                extent=[x_img[0], x_img[-1], z_bh[-1], z_bh[0]],
                vmin=-lim_gaz, vmax=lim_gaz)
    ax_g.invert_yaxis()
    ax_g.set_xlabel('Radial distance from borehole (m)')
    ax_g.set_ylabel('Depth (m)')
    fig_g.savefig(OUT_DIR / 'migrated' / f'gazdag_{run}.png', dpi=150)
    plt.close(fig_g)
    np.save(OUT_DIR / 'migrated' / f'gazdag_{run}.npy', m_gaz.T)

    # --- Back propagation files (peak-normalised and sign-bit) ---
    eps_r    = (0.299792458 / v) ** 2     # permittivity from velocity (c in m/ns)
    t0_ns    = 299.0 / sf                 # direct-wave end, from remove_mean window start
    eps_r_half = 4.0 * eps_r
    f_cut_hz   = dispersion_limited_cutoff(eps_r_half, bp_dx)

    # Back-propagate the time-lapse DIFFERENCE (current profile − reference), not the
    # absolute B-scan: `write_backprop_files` peak-normalises each trace (a nonlinear
    # step), so subtracting two independently-normalised gprMax runs afterwards (see
    # the differencing cells below) would not recover a physically meaningful
    # difference field. Differencing here instead makes gprMax back-propagate the
    # true anomaly directly in a single run, matching how Kirchhoff/Gazdag isolate it
    # via `-= ref_m_kir` / `-= ref_m_gaz` — just done before injection here instead of
    # after migration. apply_3d_to_2d_correction is linear, so correcting the
    # difference is equivalent to differencing two corrected profiles, just cheaper.
    #
    # Uses d_svd/ref_svd (pre-`linear_gain`), NOT Dt: `linear_gain` multiplies by t^1
    # as a generic attenuation compensation (see gdp.preprocessing.gain), not a
    # spreading-model-specific correction. Stacking the sqrt(t) gain below on top of
    # that would double-count range compensation (net t^1 * t^0.5 = t^1.5 instead of
    # the correct t^0.5), blowing up amplitude at long travel times/large radii.
    Dt_diff_bp = d_svd[:rad_cut, :].T - ref_svd[:rad_cut, :n].T   # (n, rad_cut)
    Dt_2d = apply_3d_to_2d_correction(Dt_diff_bp, dt=dt_bh * 1e-9, velocity=v * 1e9,
                                       time_zero_idx=299)

    # ── Back-propagation pre-processing ──────────────────────────────────────────
    # Applied in this order to minimise spectral leakage before gprMax injection:
    #   1. 3D → 2D Green's function correction (sqrt(r) gain + 1/sqrt(w)*e^(i*pi/4))
    #   2. Spatial Tukey taper  (axis 0, receivers)   alpha=0.30 → 15 % each side
    #   3. Temporal Tukey taper (axis 1, time)        alpha=0.10 → 5 % each side
    #   4. f-kz dip filter      zero evanescent bins (|kz| > f / v_mig)
    #   (RMS normalisation removed: amplifies low-SNR traces → worsens near-source noise)
    from scipy.signal.windows import tukey as _tukey_sp

    # 2. Spatial taper
    _sp_tap    = _tukey_sp(n, alpha=0.30)[:, np.newaxis]              # (n, 1)
    Dt_tapered = Dt_2d * _sp_tap

    # 3. Temporal taper
    _t_tap     = _tukey_sp(Dt_tapered.shape[1], alpha=0.10)[np.newaxis, :]  # (1, n_t)
    Dt_tapered = Dt_tapered * _t_tap

    # 4. f-kz dip filter: remove evanescent energy (|kz| > f / v_mig)
    _v_mig_bp = v / 2.0
    _D_fk     = np.fft.fft(np.fft.rfft(Dt_tapered, axis=1), axis=0)  # (n, n_f)
    _freq_bp  = np.fft.rfftfreq(Dt_tapered.shape[1], d=dt_bh)         # [GHz]
    _kz_bp    = np.fft.fftfreq(Dt_tapered.shape[0], d=dL)              # [1/m]
    _evan_bp  = np.abs(_kz_bp[:, None]) > np.abs(_freq_bp[None, :]) / _v_mig_bp
    _D_fk[_evan_bp] = 0.0
    Dt_tapered = np.real(np.fft.irfft(np.fft.ifft(_D_fk, axis=0),
                                       n=Dt_tapered.shape[1], axis=1))

    _bp_common = dict(
        tapered_ntr_nt=Dt_tapered, dt_ns=dt_bh,
        x_midpoints=z_bh, t0_ns=t0_ns,
        eps_r=eps_r, v_ice=v,
        dx=bp_dx, domain_y=bp_domain_y, src_y=bp_src_y, pml_cells=bp_pml,
    )

    # Peak-normalised version
    write_backprop_files(OUT_DIR, label=f'prof_{run}', slug=f'prof_{run}',
                         sign_bit=False, **_bp_common)
    exc_file = OUT_DIR / 'backprop' / f'prof_{run}' / 'excitation.txt'
    lowpass_filter_excitation(exc_file, f_cut_hz, edge_exclude=bp_edge_exclude)

    # # Sign-bit version
    # write_backprop_files(OUT_DIR, label=f'prof_{run} (sign-bit)', slug=f'prof_{run}_signbit',
    #                      sign_bit=True, **_bp_common)
    # exc_file_sb = OUT_DIR / 'backprop' / f'prof_{run}_signbit' / 'excitation.txt'
    # lowpass_filter_excitation(exc_file_sb, f_cut_hz, edge_exclude=bp_edge_exclude)

In [ ]:
# ── Why post-migration deconvolution does not fix the extra Kirchhoff lobes ─────────
#
# K.H (adjoint Kirchhoff) is a matched filter: its PSF at image point (x0, z0) is
# the aperture-weighted sum of wac(2*Δx*sin(θ_src)/v) over all receivers, where
# sin(θ_src) varies from 0 (far receivers) to 1 (receiver nearest the reflector).
# This makes the PSF SPATIALLY VARIANT — broader than wac because far receivers
# contribute wac near its zero-lag peak, not at the displacement lag.
#
# Deconvolving with wac: over-corrects → more oscillations (observed)
# Deconvolving with wav: effective PSF ≠ wav → no visible change (observed)
#
# The correct fix is in the main processing loop above:
#   K_bp uses wav_delta (spike) instead of wav_mig.
#   K_bp.H then acts as pure back-projection (step 1 is identity),
#   so PSF = aperture sum of wav(2*Δx*sin(θ)/v) — a wav-shaped feature
#   dominated by the nearest receiver, matching Gazdag's t=0 condition.
#
# Re-run the main loop to produce kirchhoff_bp_{run}.npy / .png files.
print("See main processing loop for kirchhoff_bp (delta-wavelet) Kirchhoff images.")
print("Re-run that cell to produce kirchhoff_bp_*.npy alongside kirchhoff_*.npy.")

# Migration

In [4]:
# Frequency spectrum of excitation vs. gprMax dispersion limit
from helper_functions.migration import dispersion_limited_cutoff

exc_path = OUT_DIR / 'backprop' / 'prof_2' / 'excitation.txt'
exc_data = np.loadtxt(exc_path, skiprows=1)
time_s   = exc_data[:, 0]
traces   = exc_data[:, 1:]
dt_s_exc = float(time_s[1] - time_s[0])

# Spectrum of the middle trace (representative)
mid      = traces.shape[1] // 2
spectrum = np.abs(np.fft.rfft(traces[:, mid]))
freq_hz  = np.fft.rfftfreq(len(time_s), dt_s_exc)
freq_ghz = freq_hz * 1e-9

# gprMax dispersion limits for the half-velocity ice material
eps_r_val      = (0.299792458 / v) ** 2
eps_r_half_val = 4.0 * eps_r_val
f_limit_hz  = dispersion_limited_cutoff(eps_r_half_val, bp_dx, safety_factor=1.0)
f_safe_hz   = dispersion_limited_cutoff(eps_r_half_val, bp_dx, safety_factor=0.7)
f_limit_ghz = f_limit_hz * 1e-9
f_safe_ghz  = f_safe_hz  * 1e-9

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(freq_ghz, spectrum / (spectrum.max() + 1e-30), label='middle trace spectrum')
ax.axvline(f_limit_ghz, color='r',      linestyle='--', label=f'hard limit  {f_limit_ghz:.3f} GHz')
ax.axvline(f_safe_ghz,  color='orange', linestyle='--', label=f'safe cutoff {f_safe_ghz:.3f} GHz')
ax.set_xlabel('Frequency (GHz)')
ax.set_ylabel('Normalised amplitude')
ax.set_title(f'Excitation spectrum  (dx={bp_dx:.3f} m, eps_r_half={eps_r_half_val:.1f})')
ax.legend()
ax.set_xlim(0, 5 * f_limit_ghz)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(OUT_DIR / 'excitation_spectrum.png', dpi=150)
plt.show()

print(f'gprMax hard limit : {f_limit_ghz:.4f} GHz')
print(f'Safe cutoff       : {f_safe_ghz:.4f} GHz')
print(f'dx={bp_dx:.4f} m   eps_r_half={eps_r_half_val:.2f}   v_half={v/2:.4f} m/ns')

gprMax hard limit : 0.3333 GHz
Safe cutoff       : 0.2333 GHz
dx=0.0500 m   eps_r_half=35.95   v_half=0.0500 m/ns


C:\Users\Administrator\AppData\Local\Temp\ipykernel_7332\3446009673.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Back-propagation Pre-processing and Quality Improvement

## Why noise appears at small radial distances

In time-reversal back-propagation every gprMax source is injected at `y ≈ bp_src_y ≈ 0 m`
(the borehole wall). At the focus time the **coherent** part of the field constructively
interferes at the reflector radius (≈ 5–7 m). Near `y = 0` the destructive interference
among all sources is **never complete**, leaving a residual "injection halo" regardless
of how much the input data is pre-processed — this is a structural property of single-sided
time-reversal from a borehole geometry.

The halo is suppressed in display by zeroing the first `BP_NEAR_MASK_M = 2.0 m` of the
radial axis in the snapshot, consecutive-difference, and reference-difference cells.
Increase `BP_NEAR_MASK_M` if the halo extends further.

**Note — trace-by-trace RMS normalisation was tested and reverted.**
Normalising each receiver trace by its RMS value amplifies low-SNR traces (e.g. receivers
far from the fluid front that contain mostly noise). These amplified noisy traces back-propagate
coherently toward `y = 0` (the borehole axis), making the injection halo *worse*, not better.

## Current pre-processing chain

| Step | Operation | Parameter |
|---:|---|---|
| 1 | Band-pass filter | 0.02–0.2 GHz |
| 2 | DC / direct-wave removal | `remove_mean` samples 299–517 |
| 3 | Trace alignment | `align_traces`, upsample ×5 |
| 4 | SVD rank-1 removal | ranks 0–1 |
| 5 | Difference from reference | `d_svd − ref_svd` |
| 6 | Linear gain | compensate geometric spreading |
| 7 | **Spatial Tukey taper** ✓ | `alpha=0.30` — 15 % of receivers tapered each side |
| 8 | **Temporal Tukey taper** ✓ | `alpha=0.10` — 5 % of time samples tapered each end |
| 9 | **f-kz dip filter** ✓ | zero bins where `|kz| > f / v_mig` |
| 10 | Low-pass filter on excitation | below gprMax dispersion limit |
| 11 | Edge source zeroing | `bp_edge_exclude = 5` |

**PML = 15 cells** (increased from 10); `bp_domain_y` and `bp_src_y` update automatically.

## Remaining avenues

| Technique | Expected benefit |
|---|---|
| Increase `BP_NEAR_MASK_M` beyond 2 m | Widen display exclusion if halo extends further |
| `sign_bit=True` in `write_backprop_files` | Uniform ±1 amplitude injection — test vs peak-normalised |
| Increase `bp_edge_exclude` to 10–15 | Zero more boundary sources at grazing PML angles |
| Spiking deconvolution per trace | Compress wavelet before injection (analogous to delta-wavelet Kirchhoff) |
| `bp_pml = 20` | Further reduce boundary reflections at cost of larger domain |

In [5]:
# Save focus-frame snapshots for completed peak-normalised back-propagation runs
import pyvista

# Timing parameters — must match write_backprop_files defaults
N_SNAP         = 30
SNAP_WIN       = 1.0    # ns
BP_NEAR_MASK_M = 1.0    # hide radial dist < this [m] (source injection artifact)

n_t_bp    = rad_cut
dt_ns_bp  = 1.0 / sf
T_ns_bp   = n_t_bp * dt_ns_bp
t0_ns_bp  = 299.0 / sf
dt_s_bp   = dt_ns_bp * 1e-9

t_focus_ns = T_ns_bp - t0_ns_bp
t_start_ns = max(dt_ns_bp, t_focus_ns - SNAP_WIN)
t_start_s  = t_start_ns * 1e-9
snap_step  = max(1, int((T_ns_bp * 1e-9 - t_start_s) / (max(1, N_SNAP - 1) * dt_s_bp)))

print(f't_focus = {t_focus_ns:.4f} ns  |  t_start = {t_start_ns:.4f} ns  |  snap_step = {snap_step}')

bp_root  = OUT_DIR / 'backprop'
snap_out = OUT_DIR / 'backprop_snapshots'
snap_out.mkdir(exist_ok=True)

# Collect completed peak-normalised runs (skip sign-bit)
completed = {}
for d in sorted(bp_root.iterdir(), key=lambda p: int(p.name.split('_')[1]) if p.name.split('_')[-1].isdigit() else 9999):
    if 'signbit' in d.name or not d.is_dir():
        continue
    snap_dir   = d / f'backprop_{d.name}_snaps'
    snap_files = sorted(snap_dir.glob('bp_snap*.vti'),
                        key=lambda p: int(p.stem.replace('bp_snap', ''))) if snap_dir.exists() else []
    if snap_files:
        completed[d.name] = snap_files
        print(f'  {d.name}: {len(snap_files)} snapshots found')

print(f'\nProcessing {len(completed)} completed run(s)...')

for slug, snap_files in completed.items():
    snap_times_ns = t_start_ns + np.arange(len(snap_files)) * snap_step * dt_ns_bp
    idx_focus     = int(np.argmin(np.abs(snap_times_ns - t_focus_ns))) + 28
    t_actual_ns   = snap_times_ns[idx_focus]

    mesh   = pyvista.read(str(snap_files[idx_focus]))
    nx_c   = mesh.dimensions[0] - 1   # depth cells  (gprMax x)
    ny_c   = mesh.dimensions[1] - 1   # radial cells (gprMax y)
    dx_m   = mesh.spacing[0]
    domain_x = nx_c * dx_m            # borehole depth extent [m]
    domain_y = ny_c * dx_m            # radial extent [m]

    # x-first cell ordering → reshape (ny_c, nx_c), then transpose to (nx_c, ny_c)
    # so rows = depth, cols = radial for imshow
    e_data = np.array(mesh['E-field'])          # (nx_c*ny_c, 3)
    ez     = e_data[:, 2].reshape(ny_c, nx_c).T   # (nx_c, ny_c): rows=depth, cols=radial
    mag    = np.linalg.norm(e_data, axis=1).reshape(ny_c, nx_c).T

    # extent: [radial_min, radial_max, depth_min, depth_max]
    extent = [0, domain_y, 0, domain_x]

    # Mask source injection zone (radial < BP_NEAR_MASK_M) for display only.
    # gprMax sources are at y ≈ 0; their near-field dominates there regardless
    # of preprocessing — the reflector focus always lies at larger radial dist.
    _mask_px = max(1, round(BP_NEAR_MASK_M / dx_m))
    ez_disp  = ez.copy();  ez_disp[:, :_mask_px]  = 0.0
    mag_disp = mag.copy(); mag_disp[:, :_mask_px] = 0.0
    clim_ez  = (np.percentile(np.abs(ez_disp[ez_disp != 0]), 100)
                if ez_disp.any() else 1.0)

    fig, axes = plt.subplots(1, 2, figsize=(10, 8))

    im0 = axes[0].imshow(mag_disp, aspect='auto', cmap='inferno',
                          extent=extent, origin='lower')
    plt.colorbar(im0, ax=axes[0], label='|E| [V/m]')
    axes[0].invert_yaxis()
    axes[0].set_title(f'{slug}  |  |E|  |  t={t_actual_ns:.3f} ns (snap {idx_focus})')
    axes[0].set_xlabel('Radial distance [m]')
    axes[0].set_ylabel('Depth [m]')
    axes[0].set_ylim(60, 85)
    axes[0].set_xlim(BP_NEAR_MASK_M, domain_y)
    axes[0].invert_yaxis()

    im1 = axes[1].imshow(ez_disp, aspect='auto', cmap='seismic',
                          extent=extent, origin='lower',
                          vmin=-clim_ez, vmax=clim_ez)
    plt.colorbar(im1, ax=axes[1], label='Ez [V/m]')
    axes[1].invert_yaxis()
    axes[1].set_title(f'{slug}  |  Ez  |  t={t_actual_ns:.3f} ns')
    axes[1].set_xlabel('Radial distance [m]')
    axes[1].set_ylabel('Depth [m]')
    axes[1].set_ylim(60, 85)
    axes[1].set_xlim(BP_NEAR_MASK_M, domain_y)
    axes[1].invert_yaxis()

    plt.tight_layout()
    out_path = snap_out / f'{slug}_focus.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f'  Saved {out_path.name}')

print('Done.')

t_focus = 0.8713 ns  |  t_start = 0.8713 ns  |  snap_step = 10
  prof_1: 29 snapshots found
  prof_2: 29 snapshots found
  prof_3: 29 snapshots found
  prof_4: 29 snapshots found
  prof_5: 29 snapshots found
  prof_7: 29 snapshots found
  prof_8: 29 snapshots found
  prof_9: 29 snapshots found
  prof_10: 29 snapshots found
  prof_13: 29 snapshots found
  prof_16: 29 snapshots found
  prof_20: 29 snapshots found
  prof_21: 29 snapshots found
  prof_38: 29 snapshots found

Processing 14 completed run(s)...


C:\Users\Administrator\AppData\Local\Temp\ipykernel_7332\2702866500.py:98: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  Saved prof_1_focus.png
  Saved prof_2_focus.png
  Saved prof_3_focus.png
  Saved prof_4_focus.png
  Saved prof_5_focus.png
  Saved prof_7_focus.png
  Saved prof_8_focus.png
  Saved prof_9_focus.png
  Saved prof_10_focus.png
  Saved prof_13_focus.png
  Saved prof_16_focus.png
  Saved prof_20_focus.png
  Saved prof_21_focus.png
  Saved prof_38_focus.png
Done.


In [6]:
# Compare migrated images (Kirchhoff, Gazdag) with back-propagation Ez focus frames
# for the key stage-boundary profiles. Use this to judge which BP snapshot index
# gives the best match to the migrated images, and to verify that the BP focus
# lands on the same reflectors as the migration.
#
# Part 1 — Side-by-side grid for all KEY_PROFILES at CMP_FOCUS_OFFSET.
# Part 2 — Offset scan: Kirchhoff + Gazdag + BP at each CMP_SCAN_OFFSETS for
#           one reference profile.
#
# CMP_KIRCHHOFF_METHOD / CMP_GAZDAG_METHOD : filename prefix of the saved .npy
#   (e.g. 'kirchhoff' → OUT_DIR/'migrated'/'kirchhoff_{run}.npy')
# CMP_FOCUS_OFFSET   : snapshot index offset used for the Part 1 grid
# CMP_DEPTH_RANGE    : depth window [m] shown in all panels
# CMP_RADIAL_RANGE   : radial window [m] shown in all panels
# CMP_SCAN_OFFSETS   : list of offsets to try in the Part 2 scan
# CMP_SCAN_PROFILE   : profile number used for the offset scan
# ─────────────────────────────────────────────────────────────────────────────────

KEY_PROFILES         = [1,3,4,8,9,20,21,38]
CMP_KIRCHHOFF_METHOD = 'kirchhoff_bp'   # change to 'kirchhoff_bp' if your files are named that way
CMP_GAZDAG_METHOD    = 'gazdag'
CMP_FOCUS_OFFSET     = 28    # snapshot offset for the grid comparison
CMP_DEPTH_RANGE      = (60, 85)    # [m]
CMP_RADIAL_RANGE     = (1.0, 14.0) # [m]
CMP_SCAN_OFFSETS     = [18, 20, 22, 24, 26, 28, 30]
CMP_SCAN_PROFILE     = 1

# ─────────────────────────────────────────────────────────────────────────────────

cmp_out = OUT_DIR / 'comparison'
cmp_out.mkdir(exist_ok=True)

def _cmp_load_mig(method, run):
    p = OUT_DIR / 'migrated' / f'{method}_{run}.npy'
    return np.load(p) if p.exists() else None

def _cmp_load_bp(slug, offset):
    snap_files = completed.get(slug)
    if not snap_files:
        return None, None
    snap_times_ns = t_start_ns + np.arange(len(snap_files)) * snap_step * dt_ns_bp
    idx = min(int(np.argmin(np.abs(snap_times_ns - t_focus_ns))) + offset,
              len(snap_files) - 1)
    mesh   = pyvista.read(str(snap_files[idx]))
    nx_c   = mesh.dimensions[0] - 1
    ny_c   = mesh.dimensions[1] - 1
    dx_m   = float(mesh.spacing[0])
    e_data = np.array(mesh['E-field'])
    ez     = e_data[:, 2].reshape(ny_c, nx_c).T   # (depth, radial)
    return ez, (nx_c * dx_m, ny_c * dx_m, snap_times_ns[idx])

# ── Part 1: grid comparison ───────────────────────────────────────────────────────
n_prof = len(KEY_PROFILES)
fig1, axes1 = plt.subplots(n_prof, 3, figsize=(13, 3.2 * n_prof),
                            sharex=True, sharey=True)
axes1[0, 0].set_title(CMP_KIRCHHOFF_METHOD.capitalize())
axes1[0, 1].set_title(CMP_GAZDAG_METHOD.capitalize())
axes1[0, 2].set_title(f'Back-propagation Ez  (offset={CMP_FOCUS_OFFSET})')
fig1.suptitle('Migrated vs back-propagation: key profiles', y=1.01)

for row, run in enumerate(KEY_PROFILES):
    slug = f'prof_{run}'

    # Kirchhoff
    img_k = _cmp_load_mig(CMP_KIRCHHOFF_METHOD, run)
    ax_k  = axes1[row, 0]
    if img_k is not None:
        n     = img_k.shape[0]
        ext_k = [x_img[0], x_img[-1], depth[n - 1], depth[0]]
        clim  = np.percentile(np.abs(img_k), 100)
        ax_k.imshow(img_k, aspect='auto', cmap='seismic',
                    extent=ext_k, origin='upper', vmin=-clim, vmax=clim)
        ax_k.invert_yaxis()
    else:
        ax_k.text(0.5, 0.5, 'not found', ha='center', va='center',
                  transform=ax_k.transAxes)
    ax_k.set_ylabel(f'prof_{run}')
    ax_k.set_ylim(*CMP_DEPTH_RANGE)
    ax_k.set_xlim(*CMP_RADIAL_RANGE)

    # Gazdag
    img_g = _cmp_load_mig(CMP_GAZDAG_METHOD, run)
    ax_g  = axes1[row, 1]
    if img_g is not None:
        n     = img_g.shape[0]
        ext_g = [x_img[0], x_img[-1], depth[n - 1], depth[0]]
        clim  = np.percentile(np.abs(img_g), 100)
        ax_g.imshow(img_g, aspect='auto', cmap='seismic',
                    extent=ext_g, origin='upper', vmin=-clim, vmax=clim)
        ax_g.invert_yaxis()
    else:
        ax_g.text(0.5, 0.5, 'not found', ha='center', va='center',
                  transform=ax_g.transAxes)
    ax_g.set_ylim(*CMP_DEPTH_RANGE)
    ax_g.set_xlim(*CMP_RADIAL_RANGE)

    # Back-propagation
    ez_bp, meta = _cmp_load_bp(slug, CMP_FOCUS_OFFSET)
    ax_bp = axes1[row, 2]
    if ez_bp is not None:
        dom_x, dom_y, t_ns = meta
        clim  = np.percentile(np.abs(ez_bp), 100)
        ax_bp.imshow(ez_bp, aspect='auto', cmap='seismic',
                     extent=[0, dom_y, 0, dom_x], origin='lower',
                     vmin=-clim, vmax=clim)
        ax_bp.invert_yaxis()
        ax_bp.text(0.98, 0.02, f't={t_ns:.2f} ns',
                   ha='right', va='bottom', transform=ax_bp.transAxes,
                   fontsize=7, color='white')
    else:
        ax_bp.text(0.5, 0.5, 'not found', ha='center', va='center',
                   transform=ax_bp.transAxes)
    ax_bp.set_ylim(*CMP_DEPTH_RANGE)
    ax_bp.set_xlim(*CMP_RADIAL_RANGE)

    if row == n_prof - 1:
        for ax in axes1[row]:
            ax.set_xlabel('Radial distance [m]')

plt.tight_layout()
fig1.savefig(cmp_out / f'comparison_grid_offset{CMP_FOCUS_OFFSET}.png', dpi=150, bbox_inches='tight')
plt.show();  plt.close(fig1)
print(f'Saved comparison_grid_offset{CMP_FOCUS_OFFSET}.png')

# ── Part 2: offset scan for one reference profile ─────────────────────────────────
slug_scan = f'prof_{CMP_SCAN_PROFILE}'
img_k_scan = _cmp_load_mig(CMP_KIRCHHOFF_METHOD, CMP_SCAN_PROFILE)
img_g_scan = _cmp_load_mig(CMP_GAZDAG_METHOD,    CMP_SCAN_PROFILE)

n_off  = len(CMP_SCAN_OFFSETS)
n_cols = 2 + n_off
fig2, axes2 = plt.subplots(1, n_cols, figsize=(3.5 * n_cols, 7), sharey=True)
fig2.suptitle(f'BP snapshot offset scan — {slug_scan}  '
              f'(depth {CMP_DEPTH_RANGE[0]}–{CMP_DEPTH_RANGE[1]} m)', y=1.01)

# Kirchhoff reference
ax0 = axes2[0]
if img_k_scan is not None:
    n     = img_k_scan.shape[0]
    ext_k = [x_img[0], x_img[-1], depth[n - 1], depth[0]]
    clim  = np.percentile(np.abs(img_k_scan), 100)
    ax0.imshow(img_k_scan, aspect='auto', cmap='seismic',
               extent=ext_k, origin='upper', vmin=-clim, vmax=clim)
    ax0.invert_yaxis()
ax0.set_ylim(*CMP_DEPTH_RANGE);  ax0.set_xlim(*CMP_RADIAL_RANGE)
ax0.set_title(CMP_KIRCHHOFF_METHOD.capitalize())
ax0.set_xlabel('Radial [m]');  ax0.set_ylabel('Depth [m]')

# Gazdag reference
ax1 = axes2[1]
if img_g_scan is not None:
    n     = img_g_scan.shape[0]
    ext_g = [x_img[0], x_img[-1], depth[n - 1], depth[0]]
    clim  = np.percentile(np.abs(img_g_scan), 100)
    ax1.imshow(img_g_scan, aspect='auto', cmap='seismic',
               extent=ext_g, origin='upper', vmin=-clim, vmax=clim)
    ax1.invert_yaxis()
ax1.set_ylim(*CMP_DEPTH_RANGE);  ax1.set_xlim(*CMP_RADIAL_RANGE)
ax1.set_title(CMP_GAZDAG_METHOD.capitalize())
ax1.set_xlabel('Radial [m]')

# BP at each offset
for j, offset in enumerate(CMP_SCAN_OFFSETS):
    ez_bp, meta = _cmp_load_bp(slug_scan, offset)
    ax_j = axes2[2 + j]
    if ez_bp is not None:
        dom_x, dom_y, t_ns = meta
        clim = np.percentile(np.abs(ez_bp), 100)
        ax_j.imshow(ez_bp, aspect='auto', cmap='seismic',
                    extent=[0, dom_y, 0, dom_x], origin='lower',
                    vmin=-clim, vmax=clim)
        ax_j.invert_yaxis()
        ax_j.set_title(f'BP  offset={offset}\nt={t_ns:.2f} ns')
    else:
        ax_j.text(0.5, 0.5, 'not found', ha='center', va='center',
                  transform=ax_j.transAxes)
        ax_j.set_title(f'BP  offset={offset}')
    ax_j.set_ylim(*CMP_DEPTH_RANGE);  ax_j.set_xlim(*CMP_RADIAL_RANGE)
    ax_j.set_xlabel('Radial [m]')

plt.tight_layout()
fig2.savefig(cmp_out / f'bp_offset_scan_{slug_scan}.png', dpi=150, bbox_inches='tight')
plt.show();  plt.close(fig2)
print(f'Saved bp_offset_scan_{slug_scan}.png')

print('Done.')


C:\Users\Administrator\AppData\Local\Temp\ipykernel_7332\697034724.py:122: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show();  plt.close(fig1)


Saved comparison_grid_offset28.png
Saved bp_offset_scan_prof_1.png
Done.


C:\Users\Administrator\AppData\Local\Temp\ipykernel_7332\697034724.py:183: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show();  plt.close(fig2)


# Time-lapse Differencing t_n - t_n-1

In [7]:
# Kirchhoff & Gazdag: image(t_n+1) - image(t_n) for consecutive runs
# Requires the main loop (previous section) to have been re-run so that
# kirchhoff_{run}.npy / gazdag_{run}.npy exist alongside the .png files.

import matplotlib.ticker as ticker

diff_dir = OUT_DIR / 'difference'
diff_dir.mkdir(exist_ok=True)

DIFF_SC        = 1.2    # color-limit scale factor
TICK_SPACING_X = 0.5    # radial-axis tick spacing [m]
TICK_SPACING_Z = 1.0    # depth-axis tick spacing [m]
NOISE_SUPPRESS = 0.15   # suppress values below this fraction of each image's peak

def _consecutive_pairs(method):
    """Yield (run_a, run_b, img_a, img_b) for runs with saved .npy arrays, in data_runs order."""
    avail = [r for r in data_runs if (OUT_DIR / 'migrated' / f'{method}_{r}.npy').exists()]
    for run_a, run_b in zip(avail[:-1], avail[1:]):
        img_a = np.load(OUT_DIR / 'migrated' / f'{method}_{run_a}.npy')
        img_b = np.load(OUT_DIR / 'migrated' / f'{method}_{run_b}.npy')
        n_common = min(img_a.shape[0], img_b.shape[0])
        yield run_a, run_b, img_a[:n_common], img_b[:n_common]

for method, label in [('kirchhoff', 'Kirchhoff'), ('gazdag', 'Gazdag')]:
    pairs = list(_consecutive_pairs(method))
    if not pairs:
        print(f'[{label}] No saved .npy arrays found — re-run the main processing loop first.')
        continue
    print(f'[{label}] {len(pairs)} consecutive pair(s) found')

    for run_a, run_b, img_a, img_b in pairs:
        diff      = img_b - img_a
        z_common  = depth[:diff.shape[0]]
        lim       = max(DIFF_SC * np.max(np.abs(diff)), 1.0)
        diff_disp = np.where(np.abs(diff) < NOISE_SUPPRESS * np.max(np.abs(diff)), 0.0, diff)

        fig, ax = plt.subplots(figsize=(8, 6))
        ax.imshow(diff_disp, aspect='auto', cmap='seismic',
                  extent=[x_img[0], x_img[-1], z_common[-1], z_common[0]],
                  vmin=-lim, vmax=lim)
        ax.invert_yaxis()
        ax.xaxis.set_major_locator(ticker.MultipleLocator(TICK_SPACING_X))
        ax.yaxis.set_major_locator(ticker.MultipleLocator(TICK_SPACING_Z))
        ax.grid(True, color='k', linewidth=0.3, alpha=0.4)
        ax.set_xlabel('Radial distance from borehole (m)')
        ax.set_ylabel('Depth (m)')
        ax.set_title(f'{label} difference: prof_{run_b} − prof_{run_a}')
        out_path = diff_dir / f'{method}_diff_{run_a}_to_{run_b}.png'
        fig.savefig(out_path, dpi=150)
        plt.close(fig)
        print(f'  Saved {out_path.name}')

print('Done.')

[Kirchhoff] 36 consecutive pair(s) found
  Saved kirchhoff_diff_1_to_2.png
  Saved kirchhoff_diff_2_to_3.png
  Saved kirchhoff_diff_3_to_4.png
  Saved kirchhoff_diff_4_to_5.png
  Saved kirchhoff_diff_5_to_7.png
  Saved kirchhoff_diff_7_to_8.png
  Saved kirchhoff_diff_8_to_9.png
  Saved kirchhoff_diff_9_to_10.png
  Saved kirchhoff_diff_10_to_11.png
  Saved kirchhoff_diff_11_to_12.png
  Saved kirchhoff_diff_12_to_13.png
  Saved kirchhoff_diff_13_to_14.png
  Saved kirchhoff_diff_14_to_15.png
  Saved kirchhoff_diff_15_to_16.png
  Saved kirchhoff_diff_16_to_17.png
  Saved kirchhoff_diff_17_to_18.png
  Saved kirchhoff_diff_18_to_19.png
  Saved kirchhoff_diff_19_to_20.png
  Saved kirchhoff_diff_20_to_21.png
  Saved kirchhoff_diff_21_to_22.png
  Saved kirchhoff_diff_22_to_23.png
  Saved kirchhoff_diff_23_to_24.png
  Saved kirchhoff_diff_24_to_25.png
  Saved kirchhoff_diff_25_to_26.png
  Saved kirchhoff_diff_26_to_27.png
  Saved kirchhoff_diff_27_to_28.png
  Saved kirchhoff_diff_28_to_29.png
  

In [8]:
# Back-propagation: Ez_focus(t_n+1) - Ez_focus(t_n) for consecutive completed runs
# Requires the snapshot cell above to have been run first (reuses `completed`,
# t_focus_ns, t_start_ns, snap_step, dt_ns_bp).

FOCUS_IDX_OFFSET  = 28    # must match the offset used in the snapshot cell above
BP_DIFF_PCT       = 97    # percentile used for difference colour limits
BP_NOISE_SUPPRESS = 0.15  # suppress values below this fraction of each image's peak
BP_NEAR_MASK_M    = 3.0   # zero radial < this [m] for display (source injection zone)

bp_diff_dir = OUT_DIR / 'backprop_snapshots' / 'difference'
bp_diff_dir.mkdir(parents=True, exist_ok=True)

def _load_focus_ez(snap_files):
    snap_times_ns = t_start_ns + np.arange(len(snap_files)) * snap_step * dt_ns_bp
    idx = min(int(np.argmin(np.abs(snap_times_ns - t_focus_ns))) + FOCUS_IDX_OFFSET,
              len(snap_files) - 1)
    mesh = pyvista.read(str(snap_files[idx]))
    nx_c = mesh.dimensions[0] - 1
    ny_c = mesh.dimensions[1] - 1
    e_data = np.array(mesh['E-field'])
    ez = e_data[:, 2].reshape(ny_c, nx_c).T          # rows=depth, cols=radial
    domain_x = nx_c * mesh.spacing[0]
    domain_y = ny_c * mesh.spacing[0]
    return ez, snap_times_ns[idx], domain_x, domain_y

avail_bp = sorted(completed.keys(), key=lambda s: int(s.split('_')[1]))
if len(avail_bp) < 2:
    print('Fewer than 2 completed back-propagation runs — nothing to difference yet.')
else:
    print(f'{len(avail_bp)} completed run(s): {avail_bp}')
    for slug_a, slug_b in zip(avail_bp[:-1], avail_bp[1:]):
        ez_a, t_a, dom_x, dom_y = _load_focus_ez(completed[slug_a])
        ez_b, t_b, _, _         = _load_focus_ez(completed[slug_b])

        n_x = min(ez_a.shape[0], ez_b.shape[0])
        n_y = min(ez_a.shape[1], ez_b.shape[1])
        diff = ez_b[:n_x, :n_y] - ez_a[:n_x, :n_y]

        extent    = [0, dom_y, 0, dom_x]
        clim      = np.percentile(np.abs(diff), BP_DIFF_PCT)
        diff_disp = np.where(np.abs(diff) < BP_NOISE_SUPPRESS * np.max(np.abs(diff)), 0.0, diff)
        _dx_bp = dom_y / ez_a.shape[1]   # [m/px]
        _mpx   = max(1, round(BP_NEAR_MASK_M / _dx_bp))
        diff_disp[:, :_mpx] = 0.0

        fig, ax = plt.subplots(figsize=(6, 8))
        im = ax.imshow(diff_disp, aspect='auto', cmap='seismic',
                       extent=extent, origin='lower', vmin=-clim, vmax=clim)
        plt.colorbar(im, ax=ax, label='ΔEz [V/m]')
        ax.set_ylim(60, 85)
        ax.set_xlim(BP_NEAR_MASK_M, dom_y)
        ax.invert_yaxis()
        ax.set_xlabel('Radial distance [m]')
        ax.set_ylabel('Depth [m]')
        ax.set_title(f'Back-propagation ΔEz: {slug_b} (t={t_b:.2f} ns) − {slug_a} (t={t_a:.2f} ns)')

        plt.tight_layout()
        out_path = bp_diff_dir / f'{slug_a}_to_{slug_b}_diff.png'
        plt.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f'  Saved {out_path.name}')

print('Done.')

14 completed run(s): ['prof_1', 'prof_2', 'prof_3', 'prof_4', 'prof_5', 'prof_7', 'prof_8', 'prof_9', 'prof_10', 'prof_13', 'prof_16', 'prof_20', 'prof_21', 'prof_38']
  Saved prof_1_to_prof_2_diff.png
  Saved prof_2_to_prof_3_diff.png
  Saved prof_3_to_prof_4_diff.png
  Saved prof_4_to_prof_5_diff.png
  Saved prof_5_to_prof_7_diff.png
  Saved prof_7_to_prof_8_diff.png
  Saved prof_8_to_prof_9_diff.png
  Saved prof_9_to_prof_10_diff.png
  Saved prof_10_to_prof_13_diff.png
  Saved prof_13_to_prof_16_diff.png
  Saved prof_16_to_prof_20_diff.png
  Saved prof_20_to_prof_21_diff.png
  Saved prof_21_to_prof_38_diff.png
Done.


# Time-lapse Differencing t_n - t_reference

In [ ]:
# Kirchhoff & Gazdag: image(t_n) - image(t_reference) for all runs vs a fixed baseline.
# Reuses DIFF_SC, TICK_SPACING_X/Z, ticker from the consecutive-pair cell above.

REF_RUN_DIFF   = 1     # prof1 (first injection measurement)
NOISE_SUPPRESS = 0.00  # suppress values below this fraction of each image's peak

ref_diff_dir = OUT_DIR / 'difference_from_ref'
ref_diff_dir.mkdir(exist_ok=True)

for method, label in [('kirchhoff', 'Kirchhoff'), ('gazdag', 'Gazdag')]:
    ref_path = OUT_DIR / 'migrated' / f'{method}_{REF_RUN_DIFF}.npy'
    if not ref_path.exists():
        print(f'[{label}] Reference {ref_path.name} not found — skipping.')
        continue
    img_ref = np.load(ref_path)

    avail = [r for r in data_runs
             if r != REF_RUN_DIFF and (OUT_DIR / 'migrated' / f'{method}_{r}.npy').exists()]
    if not avail:
        print(f'[{label}] No non-reference .npy arrays found.')
        continue
    print(f'[{label}] {len(avail)} run(s) vs ref=prof_{REF_RUN_DIFF}')

    for run in avail:
        img_n    = np.load(OUT_DIR / 'migrated' / f'{method}_{run}.npy')
        n_common = min(img_ref.shape[0], img_n.shape[0])
        diff     = img_n[:n_common] - img_ref[:n_common]
        z_common = depth[:n_common]

        lim       = max(DIFF_SC * np.max(np.abs(diff)), 1.0)
        diff_disp = np.where(np.abs(diff) < NOISE_SUPPRESS * np.max(np.abs(diff)), 0.0, diff)

        fig, ax = plt.subplots(figsize=(8, 6))
        ax.imshow(diff_disp, aspect='auto', cmap='seismic',
                  extent=[x_img[0], x_img[-1], z_common[-1], z_common[0]],
                  vmin=-lim, vmax=lim)
        ax.invert_yaxis()
        ax.xaxis.set_major_locator(ticker.MultipleLocator(TICK_SPACING_X))
        ax.yaxis.set_major_locator(ticker.MultipleLocator(TICK_SPACING_Z))
        ax.grid(True, color='k', linewidth=0.3, alpha=0.4)
        ax.set_xlabel('Radial distance from borehole (m)')
        ax.set_ylabel('Depth (m)')
        ax.set_title(f'{label} difference: prof_{run} − prof_{REF_RUN_DIFF}')
        out_path = ref_diff_dir / f'{method}_diff_ref{REF_RUN_DIFF}_to_{run}.png'
        fig.savefig(out_path, dpi=150)
        plt.close(fig)
        print(f'  Saved {out_path.name}')

print('Done.')

In [9]:
# Back-propagation: Ez_focus(t_n) - Ez_focus(t_reference) for all completed runs.
# Requires the snapshot cell to have been run (reuses completed, t_focus_ns, etc.).

BP_REF_SLUG       = 'prof_1'   # slug for prof1 (first injection measurement)
BP_REF_DIFF_PCT   = 97         # percentile for colour limits
BP_NOISE_SUPPRESS = 0.15       # suppress values below this fraction of each image's peak
BP_NEAR_MASK_M    = 3.0        # zero radial < this [m] for display (source injection zone)

bp_ref_diff_dir = OUT_DIR / 'backprop_snapshots' / 'difference_from_ref'
bp_ref_diff_dir.mkdir(parents=True, exist_ok=True)

if BP_REF_SLUG not in completed:
    print(f'{BP_REF_SLUG} not in completed dict — run the snapshot cell first.')
else:
    ez_ref, t_ref, dom_x_ref, dom_y_ref = _load_focus_ez(completed[BP_REF_SLUG])
    _dx_bp_ref = dom_y_ref / ez_ref.shape[1]   # spatial pixel size [m/px]
    avail_bp_ref = sorted(
        [s for s in completed if s != BP_REF_SLUG],
        key=lambda s: int(s.split('_')[1])
    )
    print(f'{len(avail_bp_ref)} run(s) vs ref={BP_REF_SLUG}')

    for slug in avail_bp_ref:
        ez_n, t_n, dom_x, dom_y = _load_focus_ez(completed[slug])

        n_x = min(ez_ref.shape[0], ez_n.shape[0])
        n_y = min(ez_ref.shape[1], ez_n.shape[1])
        diff = ez_n[:n_x, :n_y] - ez_ref[:n_x, :n_y]

        extent    = [0, dom_y, 0, dom_x]
        clim      = np.percentile(np.abs(diff), BP_REF_DIFF_PCT)
        diff_disp = np.where(np.abs(diff) < BP_NOISE_SUPPRESS * np.max(np.abs(diff)), 0.0, diff)
        _mpx_r = max(1, round(BP_NEAR_MASK_M / _dx_bp_ref))
        diff_disp[:, :_mpx_r] = 0.0

        fig, ax = plt.subplots(figsize=(6, 8))
        im = ax.imshow(diff_disp, aspect='auto', cmap='seismic',
                       extent=extent, origin='lower', vmin=-clim, vmax=clim)
        plt.colorbar(im, ax=ax, label='ΔEz [V/m]')
        ax.set_ylim(60, 85)
        ax.set_xlim(BP_NEAR_MASK_M, dom_y)
        ax.invert_yaxis()
        ax.set_xlabel('Radial distance [m]')
        ax.set_ylabel('Depth [m]')
        ax.set_title(f'Back-propagation ΔEz: {slug} (t={t_n:.2f} ns) − {BP_REF_SLUG} (t={t_ref:.2f} ns)')

        plt.tight_layout()
        out_path = bp_ref_diff_dir / f'{BP_REF_SLUG}_to_{slug}_diff.png'
        plt.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f'  Saved {out_path.name}')

print('Done.')

13 run(s) vs ref=prof_1
  Saved prof_1_to_prof_2_diff.png
  Saved prof_1_to_prof_3_diff.png
  Saved prof_1_to_prof_4_diff.png
  Saved prof_1_to_prof_5_diff.png
  Saved prof_1_to_prof_7_diff.png
  Saved prof_1_to_prof_8_diff.png
  Saved prof_1_to_prof_9_diff.png
  Saved prof_1_to_prof_10_diff.png
  Saved prof_1_to_prof_13_diff.png
  Saved prof_1_to_prof_16_diff.png
  Saved prof_1_to_prof_20_diff.png
  Saved prof_1_to_prof_21_diff.png
  Saved prof_1_to_prof_38_diff.png
Done.


# Time-lapse Differencing by Phase

Phase-total differences: **end-of-phase minus start-of-phase**.
Each image captures what changed over the full duration of that pumping stage.

| Phase | Start profile | End profile |
|---|---|---|
| Ref.     | prof     |          |
| Pushing  | prof\_1  | prof\_3  |
| Chasing  | prof\_4  | prof\_8  |
| Waiting  | prof\_9  | prof\_20 |
| Pulling  | prof\_21 | prof\_38 |


In [ ]:
# Kirchhoff & Gazdag: phase-total differences — end_of_phase − start_of_phase

PHASE_PAIRS = [
    ('Pushing',  1,  3),
    ('Chasing',  4,  8),
    ('Waiting',  9, 20),
    ('Pulling', 21, 38),
]

DIFF_SC        = 1.2    # colour-limit scale factor
TICK_SPACING_X = 0.5    # radial tick spacing [m]
TICK_SPACING_Z = 1.0    # depth tick spacing [m]
NOISE_SUPPRESS = 0.00   # suppress values below this fraction of the image peak

phase_diff_dir = OUT_DIR / 'difference_by_phase'
phase_diff_dir.mkdir(exist_ok=True)

for method, label in [('kirchhoff', 'Kirchhoff'), ('gazdag', 'Gazdag')]:
    for phase_name, run_start, run_end in PHASE_PAIRS:
        path_a = OUT_DIR / 'migrated' / f'{method}_{run_start}.npy'
        path_b = OUT_DIR / 'migrated' / f'{method}_{run_end}.npy'
        if not path_a.exists() or not path_b.exists():
            print(f'[{label}] {phase_name}: missing prof_{run_start} or prof_{run_end} — skipping.')
            continue
        img_a = np.load(path_a)
        img_b = np.load(path_b)
        n     = min(img_a.shape[0], img_b.shape[0])
        diff  = img_b[:n] - img_a[:n]
        z_common = depth[:n]

        lim       = max(DIFF_SC * np.max(np.abs(diff)), 1.0)
        diff_disp = np.where(np.abs(diff) < NOISE_SUPPRESS * np.max(np.abs(diff)), 0.0, diff)

        fig, ax = plt.subplots(figsize=(8, 6))
        ax.imshow(diff_disp, aspect='auto', cmap='seismic',
                  extent=[x_img[0], x_img[-1], z_common[-1], z_common[0]],
                  vmin=-lim, vmax=lim)
        ax.invert_yaxis()
        ax.xaxis.set_major_locator(ticker.MultipleLocator(TICK_SPACING_X))
        ax.yaxis.set_major_locator(ticker.MultipleLocator(TICK_SPACING_Z))
        ax.grid(True, color='k', linewidth=0.3, alpha=0.4)
        ax.set_xlabel('Radial distance from borehole (m)')
        ax.set_ylabel('Depth (m)')
        ax.set_title(f'{label} — {phase_name}: prof_{run_end} − prof_{run_start}')
        out_path = phase_diff_dir / f'{method}_phase_{phase_name.lower()}.png'
        fig.savefig(out_path, dpi=150)
        plt.close(fig)
        print(f'  Saved {out_path.name}')

print('Done.')


In [ ]:
# Back-propagation: Ez phase-total differences — end_of_phase − start_of_phase
# Reuses PHASE_PAIRS and _load_focus_ez() from the cells above.

BP_PHASE_DIFF_PCT   = 100    # percentile for colour limits
BP_PHASE_NOISE_SUPP = 0  # suppress values below this fraction of the image peak
BP_PHASE_NEAR_MASK  = 3.0   # zero radial < this [m] for display (source zone)

bp_phase_diff_dir = OUT_DIR / 'backprop_snapshots' / 'difference_by_phase'
bp_phase_diff_dir.mkdir(parents=True, exist_ok=True)

for phase_name, run_start, run_end in PHASE_PAIRS:
    slug_a = f'prof_{run_start}'
    slug_b = f'prof_{run_end}'
    if slug_a not in completed or slug_b not in completed:
        missing = slug_a if slug_a not in completed else slug_b
        print(f'[{phase_name}] {missing} not in completed — skipping.')
        continue

    ez_a, t_a, dom_x, dom_y = _load_focus_ez(completed[slug_a])
    ez_b, t_b, _,    _      = _load_focus_ez(completed[slug_b])

    n_x  = min(ez_a.shape[0], ez_b.shape[0])
    n_y  = min(ez_a.shape[1], ez_b.shape[1])
    diff = ez_b[:n_x, :n_y] - ez_a[:n_x, :n_y]

    extent    = [0, dom_y, 0, dom_x]
    clim      = np.percentile(np.abs(diff), BP_PHASE_DIFF_PCT)
    diff_disp = np.where(np.abs(diff) < BP_PHASE_NOISE_SUPP * np.max(np.abs(diff)),
                         0.0, diff)
    _dx_bp = dom_y / ez_a.shape[1]
    _mpx   = max(1, round(BP_PHASE_NEAR_MASK / _dx_bp))
    diff_disp[:, :_mpx] = 0.0

    fig, ax = plt.subplots(figsize=(6, 8))
    im = ax.imshow(diff_disp, aspect='auto', cmap='seismic',
                   extent=extent, origin='lower', vmin=-clim, vmax=clim)
    plt.colorbar(im, ax=ax, label='ΔEz [V/m]')
    ax.set_ylim(60, 85)
    ax.set_xlim(BP_PHASE_NEAR_MASK, dom_y)
    ax.invert_yaxis()
    ax.set_xlabel('Radial distance [m]')
    ax.set_ylabel('Depth [m]')
    ax.set_title(f'Back-prop — {phase_name}: {slug_b} (t={t_b:.2f} ns) − {slug_a} (t={t_a:.2f} ns)')

    plt.tight_layout()
    out_path = bp_phase_diff_dir / f'bp_phase_{phase_name.lower()}.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved {out_path.name}')

print('Done.')


# Fluid Front Movement Estimate

Pushing phase: profiles 1-3

Chasing phase: profiles 4-8

Waiting phase: profiles 9-20

Pulling phase: profiles 21-38

In [10]:
# ── Helper functions: Riesz transform, monogenic envelope, ROI, WLS phase fit ──────
from scipy.signal.windows import tukey
from matplotlib.patches import Rectangle

def _riesz_2d(img):
    """Full 2D Riesz transform.
    Returns (R_z, R_x): the two real-valued spatial components of the monogenic signal.
      R_z = IFFT2(-i * kz/|k| * FFT2(img))
      R_x = IFFT2(-i * kx/|k| * FFT2(img))
    Uses the same FFT convention as estimate_shift_2d (numpy, no fftshift).
    """
    nz, nx = img.shape
    KZ, KX = np.meshgrid(np.fft.fftfreq(nz), np.fft.fftfreq(nx), indexing='ij')
    K = np.sqrt(KZ**2 + KX**2)
    K[0, 0] = 1.0      # avoid DC divide-by-zero; DC component maps to 0 anyway
    F  = np.fft.fft2(img)
    Rz = np.real(np.fft.ifft2((-1j * KZ / K) * F))
    Rx = np.real(np.fft.ifft2((-1j * KX / K) * F))
    return Rz, Rx

def _monogenic_envelope(img):
    """Amplitude of the monogenic signal: sqrt(f^2 + Rz^2 + Rx^2).
    This is the 2D generalisation of the 1D Hilbert envelope; it is
    phase-invariant and highlights the spatial extent of coherent energy
    regardless of whether the wavelet is at a zero-crossing or a peak.
    """
    Rz, Rx = _riesz_2d(img)
    return np.sqrt(img**2 + Rz**2 + Rx**2)

def _roi_from_envelope(env, threshold_frac=0.5):
    """Tight bounding box of the high-energy region in the envelope image.
    Pixels with env >= threshold_frac * env.max() define the ROI mask;
    returns (z0, z1, x0, x1) as integer row/col indices (z1, x1 exclusive).
    Falls back to the full image if no pixels survive the threshold.
    """
    mask = env >= threshold_frac * env.max()
    rows, cols = np.where(mask)
    if rows.size == 0:
        return 0, env.shape[0], 0, env.shape[1]
    return int(rows.min()), int(rows.max()) + 1, int(cols.min()), int(cols.max()) + 1

def _estimate_shift_2d(base, mon, dz_g, dx_g, kz_cent, force_dz_zero=False, pad_fac=4):
    """WLS 2D phase-plane fit — mirrors estimate_shift_2d from TimeLapse_Processing.ipynb.

    Fits phi(kz, kx) = kz*dz + kx*dx + phi_0 to the cross-spectrum of base and mon,
    weighted by spectral amplitude and restricted to a band around kz_cent.

    Parameters
    ----------
    base, mon   : 2D arrays (n_depth, n_radial), pre-cropped to the ROI
    dz_g        : depth grid spacing [m]   (row spacing, axis 0)
    dx_g        : radial grid spacing [m]  (col spacing, axis 1)
    kz_cent     : dominant wavenumber [rad/m] = 2π * f0 / v
    force_dz_zero : if True, fit only (dx, phi_0) — use for purely lateral motion
    pad_fac     : zero-padding factor before FFT2 for denser kx/kz sampling (default 4)

    Returns (dz_est, dx_est, phi_0) all as floats [m, m, rad].
    """
    Nz, Nx = base.shape
    Nz_pad, Nx_pad = Nz * pad_fac, Nx * pad_fac
    kz_ax = np.fft.fftfreq(Nz_pad, d=dz_g) * 2 * np.pi
    kx_ax = np.fft.fftfreq(Nx_pad, d=dx_g) * 2 * np.pi
    KZ, KX = np.meshgrid(kz_ax, kx_ax, indexing='ij')
    taper = np.outer(tukey(Nz, alpha=0.15), tukey(Nx, alpha=0.15))
    XS  = (np.fft.fft2(base * taper, s=(Nz_pad, Nx_pad)) *
           np.conj(np.fft.fft2(mon  * taper, s=(Nz_pad, Nx_pad))))
    w   = np.abs(XS)
    phi = np.angle(XS)
    band = (np.abs(KZ) < 1.4 * kz_cent) & (np.abs(KX) < 1.4 * kz_cent)
    mask = (w > 0.10 * w.max()) & band & ((np.abs(KZ) + np.abs(KX)) > 0)
    W = w[mask]
    if force_dz_zero:
        A = np.column_stack([KX[mask], np.ones(mask.sum())])
        c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
        return 0.0, float(c[0]), float(c[1])
    A = np.column_stack([KZ[mask], KX[mask], np.ones(mask.sum())])
    c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
    return float(c[0]), float(c[1]), float(c[2])

## WLS Cross-Spectrum Phase Fitting — Parameter Reference

### Method overview
For each pair of migrated profiles the cross-spectrum
$\mathrm{XS}(k_z, k_x) = \mathcal{F}(\mathrm{base}) \cdot \mathcal{F}(\mathrm{mon})^*$
is computed on a zero-padded, Tukey-tapered ROI crop (`pad_fac = 10`, `alpha = 0.15`).
A **weighted least-squares plane** is fitted to the cross-spectrum phase:
$$\varphi(k_z, k_x) = k_z \,\Delta z + k_x \,\Delta x + \varphi_0$$
with cross-spectrum amplitude $|\mathrm{XS}|$ as weights. $\Delta z$ is displacement along the borehole (depth), $\Delta x$ is radial displacement.

---

### Band and amplitude parameters

| Parameter | Value | Role |
|---|---|---|
| `KX_BAND_FAC` | 2.0 | kx band: $|k_x| < 2.0 \times k_{z,c}$. Must cover the two signal lobes at $k_x \approx \pm 8\,\text{rad/m}$. Per-pair override via `MANUAL_KX_BAND_FAC`. |
| `KZ_BAND_FAC` | 0.5 (default) | Default kz band: $|k_z| < 0.5 \times k_{z,c}$. Overridden per-pair by `MANUAL_KZ_BAND_FAC`. |
| `WLS_AMP_THR` | 0.25 | Amplitude gate: only k-cells with $|\mathrm{XS}| > 0.25 \times \max|\mathrm{XS}|$ enter the WLS fit. Rejects noise and near-wrap outer cells. |
| `WLS_POW` | 3 | Weight exponent: $w_i = |\mathrm{XS}_i|^3$. Suppresses low-energy peripheral cells relative to the dominant signal lobes. |

**Why kz and kx bands are separated:** the signal energy sits in two horizontal lobes at
$k_x \approx \pm 8\,\text{rad/m},\; k_z \approx 0$ (sub-horizontal borehole reflectors).
The kx band must stay wide enough to include these lobes.
The kz band can be narrowed independently for stages where only the low-$k_z$ region
is coherent, without cutting the signal in $k_x$.

**Phase-wrapping limit:** wrapping occurs when $k_{z,\text{max}} \times |\Delta z| > \pi$.
At `KZ_BAND_FAC = 0.5` the limit depends on $k_{z,c}$. The Push and Chase stages
($\Delta z \approx 1\text{--}2\,\text{m}$) may approach this limit, but the cross-spectrum amplitude
drops below `WLS_AMP_THR` before the worst wrapping points, so wrapped cells are
largely excluded automatically.

---

### Per-pair kz band: `MANUAL_KZ_BAND_FAC`

| Stage | Pair | `_kz_fac` | Reason |
|---|---|---|---|
| Push | 1 → 3 | 0.5 (default) | Strong coherent signal; kz-phase ramp is linear |
| Chase | 3 → 8 | 0.5 (default) | Good linearity across the kz band |
| Wait | 8 → 20 | **0.35** | S-curve artifact: restrict to $|k_z| < 0.35\,k_{z,c}$ |
| Pull | 20 → 38 | **0.45** | Oscillatory 1-D profile: same restriction |

---

### Per-pair kx band: `MANUAL_KX_BAND_FAC`

| Stage | Pair | `_kx_fac` | Reason |
|---|---|---|---|
| Chase | 3 → 8 | **1.4** | Off-lobe kx cells at $|k_x| > 8.8\,\text{rad/m}$ bias the sign of $\Delta x$ |

---

### Amplitude weight exponent (`WLS_POW`)

The WLS minimises $\sum_i w_i (\hat\varphi_i - \varphi_i)^2$ where $w_i = |\mathrm{XS}_i|^p$
and $p$ is `WLS_POW`.

| `WLS_POW` | Effect |
|---|---|
| 1 | Linear amplitude weights — standard amplitude-weighted least squares. |
| 2 | Squared: a cell at 50 % of peak gets $\tfrac{1}{4}$ relative influence. |
| 3 | Cubed (current): the same cell gets $\tfrac{1}{8}$ the influence. |

**Why cubing is needed here:** the signal energy is concentrated in two narrow lobes at
$k_x \approx \pm 8\,\text{rad/m},\; k_z \approx 0$.
Between and beyond these lobes there are many k-cells with moderate amplitude
(passing the `WLS_AMP_THR` gate) that carry no coherent displacement signal.
With linear weights these peripheral cells collectively outweigh the small number of
dominant signal cells, biasing — and sometimes flipping the sign of — the fitted slope.
Cubing the weights multiplies the signal-to-peripheral influence ratio by
$(w_\text{signal}/w_\text{peripheral})^2$,
concentrating the fit on the high-amplitude lobe cells where the displacement
information actually lives.

---

### Residual phase in the 1-D diagnostic profiles

The raw cross-spectrum phase at $(k_z, k_x)$ is:
$$\varphi(k_z, k_x) = k_z \,\Delta z + k_x \,\Delta x + \varphi_0$$

Collapsing to a 1-D profile by averaging over one axis mixes both displacement
components, making it hard to judge fit quality in each direction separately.
A large $k_z \Delta z$ term causes phase wrapping at cells away from $k_z = 0$,
producing a systematic bias in the kx-marginal average.

**Partial residuals.** Before averaging, the contribution of the *other* axis is
subtracted using the current WLS estimates:

| Profile | Residual computed | Remaining slope | Fitted line overlay |
|---|---|---|---|
| 1-D $k_z$ (panel 4) | $\varphi - k_x\,\hat\Delta x$ | $k_z\,\Delta z + \varphi_0$ | $k_z\,\hat\Delta z + \hat\varphi_0$ |
| 1-D $k_x$ (panel 5) | $\varphi - k_z\,\hat\Delta z$ | $k_x\,\Delta x + \varphi_0$ | $k_x\,\hat\Delta x + \hat\varphi_0$ |

With residuals each profile shows only its own axis contribution.
The fitted line **should pass through the high-weight data points**.
If it does not — particularly if the slope is reversed — the WLS plane is being
pulled by low-amplitude peripheral cells; try increasing `WLS_POW` or narrowing
the band with `MANUAL_KX_BAND_FAC` / `MANUAL_KZ_BAND_FAC`.

---

### Interpreting the diagnostic panels

**Panel 1 — Cross-spectrum phase:** amplitude-masked phase in $(k_z, k_x)$ space.
Two lobes at $k_x \approx \pm 8\,\text{rad/m}$ carry opposite sign when $\Delta x \neq 0$.
A kz-dependent phase ramp within each lobe indicates $\Delta z$.

**Panel 2 — Energy:** $|\mathrm{XS}|$ inside the band. Cyan contour = `WLS_AMP_THR` level.
Points outside the contour are excluded from the WLS fit.

**Panel 3 — Fitted plane:** the WLS result $k_z\Delta z + k_x\Delta x + \varphi_0$.
Should resemble the phase panel if the linear model is a good fit.

**Panel 4 — 1-D kz profile (kx-corrected residual):** weighted mean of $\varphi - k_x\hat\Delta x$
collapsed over $k_x$. Dot colour = amplitude weight (viridis: purple = low, yellow = high).
A linear slope matching the red line confirms $\Delta z$.

| 1-D kz pattern | Cause | Interpretation |
|---|---|---|
| Linear ramp, slope matches fit | Good coherent signal | $\Delta z$ reliable |
| S-curve $(-k_z^3 + k_z)$ through origin | Two anti-phase kx lobes cancel unevenly | $\Delta z \approx 0$ |
| Oscillatory, multiple extrema | High-kz incoherence from long time gap | Reduce `MANUAL_KZ_BAND_FAC` |
| Flat scatter, no trend | Near-zero SNR | Treat displacement as $\approx 0$ |

**Panel 5 — 1-D kx profile (kz-corrected residual):** weighted mean of $\varphi - k_z\hat\Delta z$
collapsed over $k_z$. Slope $\approx \Delta x$.
If the high-weight dots follow a different slope than the red line,
peripheral cells are dominating — see `WLS_POW` and `MANUAL_KX_BAND_FAC`.

---

### Physical trajectory summary

| Phase | Profiles | Expected $\Delta z$ | Expected $\Delta x$ |
|---|---|---|---|
| Pushing | 1→3 | $> 0$ (fluid injected downward along fracture) | Small |
| Chasing | 3→8 | $> 0$ (continued downward migration) | Small |
| Waiting | 8→20 | $\approx 0$ (no pumping) | $\approx 0$ |
| Pulling | 20→38 | $< 0$ (fluid partially returns upward) | Opposite to push |


# Gazdag results

In [12]:
# ── Manual pair selection + ROI definition ──────────────────────────────────────────
#
# MANUAL_PAIRS  — list of (run_a, run_b) to analyse.
#
# COMPARE_TO    — None  → use pairs as written (run_a vs run_b)
#                 int   → override run_a for every pair to this run number,
#                         so you compare every profile against one fixed reference.
#                         Useful when consecutive-pair shifts are too small to detect:
#                         e.g. COMPARE_TO=1 gives (1,4), (1,8), (1,10), ...
#
# MANUAL_ROIS   — per-pair ROI in physical metres.
#                 Key is the ORIGINAL (run_a, run_b) regardless of COMPARE_TO.
#                 Format: (z_min_m, z_max_m, x_min_m, x_max_m)
#                 None → auto-detect from monogenic envelope.
#
# SHOW_DIAG     — True: plot cross-spectrum phase + fitted plane for every pair.
#                 Helps diagnose whether there is any detectable kx slope.
#
# ROI_METHOD    — 'kirchhoff' or 'gazdag'.
# FORCE_DZ_ZERO — True: fit only Δx + φ₀ (lateral fluid front).
# ROI_THRESH    — envelope threshold for auto-detected pairs.
# ────────────────────────────────────────────────────────────────────────────────────

MANUAL_PAIRS = [
    # (1,  2),
    # (2,  3),
    # (3,  4),
    # (4,  5),
    # (5,  7),
    # (7,  8),
    # (8,  9),
    # (9,  10),
    # (10, 11),
    # (11,  12),
    # (12,  13),
    # (13,  14),
    # (14,  15),
    # (15,  16),
    # (16,  17),
    # (17,  18),
    # (18,  19),
    # (19,  20),
    # (20,  21),
    # (21,  22),
    # (22,  23),
    # (23,  24),
    # (24,  25),
    # (25,  26),
    # (26,  27),
    # (27,  28),
    # (28,  29),
    # (29,  30),
    # (30,  31),
    # (31,  32),
    # (32,  33),
    # (33,  34),
    # (34,  35),
    # (35,  36),
    # (36,  37),
    # (37,  38),
    (1,3), #(1,3),
    (3,8), #(4,8),
    (8,20), #(9,20),
    (20,38), #(21,38),
]

COMPARE_TO = None         # all pairs vs prof1 (first injection measurement = earliest non-zero migration)

ROI = (70, 79, 4.5, 7.5) # shows pushing-chasing-waiting-pulling correctly with 1.5 * kz_c, w > 0.10
                         # 1.5 * kz_c, w > 0.25 shows stronger drop in pulling phase
MANUAL_ROIS = {
    # (1,  2):  ROI,
    # (2,  3):  ROI,
    # (3,  4):  ROI,
    # (4,  5):  ROI,
    # (5,  7):  ROI,
    # (7,  8):  ROI,
    # (8,  9):  ROI,
    # (9,  10): ROI,
    # (10,  11): ROI,
    # (11,  12): ROI,
    # (12,  13): ROI,
    # (13,  14): ROI,
    # (14,  15): ROI,
    # (15,  16): ROI,
    # (16,  17): ROI,
    # (17,  18): ROI,
    # (18,  19): ROI,
    # (19,  20): ROI,
    # (20,  21): ROI,
    # (21,  22): ROI,
    # (22,  23): ROI,
    # (23,  24): ROI,
    # (24,  25): ROI,
    # (25,  26): ROI,
    # (26,  27): ROI,
    # (27,  28): ROI,
    # (28,  29): ROI,
    # (29,  30): ROI,
    # (30,  31): ROI,
    # (31,  32): ROI,
    # (32,  33): ROI,
    # (33,  34): ROI,
    # (34,  35): ROI,
    # (35,  36): ROI,
    # (36,  37): ROI,
    # (37,  38): ROI,
    (1,3): (71,78,5.0,6.5), #(1,3): (71,76,5.0,6.5),
    (3,8): (70,78,4.5,7.5), #(4,8): (70,78,5.0,7.0),
    (8,20): (70,77,4.5,6.0), #(9,20): (72,77,5.0,6.0),
    (20,38): (70,78,4.5,6.5), #(21,38): (70,77,5.0,6.0),
}

ROI_METHOD    = 'gazdag'
FORCE_DZ_ZERO = False
ROI_THRESH    = 0.5
SHOW_DIAG     = True   # set False to skip cross-spectrum phase plots
# KZ_BAND_FAC   = 1.6    # band: |k| < KZ_BAND_FAC * kz_c  (WLS fit + 1-D profile display)
# WLS_AMP_THR   = 0.25   # WLS mask: discard k-cells where |XS| < WLS_AMP_THR * max
KZ_BAND_FAC = 0.5    # default kz band; overridden per-pair by MANUAL_KZ_BAND_FAC
KX_BAND_FAC = 2.0    # kx band: |kx| < KX_BAND_FAC * kz_c (must cover the ±8 rad/m lobes)
WLS_AMP_THR = 0.20
WLS_POW     = 3     # weight exponent: cubed amplitude suppresses low-energy peripheral cells
MANUAL_KZ_BAND_FAC = {   # per-pair kz band override (reduces kz range for noisy stages)
    (1,  3): 0.5,      # Push: S-curve in 1-D profile — restrict to coherent |kz| < 1.6 rad/m
    (8,  20): 0.35,      # Wait: S-curve in 1-D profile — restrict to coherent |kz| < 1.6 rad/m
    (20, 38): 0.45,      # Pull: oscillatory 1-D profile — same restriction
}
MANUAL_KX_BAND_FAC = {   # per-pair kx band override (narrows kx range around signal lobes)
    (3,  8): 2.0,      # Chase: off-lobe kx cells bias sign of Dx — focus to |kx| < 8.8 rad/m
}

# ────────────────────────────────────────────────────────────────────────────────────

def _phys_to_pix(z_common, x_img_arr, z_min, z_max, x_min, x_max):
    """Convert physical ROI bounds (metres) to pixel indices.
    z_common is DECREASING (row 0 = 85 m deepest).
    Returns (z0, z1, x0, x1), z1/x1 exclusive.
    """
    rows = np.where((z_common >= z_min) & (z_common <= z_max))[0]
    cols = np.where((x_img_arr >= x_min) & (x_img_arr <= x_max))[0]
    if rows.size == 0 or cols.size == 0:
        raise ValueError(
            f'ROI ({z_min}–{z_max} m depth, {x_min}–{x_max} m radial) '
            f'does not intersect the image grid. '
            f'Depth: [{z_common[-1]:.1f}, {z_common[0]:.1f}] m  '
            f'Radial: [{x_img_arr[0]:.2f}, {x_img_arr[-1]:.2f}] m'
        )
    return int(rows[0]), int(rows[-1]) + 1, int(cols[0]), int(cols[-1]) + 1

roi_out = OUT_DIR / 'roi_phase'
roi_out.mkdir(exist_ok=True)

dz_g = dL
dx_g = float(x_img[1] - x_img[0])
kz_c = 2.0 * np.pi * f0_mig / v

def _load_img(method, run):
    p = OUT_DIR / 'migrated' / f'{method}_{run}.npy'
    return np.load(p) if p.exists() else None

results = {}
for run_a, run_b in MANUAL_PAIRS:
    ref_run = COMPARE_TO if COMPARE_TO is not None else run_a
    img_a = _load_img(ROI_METHOD, ref_run)
    img_b = _load_img(ROI_METHOD, run_b)
    if img_a is None or img_b is None:
        print(f'[{ref_run}→{run_b}] Missing .npy — skipping.')
        continue

    n       = min(img_a.shape[0], img_b.shape[0])
    img_a   = img_a[:n];  img_b = img_b[:n]
    z_common = depth[:n]
    diff     = img_b - img_a
    env      = _monogenic_envelope(diff)

    # ROI — look up by original key (run_a, run_b) first, then (ref_run, run_b)
    roi_spec = MANUAL_ROIS.get((run_a, run_b)) or MANUAL_ROIS.get((ref_run, run_b))
    if roi_spec is not None:
        z0, z1, x0, x1 = _phys_to_pix(z_common, x_img, *roi_spec)
        roi_source = 'manual'
    else:
        z0, z1, x0, x1 = _roi_from_envelope(env, ROI_THRESH)
        roi_source = f'auto (thresh={ROI_THRESH})'

    z_roi_top = float(z_common[z0]);     z_roi_bot = float(z_common[z1 - 1])
    x_roi_lo  = float(x_img[x0]);        x_roi_hi  = float(x_img[x1 - 1])

    # ── Difference + envelope figure ────────────────────────────────────────────
    extent = [x_img[0], x_img[-1], z_common[-1], z_common[0]]
    vmax_d = np.percentile(np.abs(diff), 98)
    vmax_e = np.percentile(env, 98)

    fig, (ax_d, ax_e) = plt.subplots(1, 2, figsize=(14, 5))
    im_d = ax_d.imshow(diff, aspect='auto', cmap='RdBu_r',
                        extent=extent, origin='upper',
                        vmin=-vmax_d, vmax=vmax_d)
    plt.colorbar(im_d, ax=ax_d, label='Δ amplitude [a.u.]')
    ax_d.invert_yaxis()
    ax_d.add_patch(Rectangle((x_roi_lo, z_roi_bot),
                              width=x_roi_hi - x_roi_lo, height=z_roi_top - z_roi_bot,
                              lw=1.5, edgecolor='yellow', facecolor='none'))
    ax_d.xaxis.set_major_locator(ticker.MultipleLocator(0.5))
    ax_d.yaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_d.grid(True, color='white', lw=0.3, alpha=0.4)
    ax_d.set_title(f'{ROI_METHOD.capitalize()} diff: prof_{run_b} − prof_{ref_run}  [{roi_source}]')
    ax_d.set_xlabel('Radial distance (m)');  ax_d.set_ylabel('Depth (m)')

    im_e = ax_e.imshow(env, aspect='auto', cmap='inferno',
                        extent=extent, origin='upper', vmin=0, vmax=vmax_e)
    plt.colorbar(im_e, ax=ax_e, label='Monogenic envelope [a.u.]')
    ax_e.invert_yaxis()
    ax_e.add_patch(Rectangle((x_roi_lo, z_roi_bot),
                              width=x_roi_hi - x_roi_lo, height=z_roi_top - z_roi_bot,
                              lw=1.5, edgecolor='cyan', facecolor='none'))
    ax_e.xaxis.set_major_locator(ticker.MultipleLocator(0.5))
    ax_e.yaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_e.grid(True, color='white', lw=0.3, alpha=0.4)
    ax_e.set_title(f'Monogenic envelope  [{roi_source}]')
    ax_e.set_xlabel('Radial distance (m)');  ax_e.set_ylabel('Depth (m)')

    plt.tight_layout()
    fig.savefig(roi_out / f'{ROI_METHOD}_roi_{ref_run}_to_{run_b}.png', dpi=150)
    plt.show();  plt.close(fig)

    # ── WLS phase-plane fit (zero-padded for denser kx/kz sampling) ─────────────
    # Extend ROI by DATA_PAD pixels of real data on each side so the taper
    # rolls off through surrounding data rather than attenuating the ROI signal.
    DATA_PAD = 8
    _z0p = max(0, z0 - DATA_PAD);  _z1p = min(img_a.shape[0], z1 + DATA_PAD)
    _x0p = max(0, x0 - DATA_PAD);  _x1p = min(img_a.shape[1], x1 + DATA_PAD)
    base_crop = img_a[_z0p:_z1p, _x0p:_x1p]
    mon_crop  = img_b[_z0p:_z1p, _x0p:_x1p]
    Nz, Nx    = base_crop.shape
    pad_fac   = 10
    Nz_pad, Nx_pad = Nz * pad_fac, Nx * pad_fac

    kz_ax = np.fft.fftfreq(Nz_pad, d=dz_g) * 2 * np.pi
    kx_ax = np.fft.fftfreq(Nx_pad, d=dx_g) * 2 * np.pi
    KZ, KX = np.meshgrid(kz_ax, kx_ax, indexing='ij')

    # Taper = 1 over the ROI interior, sin²-ramp over the data-padding margin.
    def _edge_taper_1d(N, n_lo, n_hi):
        w = np.ones(N)
        if n_lo > 0:
            w[:n_lo] = np.sin(np.linspace(0, np.pi / 2, n_lo)) ** 2
        if n_hi > 0:
            w[-n_hi:] = np.sin(np.linspace(np.pi / 2, 0, n_hi)) ** 2
        return w
    _nz_lo = z0 - _z0p;  _nz_hi = _z1p - z1
    _nx_lo = x0 - _x0p;  _nx_hi = _x1p - x1
    taper = np.outer(_edge_taper_1d(Nz, _nz_lo, _nz_hi),
                     _edge_taper_1d(Nx, _nx_lo, _nx_hi))
    XS     = (np.fft.fft2(base_crop * taper, s=(Nz_pad, Nx_pad)) *
               np.conj(np.fft.fft2(mon_crop * taper, s=(Nz_pad, Nx_pad))))
    w      = np.abs(XS);  phi = np.angle(XS)
    _kz_fac = (MANUAL_KZ_BAND_FAC.get((run_a, run_b))
               or MANUAL_KZ_BAND_FAC.get((ref_run, run_b))
               or KZ_BAND_FAC)
    _kx_fac = (MANUAL_KX_BAND_FAC.get((run_a, run_b))
               or MANUAL_KX_BAND_FAC.get((ref_run, run_b))
               or KX_BAND_FAC)
    band   = (np.abs(KZ) < _kz_fac * kz_c) & (np.abs(KX) < _kx_fac * kz_c)
    mask   = (w > WLS_AMP_THR * w.max()) & band & ((np.abs(KZ) + np.abs(KX)) > 0)

    n_mask = int(mask.sum())
    if n_mask < 3:
        print(f'[{ref_run}→{run_b}] WARNING: only {n_mask} pixels pass the WLS mask '
              f'(crop {Nz}×{Nx} px, padded to {Nz_pad}×{Nx_pad}).  ROI may be too small or SNR too low.')
        dx_est = phi_0 = dz_est = 0.0
    else:
        W = w[mask] ** WLS_POW
        if FORCE_DZ_ZERO:
            A = np.column_stack([KX[mask], np.ones(n_mask)])
        else:
            A = np.column_stack([KZ[mask], KX[mask], np.ones(n_mask)])
        c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
        if FORCE_DZ_ZERO:
            dz_est, dx_est, phi_0 = 0.0, float(c[0]), float(c[1])
        else:
            dz_est, dx_est, phi_0 = float(c[0]), float(c[1]), float(c[2])

    # ── Diagnostic: cross-spectrum phase + fitted plane ──────────────────────────
    if SHOW_DIAG and n_mask >= 3:
        # Shift to centre-zero frequency for display
        phi_shift = np.fft.fftshift(phi)
        w_shift   = np.fft.fftshift(w)
        kz_disp   = np.fft.fftshift(kz_ax)
        kx_disp   = np.fft.fftshift(kx_ax)

        # Fitted plane at each pixel
        fitted = KX * dx_est + KZ * dz_est + phi_0
        fitted_shift = np.fft.fftshift(fitted)

        # 1-D kz profile: subtract kx contribution first so the slope shows Dz cleanly
        phi_kz_resid = phi - KX * dx_est
        phi_1d  = np.zeros(Nz_pad)
        w_1d    = np.zeros(Nz_pad)
        band_kx = np.abs(kx_ax) < _kx_fac * kz_c
        for i_row in range(Nz_pad):
            sel = band_kx & (w[i_row, :] > WLS_AMP_THR * w.max())
            if sel.sum() > 0:
                phi_1d[i_row] = np.average(phi_kz_resid[i_row, :][sel], weights=w[i_row, :][sel])
                w_1d[i_row]   = w[i_row, :][sel].sum()
        kz_1d_s  = np.fft.fftshift(kz_ax)
        phi_1d_s = np.fft.fftshift(phi_1d)
        w_1d_s   = np.fft.fftshift(w_1d)

        # 1-D kx profile: subtract kz contribution first so the slope shows Dx cleanly
        phi_kx_resid = phi - KZ * dz_est
        phi_1d_kx   = np.zeros(Nx_pad)
        w_1d_kx     = np.zeros(Nx_pad)
        band_kz_fit = np.abs(kz_ax) < _kz_fac * kz_c
        for j_col in range(Nx_pad):
            sel_kz = band_kz_fit & (w[:, j_col] > WLS_AMP_THR * w.max())
            if sel_kz.sum() > 0:
                phi_1d_kx[j_col] = np.average(phi_kx_resid[:, j_col][sel_kz],
                                               weights=w[:, j_col][sel_kz])
                w_1d_kx[j_col]   = w[:, j_col][sel_kz].sum()
        kx_1d_s     = np.fft.fftshift(kx_ax)
        phi_1d_kx_s = np.fft.fftshift(phi_1d_kx)
        w_1d_kx_s   = np.fft.fftshift(w_1d_kx)

        fig_d, axes_d = plt.subplots(1, 5, figsize=(28, 4))

        # Panel 1: cross-spectrum phase (fftshifted, band only)
        band_shift = np.fft.fftshift(band)
        phi_masked = np.where(band_shift & (np.fft.fftshift(w) > WLS_AMP_THR * w.max()),
                              phi_shift, np.nan)
        im_ph = axes_d[0].imshow(phi_masked, aspect='auto', cmap='RdBu_r',
                                  extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]],
                                  vmin=-np.pi, vmax=np.pi, origin='upper')
        plt.colorbar(im_ph, ax=axes_d[0], label='phase [rad]')
        axes_d[0].set_title(f'Cross-spectrum phase  (mask: {n_mask} px)')
        axes_d[0].set_xlabel('kx [rad/m]');  axes_d[0].set_ylabel('kz [rad/m]')
        axes_d[0].set_xlim(kx_disp[0] / 3, kx_disp[-1] / 3)
        axes_d[0].set_ylim(kz_disp[-1] / 3, kz_disp[0] / 3)

        # Panel 2: cross-spectrum energy with WLS amplitude threshold contour
        w_plot = np.where(band_shift, w_shift, np.nan)
        im_en = axes_d[1].imshow(w_plot, aspect='auto', cmap='inferno',
                                  extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]],
                                  origin='upper')
        plt.colorbar(im_en, ax=axes_d[1], label='|XS| [a.u.]')
        axes_d[1].contour(kx_disp, kz_disp, w_shift,
                          levels=[WLS_AMP_THR * w.max()], colors='cyan', linewidths=0.8)
        axes_d[1].set_title(f'Energy  (thr={WLS_AMP_THR:.2f}×max — cyan contour)')
        axes_d[1].set_xlabel('kx [rad/m]');  axes_d[1].set_ylabel('kz [rad/m]')
        axes_d[1].set_xlim(kx_disp[0] / 3, kx_disp[-1] / 3)
        axes_d[1].set_ylim(kz_disp[-1] / 3, kz_disp[0] / 3)

        # Panel 3: fitted plane
        fitted_masked = np.where(band_shift, fitted_shift, np.nan)
        im_fit = axes_d[2].imshow(fitted_masked, aspect='auto', cmap='RdBu_r',
                                   extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]],
                                   vmin=-np.pi, vmax=np.pi, origin='upper')
        plt.colorbar(im_fit, ax=axes_d[2], label='phase [rad]')
        axes_d[2].set_title(f'Fitted plane  Δz={dz_est:+.4f} m  Δx={dx_est:+.4f} m  φ₀={phi_0:+.3f} rad')
        axes_d[2].set_xlabel('kx [rad/m]');  axes_d[2].set_ylabel('kz [rad/m]')
        axes_d[2].set_xlim(kx_disp[0] / 3, kx_disp[-1] / 3)
        axes_d[2].set_ylim(kz_disp[-1] / 3, kz_disp[0] / 3)

        # Panel 4: 1-D kz slice — measured vs fitted
        in_band = np.abs(kz_1d_s) < _kz_fac * kz_c
        sc_kz = axes_d[3].scatter(kz_1d_s[in_band & (w_1d_s > 0)],
                          phi_1d_s[in_band & (w_1d_s > 0)],
                          c=w_1d_s[in_band & (w_1d_s > 0)],
                          cmap='viridis', s=20, label='measured (weighted)')
        plt.colorbar(sc_kz, ax=axes_d[3], label='weight [a.u.]')
        kz_fit = kz_1d_s[in_band]
        axes_d[3].plot(kz_fit, kz_fit * dz_est + phi_0, 'r-', lw=1.5, label='fitted slope')
        axes_d[3].axhline(0, color='k', lw=0.5, ls='--')
        axes_d[3].set_xlabel('kz [rad/m]');  axes_d[3].set_ylabel('phase [rad]')
        axes_d[3].set_title('1-D kz profile  (kx-corrected residual, slope = Δz)')
        axes_d[3].legend(fontsize=8)

        # Panel 5: 1-D kx slice — measured vs fitted
        in_band_kx = np.abs(kx_1d_s) < _kx_fac * kz_c
        sc_kx = axes_d[4].scatter(kx_1d_s[in_band_kx & (w_1d_kx_s > 0)],
                          phi_1d_kx_s[in_band_kx & (w_1d_kx_s > 0)],
                          c=w_1d_kx_s[in_band_kx & (w_1d_kx_s > 0)],
                          cmap='viridis', s=20, label='measured (weighted)')
        plt.colorbar(sc_kx, ax=axes_d[4], label='weight [a.u.]')
        kx_fit = kx_1d_s[in_band_kx]
        axes_d[4].plot(kx_fit, kx_fit * dx_est + phi_0, 'r-', lw=1.5, label='fitted slope')
        axes_d[4].axhline(0, color='k', lw=0.5, ls='--')
        axes_d[4].set_xlabel('kx [rad/m]');  axes_d[4].set_ylabel('phase [rad]')
        axes_d[4].set_title('1-D kx profile  (kz-corrected residual, slope = Δx)')
        axes_d[4].legend(fontsize=8)
        axes_d[4].set_xlim(-_kx_fac * kz_c, _kx_fac * kz_c)

        plt.suptitle(f'Diagnostic: prof_{ref_run} → prof_{run_b}', y=1.02)
        plt.tight_layout()
        fig_d.savefig(roi_out / f'{ROI_METHOD}_diag_{ref_run}_to_{run_b}.png',
                      dpi=150, bbox_inches='tight')
        plt.show();  plt.close(fig_d)

    results[(ref_run, run_b)] = {
        'dx': dx_est, 'dz': dz_est, 'phi_0': phi_0,
        'roi_m':  (z_roi_bot, z_roi_top, x_roi_lo, x_roi_hi),
        'roi_px': (z0, z1, x0, x1),
        'n_mask': n_mask, 'source': roi_source,
    }
    print(f'prof_{ref_run}→{run_b}  [{roi_source}]:  '
          f'Δx={dx_est:+.4f} m  Δz={dz_est:+.4f} m  φ₀={phi_0:+.4f} rad  '
          f'mask={n_mask} px  crop={Nz}×{Nx} px  '
          f'ROI depth=[{z_roi_bot:.1f},{z_roi_top:.1f}] m  '
          f'radial=[{x_roi_lo:.2f},{x_roi_hi:.2f}] m')

# ── Summary plot ─────────────────────────────────────────────────────────────────────
if results:
    pair_labels = [f'{a}→{b}' for a, b in results]
    dx_vals  = [results[k]['dx']    for k in results]
    phi_vals = [results[k]['phi_0'] for k in results]

    fig2, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
    dz_vals = [results[k]['dz'] for k in results]
    from matplotlib.patches import Patch as _Patch
    _PHASE_DEFS = [('Pushing', 1, 3, '#aed6f1'), ('Chasing', 4, 8, '#a9dfbf'),
                   ('Waiting', 9, 20, '#f9e79f'), ('Pulling', 21, 38, '#f1948a')]
    _rn_col = {n: c for _nm, lo, hi, c in _PHASE_DEFS for n in range(lo, hi + 1)}
    _pkeys  = list(results.keys())
    for _bax in (ax1, ax2):
        for _i, _k in enumerate(_pkeys):
            _c = _rn_col.get(_k[1])
            if _c:
                _bax.axvspan(_i - 0.5, _i + 0.5, color=_c, alpha=0.3, zorder=0, lw=0)
    _ph_hdl = [_Patch(facecolor=c, alpha=0.6, label=nm, edgecolor='grey', lw=0.5)
               for nm, lo, hi, c in _PHASE_DEFS
               if any(_rn_col.get(_k[1]) == c for _k in _pkeys)]
    ax1.legend(handles=_ph_hdl, loc='best', fontsize=8, framealpha=0.7)
    ax1.plot(dx_vals, 'o-', color='steelblue')
    ax1.axhline(0, color='k', lw=0.6, ls='--')
    ax1.set_ylabel('Δx (radial) [m]')
    mode_str = f'vs prof_{COMPARE_TO}' if COMPARE_TO is not None else 'consecutive pairs'
    ax1.set_title(f'WLS fluid-front shift — {ROI_METHOD.capitalize()} ({mode_str})')
    ax2.plot(dz_vals, '^-', color='seagreen')
    ax2.axhline(0, color='k', lw=0.6, ls='--')
    ax2.set_ylabel('Δz (depth) [m]')
    plt.tight_layout()
    fig2.savefig(roi_out / f'{ROI_METHOD}_dx_summary.png', dpi=150)
    plt.show();  plt.close(fig2)

print(f'\nDone — {len(results)} pair(s).  Output: {roi_out}')

C:\Users\Administrator\AppData\Local\Temp\ipykernel_7332\3864488137.py:224: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show();  plt.close(fig)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_7332\3864488137.py:396: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show();  plt.close(fig_d)


prof_1→3  [manual]:  Δx=-0.3081 m  Δz=+1.7222 m  φ₀=-0.0000 rad  mask=676 px  crop=156×50 px  ROI depth=[71.0,78.0] m  radial=[5.03,6.47] m
prof_3→8  [manual]:  Δx=-0.1134 m  Δz=+1.0462 m  φ₀=+0.0000 rad  mask=1300 px  crop=176×85 px  ROI depth=[70.0,78.0] m  radial=[4.50,7.47] m
prof_8→20  [manual]:  Δx=+0.0433 m  Δz=-0.4334 m  φ₀=-0.0000 rad  mask=852 px  crop=156×51 px  ROI depth=[70.0,77.0] m  radial=[4.50,5.99] m
prof_20→38  [manual]:  Δx=+0.1175 m  Δz=-0.2133 m  φ₀=+0.0000 rad  mask=1116 px  crop=176×62 px  ROI depth=[70.0,78.0] m  radial=[4.50,6.47] m

Done — 4 pair(s).  Output: c:\Users\Administrator\OneDrive\Thesis\TimeLapse_Notebooks\fielddata\output\roi_phase


C:\Users\Administrator\AppData\Local\Temp\ipykernel_7332\3864488137.py:442: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show();  plt.close(fig2)


## Sliding-Window ROI Scan (Gazdag)

Instead of one manually-drawn ROI per pair, slide a fixed-size window across the
whole difference B-scan and run the same WLS phase-plane fit in every window
position. This trades the hand-picked rectangle above for a 2-D map of the fit,
so spatial patterns in the estimated displacement become visible instead of a
single averaged number per pair.

- **Window size** — set as a multiple of the dominant wavelength
  (`WINDOW_LAMBDA_FRAC`), so it scales automatically if `f0_mig` or `v` change.
- **Stride** — `STRIDE_Z_PX` / `STRIDE_X_PX` pixels the window centre advances
  each step. `None` defaults to a non-overlapping stride (window width) to keep
  the scan fast; shrink it for a finer map once the QC figure below confirms the
  window size looks reasonable.
- **Output** — two maps, **Δz** (depth / vertical) and **Δx** (radial /
  lateral), plus a QC figure with the scanned window grid drawn on the
  difference B-scan so you can judge if the window is too big or too small.


In [ ]:
# ── Sliding-window ROI scan: Δz / Δx displacement maps (Gazdag) ────────────────────
#
# WINDOW_Z_LAMBDA_FRAC / WINDOW_X_LAMBDA_FRAC — window depth/radial extent, each as a
#                       multiple of the dominant wavelength (2*pi/kz_c). These are
#                       independent because the signal is anisotropic: reflectors are
#                       sub-horizontal (kz ~ 0, long coherence length in depth) while
#                       the radial lobes sit at kx ~ +/-8 rad/m (short wavelength,
#                       ~0.8 m) -- the manually-tuned ROIs above reflect this, running
#                       ~7-8 m deep but only ~1.5-3 m radial. A window narrower than
#                       ~1 radial wavelength cannot resolve a stable phase gradient in
#                       x, and just fits noise (this is what produced salt-and-pepper
#                       dz/dx maps with WINDOW_LAMBDA_FRAC=0.5 in an earlier version).
#                       Defaults below reproduce the manually-tuned ROI footprint.
# STRIDE_Z_PX / STRIDE_X_PX — pixels the window centre advances each step.
#                       None -> full window width (non-overlapping), which keeps the
#                       scan fast. Shrink once the QC figure confirms the window size,
#                       and only once WINDOW_*_LAMBDA_FRAC is large enough to give a
#                       stable per-window fit -- a fine stride cannot fix a window
#                       that's fundamentally too small.
# SCAN_Z_RANGE_M / SCAN_X_RANGE_M — physical sub-region to scan (None -> full domain).
# WIN_MIN_MASK_PX   — minimum WLS-mask pixel count; windows below this are left as
#                      NaN in the maps (too few k-cells passed the amplitude gate).
# WIN_ENERGY_THR    — the WLS amplitude gate is relative to *each window's own* peak,
#                      so a window sitting in pure noise still clears WIN_MIN_MASK_PX
#                      with a "confident" but meaningless fit. WIN_ENERGY_THR adds an
#                      absolute gate: a window's mean |Δ amplitude| must reach this
#                      fraction of the pair's global 98th-percentile |Δ| to be trusted.
#                      Windows failing either gate are left as NaN in the maps.
# Other WIN_* params mirror the manual-ROI WLS settings above (band factors, amplitude
# threshold, weight exponent) so the fit itself is unchanged — only the ROI placement
# is now automatic.
# ────────────────────────────────────────────────────────────────────────────────────

WINDOW_Z_LAMBDA_FRAC = 7.5   # -> ~7.5 m depth, matching the manual ROIs' 7-8 m
WINDOW_X_LAMBDA_FRAC = 2.0   # -> ~2.0 m radial, matching the manual ROIs' 1.5-3 m
STRIDE_Z_PX = None
STRIDE_X_PX = None
SCAN_Z_RANGE_M = None
SCAN_X_RANGE_M = None
WIN_DATA_PAD_PX = 8
WIN_KZ_BAND_FAC = KZ_BAND_FAC
WIN_KX_BAND_FAC = KX_BAND_FAC
WIN_AMP_THR = WLS_AMP_THR
WIN_WLS_POW = WLS_POW
WIN_PAD_FAC = 6
WIN_MIN_MASK_PX = 3
WIN_ENERGY_THR = 0.15
WIN_FORCE_DZ_ZERO = FORCE_DZ_ZERO
QC_PAIR_INDEX = 0   # which MANUAL_PAIRS entry to draw the QC window-grid figure for

wavelength_m = 2.0 * np.pi / kz_c
window_z_m   = WINDOW_Z_LAMBDA_FRAC * wavelength_m
window_x_m   = WINDOW_X_LAMBDA_FRAC * wavelength_m
print(f'Dominant wavelength = {wavelength_m:.3f} m  ->  window = {window_z_m:.2f} m (depth, '
      f'{WINDOW_Z_LAMBDA_FRAC:.1f} x lambda) x {window_x_m:.2f} m (radial, '
      f'{WINDOW_X_LAMBDA_FRAC:.1f} x lambda)')


def _edge_taper_1d(N, n_lo, n_hi):
    w = np.ones(N)
    if n_lo > 0:
        w[:n_lo] = np.sin(np.linspace(0, np.pi / 2, n_lo)) ** 2
    if n_hi > 0:
        w[-n_hi:] = np.sin(np.linspace(np.pi / 2, 0, n_hi)) ** 2
    return w


def _window_wls_fit(img_a, img_b, z0, z1, x0, x1, data_pad, dz_g, dx_g, kz_c,
                     kz_band_fac, kx_band_fac, amp_thr, wls_pow, pad_fac,
                     force_dz_zero, min_mask_px):
    """WLS phase-plane fit on one window, padded with real data + edge taper
    (same recipe as the manual-ROI cell above). Returns (dz, dx, n_mask)."""
    Nz_img, Nx_img = img_a.shape
    z0p, z1p = max(0, z0 - data_pad), min(Nz_img, z1 + data_pad)
    x0p, x1p = max(0, x0 - data_pad), min(Nx_img, x1 + data_pad)
    base_crop = img_a[z0p:z1p, x0p:x1p]
    mon_crop  = img_b[z0p:z1p, x0p:x1p]
    Nz, Nx    = base_crop.shape
    Nz_pad, Nx_pad = Nz * pad_fac, Nx * pad_fac

    taper = np.outer(_edge_taper_1d(Nz, z0 - z0p, z1p - z1),
                      _edge_taper_1d(Nx, x0 - x0p, x1p - x1))
    XS = (np.fft.fft2(base_crop * taper, s=(Nz_pad, Nx_pad)) *
          np.conj(np.fft.fft2(mon_crop * taper, s=(Nz_pad, Nx_pad))))
    w, phi = np.abs(XS), np.angle(XS)

    kz_ax = np.fft.fftfreq(Nz_pad, d=dz_g) * 2 * np.pi
    kx_ax = np.fft.fftfreq(Nx_pad, d=dx_g) * 2 * np.pi
    KZ, KX = np.meshgrid(kz_ax, kx_ax, indexing='ij')
    band = (np.abs(KZ) < kz_band_fac * kz_c) & (np.abs(KX) < kx_band_fac * kz_c)
    mask = (w > amp_thr * w.max()) & band & ((np.abs(KZ) + np.abs(KX)) > 0)
    n_mask = int(mask.sum())
    if n_mask < min_mask_px:
        return np.nan, np.nan, n_mask

    W = w[mask] ** wls_pow
    if force_dz_zero:
        A = np.column_stack([KX[mask], np.ones(n_mask)])
        c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
        return 0.0, float(c[0]), n_mask
    A = np.column_stack([KZ[mask], KX[mask], np.ones(n_mask)])
    c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
    return float(c[0]), float(c[1]), n_mask


def sliding_window_scan(img_a, img_b, diff, z_common, x_img_arr, dz_g, dx_g, kz_c,
                         window_z_m, window_x_m, stride_z_px=None, stride_x_px=None,
                         z_range_m=None, x_range_m=None, data_pad=8,
                         kz_band_fac=0.5, kx_band_fac=2.0, amp_thr=0.20,
                         wls_pow=3, pad_fac=6, force_dz_zero=False, min_mask_px=3):
    """Slide a fixed-size window across (img_a, img_b), running the WLS phase-plane
    fit in each window. window_z_m / window_x_m are the window's physical depth and
    radial extent (independent -- see WINDOW_Z_LAMBDA_FRAC / WINDOW_X_LAMBDA_FRAC
    above for why they shouldn't be equal). Returns dz_map, dx_map, n_map, e_map
    (2-D; e_map is the window's mean |diff| amplitude, for the absolute energy gate),
    z_centers_m, x_centers_m (1-D map axes, physical units), and window_px (list of
    (z0,z1,x0,x1) pixel boxes, for QC plotting)."""
    Nz_img, Nx_img = img_a.shape
    nz_half = max(1, int(round((window_z_m / 2) / dz_g)))
    nx_half = max(1, int(round((window_x_m / 2) / dx_g)))
    if stride_z_px is None:
        stride_z_px = 2 * nz_half
    if stride_x_px is None:
        stride_x_px = 2 * nx_half

    if z_range_m is None:
        z0_lim, z1_lim = 0, Nz_img
    else:
        rows = np.where((z_common >= z_range_m[0]) & (z_common <= z_range_m[1]))[0]
        z0_lim, z1_lim = int(rows.min()), int(rows.max()) + 1
    if x_range_m is None:
        x0_lim, x1_lim = 0, Nx_img
    else:
        cols = np.where((x_img_arr >= x_range_m[0]) & (x_img_arr <= x_range_m[1]))[0]
        x0_lim, x1_lim = int(cols.min()), int(cols.max()) + 1

    z_centers_px = list(range(z0_lim + nz_half, z1_lim - nz_half + 1, stride_z_px))
    x_centers_px = list(range(x0_lim + nx_half, x1_lim - nx_half + 1, stride_x_px))
    if not z_centers_px or not x_centers_px:
        raise ValueError('Scan region is smaller than one window -- widen the range '
                          'or shrink WINDOW_Z_LAMBDA_FRAC / WINDOW_X_LAMBDA_FRAC.')

    dz_map = np.full((len(z_centers_px), len(x_centers_px)), np.nan)
    dx_map = np.full_like(dz_map, np.nan)
    n_map  = np.zeros_like(dz_map, dtype=int)
    e_map  = np.zeros_like(dz_map)
    window_px = []
    for iz, zc in enumerate(z_centers_px):
        for ix, xc in enumerate(x_centers_px):
            z0, z1 = zc - nz_half, zc + nz_half
            x0, x1 = xc - nx_half, xc + nx_half
            dz_est, dx_est, n_mask = _window_wls_fit(
                img_a, img_b, z0, z1, x0, x1, data_pad, dz_g, dx_g, kz_c,
                kz_band_fac, kx_band_fac, amp_thr, wls_pow, pad_fac,
                force_dz_zero, min_mask_px)
            dz_map[iz, ix] = dz_est
            dx_map[iz, ix] = dx_est
            n_map[iz, ix]  = n_mask
            e_map[iz, ix]  = np.abs(diff[z0:z1, x0:x1]).mean()
            window_px.append((z0, z1, x0, x1))

    z_centers_m = z_common[z_centers_px]
    x_centers_m = x_img_arr[x_centers_px]
    return dz_map, dx_map, n_map, e_map, z_centers_m, x_centers_m, window_px, (nz_half, nx_half)


def _plot_window_qc(diff, z_common, x_img_arr, window_px, nz_half, nx_half, dz_g, dx_g,
                     title, save_path, max_boxes=250, thesis_name=None):
    """QC figure: difference B-scan with every scanned window drawn as a thin rectangle,
    plus one window highlighted in green so its size is easy to judge against the data
    (too big -> averages out the front; too small -> noisy / too few k-cells)."""
    extent = [x_img_arr[0], x_img_arr[-1], z_common[-1], z_common[0]]
    vmax_d = np.percentile(np.abs(diff), 98)
    fig, ax = plt.subplots(figsize=(9, 6))
    im = ax.imshow(diff, aspect='auto', cmap='RdBu_r', extent=extent, origin='upper',
                    vmin=-vmax_d, vmax=vmax_d)
    plt.colorbar(im, ax=ax, label='Delta amplitude [a.u.]')
    ax.invert_yaxis()

    step = max(1, len(window_px) // max_boxes)
    for (z0, z1, x0, x1) in window_px[::step]:
        z_top, z_bot = z_common[z0], z_common[z1 - 1]
        x_lo, x_hi   = x_img_arr[x0], x_img_arr[x1 - 1]
        ax.add_patch(Rectangle((x_lo, z_bot), width=x_hi - x_lo, height=z_top - z_bot,
                                lw=0.6, edgecolor='yellow', facecolor='none', alpha=0.6))

    z0, z1, x0, x1 = window_px[len(window_px) // 2]
    z_top, z_bot = z_common[z0], z_common[z1 - 1]
    x_lo, x_hi   = x_img_arr[x0], x_img_arr[x1 - 1]
    ax.add_patch(Rectangle((x_lo, z_bot), width=x_hi - x_lo, height=z_top - z_bot,
                            lw=2.0, edgecolor='lime', facecolor='none'))
    ax.set_title(f'{title}\none window = {2*nz_half*dz_g:.2f} m (depth) x '
                 f'{2*nx_half*dx_g:.2f} m (radial)  --  {len(window_px)} windows total')
    ax.set_xlabel('Radial distance (m)'); ax.set_ylabel('Depth (m)')
    plt.tight_layout()
    fig.savefig(save_path, dpi=150)
    if thesis_name is not None:
        save_fig(fig, thesis_name, study='FieldData_Study', prefix='FD_', category='Compilations')
    plt.show(); plt.close(fig)


def _plot_disp_maps(dz_map, dx_map, valid, z_centers_m, x_centers_m,
                     title_prefix, save_path, thesis_name=None):
    """Delta-z and Delta-x maps side-by-side; windows failing the valid mask
    (WIN_MIN_MASK_PX and/or WIN_ENERGY_THR) are blanked."""
    dz_plot = np.where(valid, dz_map, np.nan)
    dx_plot = np.where(valid, dx_map, np.nan)
    extent = [x_centers_m[0], x_centers_m[-1], z_centers_m[-1], z_centers_m[0]]

    fig, (ax_z, ax_x) = plt.subplots(1, 2, figsize=(13, 5))
    for ax, m, lbl, cmap in ((ax_z, dz_plot, 'Delta z (depth) [m]', 'RdBu_r'),
                              (ax_x, dx_plot, 'Delta x (radial) [m]', 'PuOr_r')):
        finite = m[np.isfinite(m)]
        vmax = np.percentile(np.abs(finite), 98) if finite.size else 1.0
        vmax = vmax if vmax > 0 else 1.0
        im = ax.imshow(m, aspect='auto', cmap=cmap, extent=extent, origin='upper',
                        vmin=-vmax, vmax=vmax)
        plt.colorbar(im, ax=ax, label=lbl)
        ax.invert_yaxis()
        ax.set_xlabel('Radial distance (m)'); ax.set_ylabel('Depth (m)')
        ax.set_title(f'{title_prefix}: {lbl}')
    plt.tight_layout()
    fig.savefig(save_path, dpi=150)
    if thesis_name is not None:
        save_fig(fig, thesis_name, study='FieldData_Study', prefix='FD_', category='Compilations')
    plt.show(); plt.close(fig)


# ── Run the scan for the same pairs as the manual-ROI analysis above ───────────────
# THESIS_QC_NAME / THESIS_MAPS_NAME_FMT — pass a name to also export the QC figure and
# each pair's dz/dx maps to TimeLapse_Figures/FieldData_Study/Compilations/ (protocol:
# .wiki/FIGURES_PROTOCOL.md), in addition to the ad-hoc roi_out save above. None skips
# the thesis export.
THESIS_QC_NAME       = 'sliding_window_qc'
THESIS_MAPS_NAME_FMT = 'sliding_window_maps_{ref_run}_to_{run_b}'

scan_results = {}
for i_pair, (run_a, run_b) in enumerate(MANUAL_PAIRS):
    ref_run = COMPARE_TO if COMPARE_TO is not None else run_a
    img_a = _load_img(ROI_METHOD, ref_run)
    img_b = _load_img(ROI_METHOD, run_b)
    if img_a is None or img_b is None:
        print(f'[{ref_run}->{run_b}] Missing .npy -- skipping sliding-window scan.')
        continue
    n = min(img_a.shape[0], img_b.shape[0])
    img_a, img_b = img_a[:n], img_b[:n]
    z_common = depth[:n]
    diff = img_b - img_a

    (dz_map, dx_map, n_map, e_map, z_centers_m, x_centers_m, window_px,
     (nz_half, nx_half)) = sliding_window_scan(
        img_a, img_b, diff, z_common, x_img, dz_g, dx_g, kz_c, window_z_m, window_x_m,
        stride_z_px=STRIDE_Z_PX, stride_x_px=STRIDE_X_PX,
        z_range_m=SCAN_Z_RANGE_M, x_range_m=SCAN_X_RANGE_M,
        data_pad=WIN_DATA_PAD_PX, kz_band_fac=WIN_KZ_BAND_FAC, kx_band_fac=WIN_KX_BAND_FAC,
        amp_thr=WIN_AMP_THR, wls_pow=WIN_WLS_POW, pad_fac=WIN_PAD_FAC,
        force_dz_zero=WIN_FORCE_DZ_ZERO, min_mask_px=WIN_MIN_MASK_PX)

    vmax_d_global = np.percentile(np.abs(diff), 98)
    valid = (n_map >= WIN_MIN_MASK_PX) & (e_map >= WIN_ENERGY_THR * vmax_d_global)

    scan_results[(ref_run, run_b)] = dict(dz_map=dz_map, dx_map=dx_map, n_map=n_map,
                                           e_map=e_map, valid=valid,
                                           z_centers_m=z_centers_m, x_centers_m=x_centers_m)

    print(f'prof_{ref_run}->{run_b}: {len(window_px)} windows scanned '
          f'({len(z_centers_m)} x {len(x_centers_m)} grid), {int(valid.sum())} pass '
          f'both WIN_MIN_MASK_PX and WIN_ENERGY_THR gates')

    if i_pair == QC_PAIR_INDEX:
        _plot_window_qc(diff, z_common, x_img, window_px, nz_half, nx_half, dz_g, dx_g,
                         title=f'Sliding-window ROI grid -- {ROI_METHOD} prof_{ref_run}->{run_b}',
                         save_path=roi_out / f'{ROI_METHOD}_scan_qc_{ref_run}_to_{run_b}.png',
                         thesis_name=THESIS_QC_NAME)

    _plot_disp_maps(dz_map, dx_map, valid, z_centers_m, x_centers_m,
                     title_prefix=f'prof_{ref_run}->{run_b}',
                     save_path=roi_out / f'{ROI_METHOD}_scan_maps_{ref_run}_to_{run_b}.png',
                     thesis_name=(THESIS_MAPS_NAME_FMT.format(ref_run=ref_run, run_b=run_b)
                                  if THESIS_MAPS_NAME_FMT else None))

print(f'\nSliding-window scan done -- {len(scan_results)} pair(s). Output: {roi_out}')


## Manual k-Space Pixel Selection (Napari)

The automatic WLS fit above selects k-cells with a band (`KZ_BAND_FAC`/`KX_BAND_FAC`)
and a relative amplitude gate (`WLS_AMP_THR`). Here we investigate the alternative:
picking the cross-spectrum pixels by hand, in an interactive [napari](https://napari.org)
viewer, and refitting the WLS plane on exactly the pixels you paint.

This is two cells because napari is interactive and can't run to completion inside a
single non-interactive cell execution:

1. **Launch** — computes the cross-spectrum for a chosen pair/ROI (same crop + taper
   recipe as the manual-ROI cell above) and opens a napari viewer with the amplitude
   as an image layer and an empty, paintable `manual_mask` Labels layer on top. Paint
   over the k-cells you want to include (label 1); leave everything else at 0.
2. **Harvest** — run *after* you've painted. Reads `manual_mask` back, refits the WLS
   plane on your selected pixels, and plots it side by side with the automatic
   band+amplitude selection for the same window so you can compare the two directly.


In [17]:
# ── Manual k-space pixel selection (napari) — 1: build cross-spectrum & launch viewer ──
#
# NAPARI_PAIR  — which (run_a, run_b) from MANUAL_PAIRS to investigate.
# NAPARI_ROI_M — ROI in physical metres (z_min, z_max, x_min, x_max); None -> reuse the
#                manual ROI already defined for this pair in MANUAL_ROIS above, falling
#                back to the monogenic-envelope auto-ROI if there isn't one.
# NAPARI_KDISP_FAC — half-width of the displayed k-space crop, as a multiple of kz_c.
#                Only affects what's shown/paintable, not the fit itself.
#
# Paint over the 'manual_mask' layer (paintbrush, label 1) to mark the k-cells you want
# in the WLS fit -- everything left at 0 is excluded. Zoom/pan freely; painting only
# writes where you actually paint. Run cell 2 once you're done.
# ─────────────────────────────────────────────────────────────────────────────────────

import napari

NAPARI_PAIR      = MANUAL_PAIRS[3]
NAPARI_ROI_M     = None
NAPARI_KDISP_FAC = 3.0

run_a, run_b = NAPARI_PAIR
ref_run = COMPARE_TO if COMPARE_TO is not None else run_a
img_a = _load_img(ROI_METHOD, ref_run)
img_b = _load_img(ROI_METHOD, run_b)
if img_a is None or img_b is None:
    raise RuntimeError(f'Missing .npy for pair {NAPARI_PAIR}')

n = min(img_a.shape[0], img_b.shape[0])
img_a, img_b = img_a[:n], img_b[:n]
z_common = depth[:n]

roi_spec = (NAPARI_ROI_M or MANUAL_ROIS.get((run_a, run_b))
            or MANUAL_ROIS.get((ref_run, run_b)))
if roi_spec is not None:
    z0, z1, x0, x1 = _phys_to_pix(z_common, x_img, *roi_spec)
else:
    z0, z1, x0, x1 = _roi_from_envelope(_monogenic_envelope(img_b - img_a), ROI_THRESH)

DATA_PAD = 8
_z0p = max(0, z0 - DATA_PAD);  _z1p = min(img_a.shape[0], z1 + DATA_PAD)
_x0p = max(0, x0 - DATA_PAD);  _x1p = min(img_a.shape[1], x1 + DATA_PAD)
base_crop = img_a[_z0p:_z1p, _x0p:_x1p]
mon_crop  = img_b[_z0p:_z1p, _x0p:_x1p]
Nz, Nx    = base_crop.shape
pad_fac   = 10
Nz_pad, Nx_pad = Nz * pad_fac, Nx * pad_fac

kz_ax = np.fft.fftfreq(Nz_pad, d=dz_g) * 2 * np.pi
kx_ax = np.fft.fftfreq(Nx_pad, d=dx_g) * 2 * np.pi
KZ, KX = np.meshgrid(kz_ax, kx_ax, indexing='ij')

_nz_lo = z0 - _z0p;  _nz_hi = _z1p - z1
_nx_lo = x0 - _x0p;  _nx_hi = _x1p - x1
taper = np.outer(_edge_taper_1d(Nz, _nz_lo, _nz_hi), _edge_taper_1d(Nx, _nx_lo, _nx_hi))
XS  = (np.fft.fft2(base_crop * taper, s=(Nz_pad, Nx_pad)) *
       np.conj(np.fft.fft2(mon_crop * taper, s=(Nz_pad, Nx_pad))))
w   = np.abs(XS);  phi = np.angle(XS)

# Centre-zero display, cropped to +/- NAPARI_KDISP_FAC * kz_c so the viewer isn't 90%
# empty high-frequency padding. NAPARI_IZ0/IX0 remember where this crop sits inside the
# full (unshifted) KZ/KX grid, so cell 2 can map the painted mask back correctly.
kz_disp = np.fft.fftshift(kz_ax);  kx_disp = np.fft.fftshift(kx_ax)
_iz = np.where(np.abs(kz_disp) < NAPARI_KDISP_FAC * kz_c)[0]
_ix = np.where(np.abs(kx_disp) < NAPARI_KDISP_FAC * kz_c)[0]
NAPARI_IZ0, NAPARI_IZ1 = int(_iz.min()), int(_iz.max()) + 1
NAPARI_IX0, NAPARI_IX1 = int(_ix.min()), int(_ix.max()) + 1

w_shift   = np.fft.fftshift(w)[NAPARI_IZ0:NAPARI_IZ1, NAPARI_IX0:NAPARI_IX1]
phi_shift = np.fft.fftshift(phi)[NAPARI_IZ0:NAPARI_IZ1, NAPARI_IX0:NAPARI_IX1]
kz_crop   = kz_disp[NAPARI_IZ0:NAPARI_IZ1]
kx_crop   = kx_disp[NAPARI_IX0:NAPARI_IX1]

napari_viewer = napari.Viewer(title=f'k-space manual selection -- prof_{ref_run}->{run_b}')
napari_viewer.add_image(w_shift, name='amplitude |XS|', colormap='inferno')
napari_viewer.add_image(phi_shift, name='phase (rad)', colormap='twilight', visible=False)
manual_labels = napari_viewer.add_labels(
    np.zeros(w_shift.shape, dtype=np.uint8), name='manual_mask')
napari_viewer.layers.selection.active = manual_labels
manual_labels.mode = 'paint'
manual_labels.brush_size = max(1, min(w_shift.shape) // 20)

print(f'Viewer open for prof_{ref_run}->{run_b}.  k-space crop: '
      f'{w_shift.shape[0]} x {w_shift.shape[1]} px  (|kz|,|kx| < {NAPARI_KDISP_FAC * kz_c:.1f} rad/m).')
print("Paint the 'manual_mask' layer (label 1) over the k-cells you want in the fit, "
      "then run the next cell.")


Viewer open for prof_20->38.  k-space crop: 527 x 163 px  (|kz|,|kx| < 18.8 rad/m).
Paint the 'manual_mask' layer (label 1) over the k-cells you want in the fit, then run the next cell.


In [ ]:
# ── Manual k-space pixel selection (napari) — 2: harvest selection & refit ─────────────
#
# Run this AFTER painting the 'manual_mask' layer in the viewer from cell 1. Refits the
# WLS plane using only your painted k-cells, and compares it against the automatic
# band+amplitude selection for the same window.
# ─────────────────────────────────────────────────────────────────────────────────────

manual_mask_crop = manual_labels.data > 0
n_manual = int(manual_mask_crop.sum())
print(f'{n_manual} k-cells painted.')
if n_manual < 3:
    raise RuntimeError("Paint at least 3 k-cells in the 'manual_mask' layer "
                        "(napari_viewer, cell 1) before running this cell.")

# Map the painted (cropped, fftshifted) mask back onto the full, unshifted KZ/KX grid.
manual_mask_shift = np.zeros(np.fft.fftshift(w).shape, dtype=bool)
manual_mask_shift[NAPARI_IZ0:NAPARI_IZ1, NAPARI_IX0:NAPARI_IX1] = manual_mask_crop
manual_mask_full = np.fft.ifftshift(manual_mask_shift)


def _wls_fit_from_mask(mask, w, phi, KZ, KX, wls_pow, force_dz_zero):
    n_mask = int(mask.sum())
    W = w[mask] ** wls_pow
    if force_dz_zero:
        A = np.column_stack([KX[mask], np.ones(n_mask)])
        c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
        return 0.0, float(c[0]), float(c[1]), n_mask
    A = np.column_stack([KZ[mask], KX[mask], np.ones(n_mask)])
    c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
    return float(c[0]), float(c[1]), float(c[2]), n_mask


dz_man, dx_man, phi0_man, n_man = _wls_fit_from_mask(
    manual_mask_full, w, phi, KZ, KX, WLS_POW, FORCE_DZ_ZERO)

# Automatic fit for the same window, same recipe as the manual-ROI cell above.
_kz_fac = (MANUAL_KZ_BAND_FAC.get((run_a, run_b))
           or MANUAL_KZ_BAND_FAC.get((ref_run, run_b)) or KZ_BAND_FAC)
_kx_fac = (MANUAL_KX_BAND_FAC.get((run_a, run_b))
           or MANUAL_KX_BAND_FAC.get((ref_run, run_b)) or KX_BAND_FAC)
auto_band = (np.abs(KZ) < _kz_fac * kz_c) & (np.abs(KX) < _kx_fac * kz_c)
auto_mask = (w > WLS_AMP_THR * w.max()) & auto_band & ((np.abs(KZ) + np.abs(KX)) > 0)
dz_auto, dx_auto, phi0_auto, n_auto = _wls_fit_from_mask(
    auto_mask, w, phi, KZ, KX, WLS_POW, FORCE_DZ_ZERO)

print(f'Manual    ({n_man:5d} px):  dz={dz_man:+.4f} m  dx={dx_man:+.4f} m  phi0={phi0_man:+.3f} rad')
print(f'Automatic ({n_auto:5d} px):  dz={dz_auto:+.4f} m  dx={dx_auto:+.4f} m  phi0={phi0_auto:+.3f} rad')

# ── Comparison figure: phase panel with each selection outlined ────────────────────────
extent_k = [kx_crop[0], kx_crop[-1], kz_crop[-1], kz_crop[0]]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, mask_full, dz_e, dx_e, n_e, title in (
        (axes[0], manual_mask_full, dz_man, dx_man, n_man, 'Manual selection'),
        (axes[1], auto_mask, dz_auto, dx_auto, n_auto, 'Automatic band + amplitude gate')):
    mask_shift_crop = np.fft.fftshift(mask_full)[NAPARI_IZ0:NAPARI_IZ1, NAPARI_IX0:NAPARI_IX1]
    im = ax.imshow(phi_shift, aspect='auto', cmap='RdBu_r', extent=extent_k,
                    vmin=-np.pi, vmax=np.pi, origin='upper')
    ax.contour(kx_crop, kz_crop, mask_shift_crop.astype(float), levels=[0.5],
               colors='lime', linewidths=1.2)
    ax.set_title(f'{title}\ndz={dz_e:+.4f} m  dx={dx_e:+.4f} m  n={n_e} px')
    ax.set_xlabel('kx [rad/m]');  ax.set_ylabel('kz [rad/m]')
plt.colorbar(im, ax=axes, label='phase [rad]', shrink=0.8)
plt.suptitle(f'Manual vs automatic k-space selection -- prof_{ref_run}->{run_b}', y=1.03)
fig.savefig(roi_out / f'{ROI_METHOD}_napari_manual_vs_auto_{ref_run}_to_{run_b}.png',
            dpi=150, bbox_inches='tight')
# THESIS_NAME -- also export to TimeLapse_Figures/FieldData_Study/Compilations/ (protocol:
# .wiki/FIGURES_PROTOCOL.md). Set to None to skip; picks a fixed thesis-facing name so
# re-running for a different pair overwrites rather than accumulating stale copies.
THESIS_NAME = 'napari_kspace_manual_vs_auto'
if THESIS_NAME is not None:
    save_fig(fig, THESIS_NAME, study='FieldData_Study', prefix='FD_', category='Compilations')
plt.show(); plt.close(fig)


## Manual Pixel Selection in Depth-Radial Space (Napari)

Same idea as the k-space selection above, but now painting directly on the
**difference B-scan** (depth x radial distance) instead of on the cross-spectrum.
This lets you trace the actual shape of the main reflection -- which is rarely a
clean rectangle -- rather than being limited to the axis-aligned boxes in
`MANUAL_ROIS` above.

Two cells, same launch/harvest pattern as before:

1. **Launch** — opens a napari viewer on the difference B-scan for a chosen pair,
   with an empty paintable `manual_roi_mask` Labels layer on top. Paint over the
   main reflection (label 1); everything else stays excluded.
2. **Harvest** — run *after* painting. Your painted mask is Gaussian-softened
   (to avoid the spectral leakage a hard-edged mask would introduce, the same
   issue the k-space investigation above ran into) and used as the spatial
   window for the cross-spectrum, in place of the standard ROI-box + edge-taper.
   Refits the WLS plane and plots it next to the fit from the equivalent
   rectangular `MANUAL_ROIS` box for the same pair.


In [ ]:
# ── Manual depth-radial pixel selection (napari) — 1: launch viewer ────────────────────
#
# NAPARI_IMG_PAIR — which (run_a, run_b) from MANUAL_PAIRS to investigate.
#
# Paint over the 'manual_roi_mask' layer (paintbrush, label 1) directly on the
# difference B-scan to mark where the main reflection actually sits -- follow its
# shape, it doesn't have to be a rectangle. Run cell 2 once you're done.
# ─────────────────────────────────────────────────────────────────────────────────────

import napari

NAPARI_IMG_PAIR = MANUAL_PAIRS[3]

img_run_a, img_run_b = NAPARI_IMG_PAIR
img_ref_run = COMPARE_TO if COMPARE_TO is not None else img_run_a
img_a_bscan = _load_img(ROI_METHOD, img_ref_run)
img_b_bscan = _load_img(ROI_METHOD, img_run_b)
if img_a_bscan is None or img_b_bscan is None:
    raise RuntimeError(f'Missing .npy for pair {NAPARI_IMG_PAIR}')

n_bscan = min(img_a_bscan.shape[0], img_b_bscan.shape[0])
img_a_bscan, img_b_bscan = img_a_bscan[:n_bscan], img_b_bscan[:n_bscan]
z_common_bscan = depth[:n_bscan]
diff_bscan = img_b_bscan - img_a_bscan

vmax_bscan = np.percentile(np.abs(diff_bscan), 98)
diff_cmap = napari.utils.Colormap(['blue', 'white', 'red'], name='diff_rdbu')
bscan_viewer = napari.Viewer(title=f'Depth-radial manual ROI -- prof_{img_ref_run}->{img_run_b}')
bscan_viewer.add_image(diff_bscan, name='difference B-scan',
                        colormap=diff_cmap,
                        contrast_limits=[-vmax_bscan, vmax_bscan])
manual_roi_labels = bscan_viewer.add_labels(
    np.zeros(diff_bscan.shape, dtype=np.uint8), name='manual_roi_mask')
bscan_viewer.layers.selection.active = manual_roi_labels
manual_roi_labels.mode = 'paint'
manual_roi_labels.brush_size = max(1, min(diff_bscan.shape) // 40)

print(f'Viewer open for prof_{img_ref_run}->{img_run_b}.  Image shape (depth px x radial px): '
      f'{diff_bscan.shape}.')
print(f'Depth axis:   {z_common_bscan[0]:.1f} m (row 0)  ->  {z_common_bscan[-1]:.1f} m (row {n_bscan - 1}).')
print(f'Radial axis:  {x_img[0]:.2f} m (col 0)  ->  {x_img[-1]:.2f} m (col {len(x_img) - 1}).')
print("Paint the 'manual_roi_mask' layer (label 1) over the main reflection, then run the next cell.")


In [ ]:
# ── Manual depth-radial pixel selection (napari) — 2: harvest selection & refit ────────
#
# Run this AFTER painting the 'manual_roi_mask' layer in the viewer from cell 1.
#
# GAUSS_SIGMA_PX — the painted mask is a hard 0/1 blob; used raw, its edges would leak
#                  spectral energy across a wide kx/kz range the same way the sharp-
#                  edged synthetic reflector did in the k-space investigation above.
#                  Blurring it by this many pixels softens the edges before it's used
#                  as the spatial window for the cross-spectrum.
# ─────────────────────────────────────────────────────────────────────────────────────

from scipy.ndimage import gaussian_filter

GAUSS_SIGMA_PX = 3.0

mask_full = manual_roi_labels.data > 0
n_painted = int(mask_full.sum())
print(f'{n_painted} image px painted.')
if n_painted < 20:
    raise RuntimeError("Paint a larger region in the 'manual_roi_mask' layer "
                        "(bscan_viewer, cell 1) before running this cell.")

rows, cols = np.where(mask_full)
z0, z1 = int(rows.min()), int(rows.max()) + 1
x0, x1 = int(cols.min()), int(cols.max()) + 1

MARGIN_PX = int(np.ceil(3 * GAUSS_SIGMA_PX))
_z0p = max(0, z0 - MARGIN_PX);  _z1p = min(img_a_bscan.shape[0], z1 + MARGIN_PX)
_x0p = max(0, x0 - MARGIN_PX);  _x1p = min(img_a_bscan.shape[1], x1 + MARGIN_PX)
base_crop = img_a_bscan[_z0p:_z1p, _x0p:_x1p]
mon_crop  = img_b_bscan[_z0p:_z1p, _x0p:_x1p]
mask_crop = mask_full[_z0p:_z1p, _x0p:_x1p].astype(float)

soft_win = gaussian_filter(mask_crop, sigma=GAUSS_SIGMA_PX)
if soft_win.max() > 0:
    soft_win = soft_win / soft_win.max()

Nz, Nx = base_crop.shape
pad_fac = 10
Nz_pad, Nx_pad = Nz * pad_fac, Nx * pad_fac
kz_ax = np.fft.fftfreq(Nz_pad, d=dz_g) * 2 * np.pi
kx_ax = np.fft.fftfreq(Nx_pad, d=dx_g) * 2 * np.pi
KZ, KX = np.meshgrid(kz_ax, kx_ax, indexing='ij')

XS = (np.fft.fft2(base_crop * soft_win, s=(Nz_pad, Nx_pad)) *
      np.conj(np.fft.fft2(mon_crop * soft_win, s=(Nz_pad, Nx_pad))))
w, phi = np.abs(XS), np.angle(XS)

_kz_fac = (MANUAL_KZ_BAND_FAC.get((img_run_a, img_run_b))
           or MANUAL_KZ_BAND_FAC.get((img_ref_run, img_run_b)) or KZ_BAND_FAC)
_kx_fac = (MANUAL_KX_BAND_FAC.get((img_run_a, img_run_b))
           or MANUAL_KX_BAND_FAC.get((img_ref_run, img_run_b)) or KX_BAND_FAC)
band  = (np.abs(KZ) < _kz_fac * kz_c) & (np.abs(KX) < _kx_fac * kz_c)
kmask = (w > WLS_AMP_THR * w.max()) & band & ((np.abs(KZ) + np.abs(KX)) > 0)
dz_paint, dx_paint, phi0_paint, n_paint_k = _wls_fit_from_mask(
    kmask, w, phi, KZ, KX, WLS_POW, FORCE_DZ_ZERO)

# ── Same window, but the standard rectangular MANUAL_ROIS box + edge taper, for
#    comparison -- the exact recipe used in the manual-ROI cell above. ─────────────────
roi_spec = MANUAL_ROIS.get((img_run_a, img_run_b)) or MANUAL_ROIS.get((img_ref_run, img_run_b))
if roi_spec is not None:
    rz0, rz1, rx0, rx1 = _phys_to_pix(z_common_bscan, x_img, *roi_spec)
else:
    rz0, rz1, rx0, rx1 = _roi_from_envelope(_monogenic_envelope(diff_bscan), ROI_THRESH)

DATA_PAD = 8
_rz0p = max(0, rz0 - DATA_PAD);  _rz1p = min(img_a_bscan.shape[0], rz1 + DATA_PAD)
_rx0p = max(0, rx0 - DATA_PAD);  _rx1p = min(img_a_bscan.shape[1], rx1 + DATA_PAD)
rbase_crop = img_a_bscan[_rz0p:_rz1p, _rx0p:_rx1p]
rmon_crop  = img_b_bscan[_rz0p:_rz1p, _rx0p:_rx1p]
rNz, rNx   = rbase_crop.shape
rNz_pad, rNx_pad = rNz * pad_fac, rNx * pad_fac
rkz_ax = np.fft.fftfreq(rNz_pad, d=dz_g) * 2 * np.pi
rkx_ax = np.fft.fftfreq(rNx_pad, d=dx_g) * 2 * np.pi
RKZ, RKX = np.meshgrid(rkz_ax, rkx_ax, indexing='ij')
rtaper = np.outer(_edge_taper_1d(rNz, rz0 - _rz0p, _rz1p - rz1),
                   _edge_taper_1d(rNx, rx0 - _rx0p, _rx1p - rx1))
RXS = (np.fft.fft2(rbase_crop * rtaper, s=(rNz_pad, rNx_pad)) *
       np.conj(np.fft.fft2(rmon_crop * rtaper, s=(rNz_pad, rNx_pad))))
rw, rphi = np.abs(RXS), np.angle(RXS)
rband  = (np.abs(RKZ) < _kz_fac * kz_c) & (np.abs(RKX) < _kx_fac * kz_c)
rkmask = (rw > WLS_AMP_THR * rw.max()) & rband & ((np.abs(RKZ) + np.abs(RKX)) > 0)
dz_rect, dx_rect, phi0_rect, n_rect_k = _wls_fit_from_mask(
    rkmask, rw, rphi, RKZ, RKX, WLS_POW, FORCE_DZ_ZERO)

print(f'Painted mask    ({n_painted:6d} img px, {n_paint_k:5d} k px):  '
      f'dz={dz_paint:+.4f} m  dx={dx_paint:+.4f} m  phi0={phi0_paint:+.3f} rad')
print(f'Rectangular ROI ({rNz * rNx:6d} img px, {n_rect_k:5d} k px):  '
      f'dz={dz_rect:+.4f} m  dx={dx_rect:+.4f} m  phi0={phi0_rect:+.3f} rad')

# ── Comparison figure ───────────────────────────────────────────────────────────────
extent = [x_img[0], x_img[-1], z_common_bscan[-1], z_common_bscan[0]]
vmax_d = np.percentile(np.abs(diff_bscan), 98)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, title, dz_e, dx_e in ((axes[0], 'Painted selection', dz_paint, dx_paint),
                               (axes[1], 'Rectangular ROI', dz_rect, dx_rect)):
    im = ax.imshow(diff_bscan, aspect='auto', cmap='RdBu_r', extent=extent, origin='upper',
                    vmin=-vmax_d, vmax=vmax_d)
    ax.invert_yaxis()
    ax.set_xlabel('Radial distance (m)');  ax.set_ylabel('Depth (m)')
    ax.set_title(f'{title}\ndz={dz_e:+.4f} m  dx={dx_e:+.4f} m')
plt.colorbar(im, ax=axes, label='Delta amplitude [a.u.]', shrink=0.8)

axes[0].contour(x_img, z_common_bscan, mask_full.astype(float), levels=[0.5],
                colors='lime', linewidths=1.2)
z_top, z_bot = z_common_bscan[rz0], z_common_bscan[rz1 - 1]
x_lo, x_hi   = x_img[rx0], x_img[rx1 - 1]
axes[1].add_patch(Rectangle((x_lo, z_bot), width=x_hi - x_lo, height=z_top - z_bot,
                             lw=1.5, edgecolor='yellow', facecolor='none'))

plt.suptitle(f'Manual image-space selection vs rectangular ROI -- prof_{img_ref_run}->{img_run_b}', y=1.03)
fig.savefig(roi_out / f'{ROI_METHOD}_napari_bscan_manual_vs_rect_{img_ref_run}_to_{img_run_b}.png',
            dpi=150, bbox_inches='tight')
# THESIS_NAME -- also export to TimeLapse_Figures/FieldData_Study/Compilations/ (protocol:
# .wiki/FIGURES_PROTOCOL.md). Set to None to skip; picks a fixed thesis-facing name so
# re-running for a different pair overwrites rather than accumulating stale copies.
THESIS_NAME = 'napari_bscan_manual_vs_rect'
if THESIS_NAME is not None:
    save_fig(fig, THESIS_NAME, study='FieldData_Study', prefix='FD_', category='Compilations')
plt.show(); plt.close(fig)


# Kirchhoff Results

In [ ]:
# ── Manual pair selection + ROI definition ──────────────────────────────────────────
#
# MANUAL_PAIRS  — list of (run_a, run_b) to analyse.
#
# COMPARE_TO    — None  → use pairs as written (run_a vs run_b)
#                 int   → override run_a for every pair to this run number,
#                         so you compare every profile against one fixed reference.
#                         Useful when consecutive-pair shifts are too small to detect:
#                         e.g. COMPARE_TO=1 gives (1,4), (1,8), (1,10), ...
#
# MANUAL_ROIS   — per-pair ROI in physical metres.
#                 Key is the ORIGINAL (run_a, run_b) regardless of COMPARE_TO.
#                 Format: (z_min_m, z_max_m, x_min_m, x_max_m)
#                 None → auto-detect from monogenic envelope.
#
# SHOW_DIAG     — True: plot cross-spectrum phase + fitted plane for every pair.
#                 Helps diagnose whether there is any detectable kx slope.
#
# ROI_METHOD    — 'kirchhoff' or 'gazdag'.
# FORCE_DZ_ZERO — True: fit only Δx + φ₀ (lateral fluid front).
# ROI_THRESH    — envelope threshold for auto-detected pairs.
# ────────────────────────────────────────────────────────────────────────────────────

MANUAL_PAIRS = [
    # (1,  2),
    # (2,  3),
    # (3,  4),
    # (4,  5),
    # (5,  7),
    # (7,  8),
    # (8,  9),
    # (9,  10),
    # (10, 11),
    # (11,  12),
    # (12,  13),
    # (13,  14),
    # (14,  15),
    # (15,  16),
    # (16,  17),
    # (17,  18),
    # (18,  19),
    # (19,  20),
    # (20,  21),
    # (21,  22),
    # (22,  23),
    # (23,  24),
    # (24,  25),
    # (25,  26),
    # (26,  27),
    # (27,  28),
    # (28,  29),
    # (29,  30),
    # (30,  31),
    # (31,  32),
    # (32,  33),
    # (33,  34),
    # (34,  35),
    # (35,  36),
    # (36,  37),
    # (37,  38),
    (1,3),
    (4,8),
    (9,20),
    (21,38),
]

COMPARE_TO = 1         # all pairs vs prof1 (first injection measurement = earliest non-zero migration)

ROI = (70, 79, 4.5, 7.5) # shows pushing-chasing-waiting-pulling correctly with 1.5 * kz_c, w > 0.10
                         # 1.5 * kz_c, w > 0.25 shows stronger drop in pulling phase
MANUAL_ROIS = {
    # (1,  2):  ROI,
    # (2,  3):  ROI,
    # (3,  4):  ROI,
    # (4,  5):  ROI,
    # (5,  7):  ROI,
    # (7,  8):  ROI,
    # (8,  9):  ROI,
    # (9,  10): ROI,
    # (10,  11): ROI,
    # (11,  12): ROI,
    # (12,  13): ROI,
    # (13,  14): ROI,
    # (14,  15): ROI,
    # (15,  16): ROI,
    # (16,  17): ROI,
    # (17,  18): ROI,
    # (18,  19): ROI,
    # (19,  20): ROI,
    # (20,  21): ROI,
    # (21,  22): ROI,
    # (22,  23): ROI,
    # (23,  24): ROI,
    # (24,  25): ROI,
    # (25,  26): ROI,
    # (26,  27): ROI,
    # (27,  28): ROI,
    # (28,  29): ROI,
    # (29,  30): ROI,
    # (30,  31): ROI,
    # (31,  32): ROI,
    # (32,  33): ROI,
    # (33,  34): ROI,
    # (34,  35): ROI,
    # (35,  36): ROI,
    # (36,  37): ROI,
    # (37,  38): ROI,
    (1,3): (71,76,5.0,6.5),
    (4,8): (70,78,5.0,7.0),
    (9,20): (72,77,5.0,6.0),
    (21,38): (70,77,5.0,6.0),
}

ROI_METHOD    = 'kirchhoff_bp'
FORCE_DZ_ZERO = False
ROI_THRESH    = 0.5
SHOW_DIAG     = True   # set False to skip cross-spectrum phase plots
# KZ_BAND_FAC   = 1.6    # band: |k| < KZ_BAND_FAC * kz_c  (WLS fit + 1-D profile display)
# WLS_AMP_THR   = 0.25   # WLS mask: discard k-cells where |XS| < WLS_AMP_THR * max
KZ_BAND_FAC = 0.5    # default kz band; overridden per-pair by MANUAL_KZ_BAND_FAC
KX_BAND_FAC = 2.0    # kx band: |kx| < KX_BAND_FAC * kz_c (must cover the ±8 rad/m lobes)
WLS_AMP_THR = 0.20
MANUAL_KZ_BAND_FAC = {   # per-pair kz band override (reduces kz range for noisy stages)
    (1,  3): 0.5,      # Push: S-curve in 1-D profile — restrict to coherent |kz| < 1.6 rad/m
    (9,  20): 0.25,      # Wait: S-curve in 1-D profile — restrict to coherent |kz| < 1.6 rad/m
    (21, 38): 0.15,      # Pull: oscillatory 1-D profile — same restriction
}

# ────────────────────────────────────────────────────────────────────────────────────

def _phys_to_pix(z_common, x_img_arr, z_min, z_max, x_min, x_max):
    """Convert physical ROI bounds (metres) to pixel indices.
    z_common is DECREASING (row 0 = 85 m deepest).
    Returns (z0, z1, x0, x1), z1/x1 exclusive.
    """
    rows = np.where((z_common >= z_min) & (z_common <= z_max))[0]
    cols = np.where((x_img_arr >= x_min) & (x_img_arr <= x_max))[0]
    if rows.size == 0 or cols.size == 0:
        raise ValueError(
            f'ROI ({z_min}–{z_max} m depth, {x_min}–{x_max} m radial) '
            f'does not intersect the image grid. '
            f'Depth: [{z_common[-1]:.1f}, {z_common[0]:.1f}] m  '
            f'Radial: [{x_img_arr[0]:.2f}, {x_img_arr[-1]:.2f}] m'
        )
    return int(rows[0]), int(rows[-1]) + 1, int(cols[0]), int(cols[-1]) + 1

roi_out = OUT_DIR / 'roi_phase'
roi_out.mkdir(exist_ok=True)

dz_g = dL
dx_g = float(x_img[1] - x_img[0])
kz_c = 2.0 * np.pi * f0_mig / v

def _load_img(method, run):
    p = OUT_DIR / 'migrated' / f'{method}_{run}.npy'
    return np.load(p) if p.exists() else None

results = {}
for run_a, run_b in MANUAL_PAIRS:
    ref_run = COMPARE_TO if COMPARE_TO is not None else run_a
    img_a = _load_img(ROI_METHOD, ref_run)
    img_b = _load_img(ROI_METHOD, run_b)
    if img_a is None or img_b is None:
        print(f'[{ref_run}→{run_b}] Missing .npy — skipping.')
        continue

    n       = min(img_a.shape[0], img_b.shape[0])
    img_a   = img_a[:n];  img_b = img_b[:n]
    z_common = depth[:n]
    diff     = img_b - img_a
    env      = _monogenic_envelope(diff)

    # ROI — look up by original key (run_a, run_b) first, then (ref_run, run_b)
    roi_spec = MANUAL_ROIS.get((run_a, run_b)) or MANUAL_ROIS.get((ref_run, run_b))
    if roi_spec is not None:
        z0, z1, x0, x1 = _phys_to_pix(z_common, x_img, *roi_spec)
        roi_source = 'manual'
    else:
        z0, z1, x0, x1 = _roi_from_envelope(env, ROI_THRESH)
        roi_source = f'auto (thresh={ROI_THRESH})'

    z_roi_top = float(z_common[z0]);     z_roi_bot = float(z_common[z1 - 1])
    x_roi_lo  = float(x_img[x0]);        x_roi_hi  = float(x_img[x1 - 1])

    # ── Difference + envelope figure ────────────────────────────────────────────
    extent = [x_img[0], x_img[-1], z_common[-1], z_common[0]]
    vmax_d = np.percentile(np.abs(diff), 98)
    vmax_e = np.percentile(env, 98)

    fig, (ax_d, ax_e) = plt.subplots(1, 2, figsize=(14, 5))
    im_d = ax_d.imshow(diff, aspect='auto', cmap='RdBu_r',
                        extent=extent, origin='upper',
                        vmin=-vmax_d, vmax=vmax_d)
    plt.colorbar(im_d, ax=ax_d, label='Δ amplitude [a.u.]')
    ax_d.invert_yaxis()
    ax_d.add_patch(Rectangle((x_roi_lo, z_roi_bot),
                              width=x_roi_hi - x_roi_lo, height=z_roi_top - z_roi_bot,
                              lw=1.5, edgecolor='yellow', facecolor='none'))
    ax_d.xaxis.set_major_locator(ticker.MultipleLocator(0.5))
    ax_d.yaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_d.grid(True, color='white', lw=0.3, alpha=0.4)
    ax_d.set_title(f'{ROI_METHOD.capitalize()} diff: prof_{run_b} − prof_{ref_run}  [{roi_source}]')
    ax_d.set_xlabel('Radial distance (m)');  ax_d.set_ylabel('Depth (m)')

    im_e = ax_e.imshow(env, aspect='auto', cmap='inferno',
                        extent=extent, origin='upper', vmin=0, vmax=vmax_e)
    plt.colorbar(im_e, ax=ax_e, label='Monogenic envelope [a.u.]')
    ax_e.invert_yaxis()
    ax_e.add_patch(Rectangle((x_roi_lo, z_roi_bot),
                              width=x_roi_hi - x_roi_lo, height=z_roi_top - z_roi_bot,
                              lw=1.5, edgecolor='cyan', facecolor='none'))
    ax_e.xaxis.set_major_locator(ticker.MultipleLocator(0.5))
    ax_e.yaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_e.grid(True, color='white', lw=0.3, alpha=0.4)
    ax_e.set_title(f'Monogenic envelope  [{roi_source}]')
    ax_e.set_xlabel('Radial distance (m)');  ax_e.set_ylabel('Depth (m)')

    plt.tight_layout()
    fig.savefig(roi_out / f'{ROI_METHOD}_roi_{ref_run}_to_{run_b}.png', dpi=150)
    plt.show();  plt.close(fig)

    # ── WLS phase-plane fit (zero-padded for denser kx/kz sampling) ─────────────
    base_crop = img_a[z0:z1, x0:x1]
    mon_crop  = img_b[z0:z1, x0:x1]
    Nz, Nx    = base_crop.shape
    pad_fac   = 10
    Nz_pad, Nx_pad = Nz * pad_fac, Nx * pad_fac

    kz_ax = np.fft.fftfreq(Nz_pad, d=dz_g) * 2 * np.pi
    kx_ax = np.fft.fftfreq(Nx_pad, d=dx_g) * 2 * np.pi
    KZ, KX = np.meshgrid(kz_ax, kx_ax, indexing='ij')
    taper  = np.outer(tukey(Nz, alpha=0.15), tukey(Nx, alpha=0.15))
    XS     = (np.fft.fft2(base_crop * taper, s=(Nz_pad, Nx_pad)) *
               np.conj(np.fft.fft2(mon_crop * taper, s=(Nz_pad, Nx_pad))))
    w      = np.abs(XS);  phi = np.angle(XS)
    _kz_fac = (MANUAL_KZ_BAND_FAC.get((run_a, run_b))
               or MANUAL_KZ_BAND_FAC.get((ref_run, run_b))
               or KZ_BAND_FAC)
    band   = (np.abs(KZ) < _kz_fac * kz_c) & (np.abs(KX) < KX_BAND_FAC * kz_c)
    mask   = (w > WLS_AMP_THR * w.max()) & band & ((np.abs(KZ) + np.abs(KX)) > 0)

    n_mask = int(mask.sum())
    if n_mask < 3:
        print(f'[{ref_run}→{run_b}] WARNING: only {n_mask} pixels pass the WLS mask '
              f'(crop {Nz}×{Nx} px, padded to {Nz_pad}×{Nx_pad}).  ROI may be too small or SNR too low.')
        dx_est = phi_0 = dz_est = 0.0
    else:
        W = w[mask]
        if FORCE_DZ_ZERO:
            A = np.column_stack([KX[mask], np.ones(n_mask)])
        else:
            A = np.column_stack([KZ[mask], KX[mask], np.ones(n_mask)])
        c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
        if FORCE_DZ_ZERO:
            dz_est, dx_est, phi_0 = 0.0, float(c[0]), float(c[1])
        else:
            dz_est, dx_est, phi_0 = float(c[0]), float(c[1]), float(c[2])

    # ── Diagnostic: cross-spectrum phase + fitted plane ──────────────────────────
    if SHOW_DIAG and n_mask >= 3:
        # Shift to centre-zero frequency for display
        phi_shift = np.fft.fftshift(phi)
        w_shift   = np.fft.fftshift(w)
        kz_disp   = np.fft.fftshift(kz_ax)
        kx_disp   = np.fft.fftshift(kx_ax)

        # Fitted plane at each pixel
        fitted = KX * dx_est + KZ * dz_est + phi_0
        fitted_shift = np.fft.fftshift(fitted)

        # 1-D kz profile: amplitude-weighted mean phase over all kx in band
        # Fracture is near-parallel to borehole => primary shift is along z.
        phi_1d  = np.zeros(Nz_pad)
        w_1d    = np.zeros(Nz_pad)
        band_kx = (kx_ax > 0) & (kx_ax < KX_BAND_FAC * kz_c)  # positive kx only — avoids S-curve from anti-phase lobes
        for i_row in range(Nz_pad):
            sel = band_kx & (w[i_row, :] > WLS_AMP_THR * w.max())
            if sel.sum() > 0:
                phi_1d[i_row] = np.average(phi[i_row, :][sel], weights=w[i_row, :][sel])
                w_1d[i_row]   = w[i_row, :][sel].sum()
        kz_1d_s  = np.fft.fftshift(kz_ax)
        phi_1d_s = np.fft.fftshift(phi_1d)
        w_1d_s   = np.fft.fftshift(w_1d)

        # 1-D kx profile: amplitude-weighted mean phase over all kz in band
        phi_1d_kx   = np.zeros(Nx_pad)
        w_1d_kx     = np.zeros(Nx_pad)
        band_kz_fit = np.abs(kz_ax) < _kz_fac * kz_c
        for j_col in range(Nx_pad):
            sel_kz = band_kz_fit & (w[:, j_col] > WLS_AMP_THR * w.max())
            if sel_kz.sum() > 0:
                phi_1d_kx[j_col] = np.average(phi[:, j_col][sel_kz],
                                               weights=w[:, j_col][sel_kz])
                w_1d_kx[j_col]   = w[:, j_col][sel_kz].sum()
        kx_1d_s     = np.fft.fftshift(kx_ax)
        phi_1d_kx_s = np.fft.fftshift(phi_1d_kx)
        w_1d_kx_s   = np.fft.fftshift(w_1d_kx)

        fig_d, axes_d = plt.subplots(1, 5, figsize=(28, 4))

        # Panel 1: cross-spectrum phase (fftshifted, band only)
        band_shift = np.fft.fftshift(band)
        phi_masked = np.where(band_shift & (np.fft.fftshift(w) > WLS_AMP_THR * w.max()),
                              phi_shift, np.nan)
        im_ph = axes_d[0].imshow(phi_masked, aspect='auto', cmap='RdBu_r',
                                  extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]],
                                  vmin=-np.pi, vmax=np.pi, origin='upper')
        plt.colorbar(im_ph, ax=axes_d[0], label='phase [rad]')
        axes_d[0].set_title(f'Cross-spectrum phase  (mask: {n_mask} px)')
        axes_d[0].set_xlabel('kx [rad/m]');  axes_d[0].set_ylabel('kz [rad/m]')
        axes_d[0].set_xlim(kx_disp[0] / 3, kx_disp[-1] / 3)
        axes_d[0].set_ylim(kz_disp[-1] / 3, kz_disp[0] / 3)

        # Panel 2: cross-spectrum energy with WLS amplitude threshold contour
        w_plot = np.where(band_shift, w_shift, np.nan)
        im_en = axes_d[1].imshow(w_plot, aspect='auto', cmap='inferno',
                                  extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]],
                                  origin='upper')
        plt.colorbar(im_en, ax=axes_d[1], label='|XS| [a.u.]')
        axes_d[1].contour(kx_disp, kz_disp, w_shift,
                          levels=[WLS_AMP_THR * w.max()], colors='cyan', linewidths=0.8)
        axes_d[1].set_title(f'Energy  (thr={WLS_AMP_THR:.2f}×max — cyan contour)')
        axes_d[1].set_xlabel('kx [rad/m]');  axes_d[1].set_ylabel('kz [rad/m]')
        axes_d[1].set_xlim(kx_disp[0] / 3, kx_disp[-1] / 3)
        axes_d[1].set_ylim(kz_disp[-1] / 3, kz_disp[0] / 3)

        # Panel 3: fitted plane
        fitted_masked = np.where(band_shift, fitted_shift, np.nan)
        im_fit = axes_d[2].imshow(fitted_masked, aspect='auto', cmap='RdBu_r',
                                   extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]],
                                   vmin=-np.pi, vmax=np.pi, origin='upper')
        plt.colorbar(im_fit, ax=axes_d[2], label='phase [rad]')
        axes_d[2].set_title(f'Fitted plane  Δz={dz_est:+.4f} m  Δx={dx_est:+.4f} m  φ₀={phi_0:+.3f} rad')
        axes_d[2].set_xlabel('kx [rad/m]');  axes_d[2].set_ylabel('kz [rad/m]')
        axes_d[2].set_xlim(kx_disp[0] / 3, kx_disp[-1] / 3)
        axes_d[2].set_ylim(kz_disp[-1] / 3, kz_disp[0] / 3)

        # Panel 4: 1-D kz slice — measured vs fitted
        in_band = np.abs(kz_1d_s) < _kz_fac * kz_c
        sc_kz = axes_d[3].scatter(kz_1d_s[in_band & (w_1d_s > 0)],
                          phi_1d_s[in_band & (w_1d_s > 0)],
                          c=w_1d_s[in_band & (w_1d_s > 0)],
                          cmap='viridis', s=20, label='measured (weighted)')
        plt.colorbar(sc_kz, ax=axes_d[3], label='weight [a.u.]')
        kz_fit = kz_1d_s[in_band]
        axes_d[3].plot(kz_fit, kz_fit * dz_est + phi_0, 'r-', lw=1.5, label='fitted slope')
        axes_d[3].axhline(0, color='k', lw=0.5, ls='--')
        axes_d[3].set_xlabel('kz [rad/m]');  axes_d[3].set_ylabel('phase [rad]')
        axes_d[3].set_title('1-D kz phase profile (kx>0 only, both kz±)')
        axes_d[3].legend(fontsize=8)

        # Panel 5: 1-D kx slice — measured vs fitted
        in_band_kx = (kx_1d_s > 0) & (kx_1d_s < KX_BAND_FAC * kz_c)  # positive kx only
        sc_kx = axes_d[4].scatter(kx_1d_s[in_band_kx & (w_1d_kx_s > 0)],
                          phi_1d_kx_s[in_band_kx & (w_1d_kx_s > 0)],
                          c=w_1d_kx_s[in_band_kx & (w_1d_kx_s > 0)],
                          cmap='viridis', s=20, label='measured (weighted)')
        plt.colorbar(sc_kx, ax=axes_d[4], label='weight [a.u.]')
        kx_fit = kx_1d_s[in_band_kx]
        axes_d[4].plot(kx_fit, kx_fit * dx_est + phi_0, 'r-', lw=1.5, label='fitted slope')
        axes_d[4].axhline(0, color='k', lw=0.5, ls='--')
        axes_d[4].set_xlabel('kx [rad/m]');  axes_d[4].set_ylabel('phase [rad]')
        axes_d[4].set_title('1-D kx phase profile (positive kx only, kz± collapsed)')
        axes_d[4].legend(fontsize=8)

        plt.suptitle(f'Diagnostic: prof_{ref_run} → prof_{run_b}', y=1.02)
        plt.tight_layout()
        fig_d.savefig(roi_out / f'{ROI_METHOD}_diag_{ref_run}_to_{run_b}.png',
                      dpi=150, bbox_inches='tight')
        plt.show();  plt.close(fig_d)

    results[(ref_run, run_b)] = {
        'dx': dx_est, 'dz': dz_est, 'phi_0': phi_0,
        'roi_m':  (z_roi_bot, z_roi_top, x_roi_lo, x_roi_hi),
        'roi_px': (z0, z1, x0, x1),
        'n_mask': n_mask, 'source': roi_source,
    }
    print(f'prof_{ref_run}→{run_b}  [{roi_source}]:  '
          f'Δx={dx_est:+.4f} m  Δz={dz_est:+.4f} m  φ₀={phi_0:+.4f} rad  '
          f'mask={n_mask} px  crop={Nz}×{Nx} px  '
          f'ROI depth=[{z_roi_bot:.1f},{z_roi_top:.1f}] m  '
          f'radial=[{x_roi_lo:.2f},{x_roi_hi:.2f}] m')

# ── Summary plot ─────────────────────────────────────────────────────────────────────
if results:
    pair_labels = [f'{a}→{b}' for a, b in results]
    dx_vals  = [results[k]['dx']    for k in results]
    phi_vals = [results[k]['phi_0'] for k in results]

    fig2, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
    dz_vals = [results[k]['dz'] for k in results]
    from matplotlib.patches import Patch as _Patch
    _PHASE_DEFS = [('Pushing', 1, 3, '#aed6f1'), ('Chasing', 4, 8, '#a9dfbf'),
                   ('Waiting', 9, 20, '#f9e79f'), ('Pulling', 21, 38, '#f1948a')]
    _rn_col = {n: c for _nm, lo, hi, c in _PHASE_DEFS for n in range(lo, hi + 1)}
    _pkeys  = list(results.keys())
    for _bax in (ax1, ax2):
        for _i, _k in enumerate(_pkeys):
            _c = _rn_col.get(_k[1])
            if _c:
                _bax.axvspan(_i - 0.5, _i + 0.5, color=_c, alpha=0.3, zorder=0, lw=0)
    _ph_hdl = [_Patch(facecolor=c, alpha=0.6, label=nm, edgecolor='grey', lw=0.5)
               for nm, lo, hi, c in _PHASE_DEFS
               if any(_rn_col.get(_k[1]) == c for _k in _pkeys)]
    ax1.legend(handles=_ph_hdl, loc='best', fontsize=8, framealpha=0.7)
    ax1.plot(dx_vals, 'o-', color='steelblue')
    ax1.axhline(0, color='k', lw=0.6, ls='--')
    ax1.set_ylabel('Δx (radial) [m]')
    mode_str = f'vs prof_{COMPARE_TO}' if COMPARE_TO is not None else 'consecutive pairs'
    ax1.set_title(f'WLS fluid-front shift — {ROI_METHOD.capitalize()} ({mode_str})')
    ax2.plot(dz_vals, '^-', color='seagreen')
    ax2.axhline(0, color='k', lw=0.6, ls='--')
    ax2.set_ylabel('Δz (depth) [m]')
    plt.tight_layout()
    fig2.savefig(roi_out / f'{ROI_METHOD}_dx_summary.png', dpi=150)
    plt.show();  plt.close(fig2)

print(f'\nDone — {len(results)} pair(s).  Output: {roi_out}')

# Back-Propagation Results

## Why Δz Estimation Is Harder for Back-Propagation

Three structural differences make the WLS phase fit less reliable for BP Ez focus
frames than for Kirchhoff/Gazdag B-scans.

### 1. Flat cross-spectrum amplitude — no natural band

Kirchhoff/Gazdag migrate energy from diffraction hyperbolas into localized
reflectors, but the image still contains a wavelet envelope spread over many rows.
This gives useful kz content at non-zero wavenumbers and natural amplitude lobes
that guide band selection.

The BP Ez frame is a *focused* field: energy collapses to a small spatial blob.
A localized blob has a near-flat cross-spectrum amplitude (the Fourier transform of
an approximate delta function is approximately flat), so the amplitude weighting
loses its discriminatory power and there are no clear lobes to restrict the band to.

### 2. Near-DC contamination

The focused peak has a large DC component ($k_z \approx 0$, $k_x \approx 0$) that
dominates the cross-spectrum amplitude. The `(|KZ| + |KX|) > 0` mask excludes the
exact origin, but the near-DC region still carries high amplitude and little phase
information — diluting the $\Delta z$ signal for the WLS.

### 3. Focus-quality sensitivity

If the snapshot offset is slightly off, the difference between two BP frames contains
*defocusing artifacts* on top of the spatial shift. Unlike a B-scan where defocus
simply blurs the image, a defocused BP frame has a non-trivial phase field: the
cross-spectrum phase is no longer a clean linear ramp and the WLS fit is pulled
toward a biased solution.

---

### Approach: Tight ROI + Reduced Amplitude Threshold (Option C)

We stay with the WLS method but improve conditioning by two complementary changes.

**Tighter ROI** — restrict the analysis window to the immediate neighbourhood of the
focus peak (≈ 2–3 m depth × 1 m radial). A smaller crop excludes background noise
that dilutes the cross-spectrum, raises the effective SNR of the phase information,
and reduces contamination from off-focus reflectors.

**Lower `BP_WLS_AMP_THR`** — reducing the threshold from 0.20 to 0.10 allows more
k-space cells to contribute to the WLS fit. Because the cross-spectrum amplitude is
nearly flat (see point 1), a high threshold would exclude most of the k-space,
leaving very few WLS constraints.

These two changes act in opposite directions in terms of included data: the tighter
ROI makes the cross-spectrum *cleaner* (less noise energy), while the lower threshold
*expands* the k-space mask on that cleaner spectrum. Together they keep the WLS mask
well-populated with cells that carry genuine phase information.

> **Practical note:** start with the comparison grid (`mig_bp_comparison` cell) to
> locate the focus peak for each pair, then centre `BP_MANUAL_ROIS` on that peak.
> A window of ±1.5 m in depth and ±0.5 m in radial is a reasonable starting point.


In [ ]:
# ── Manual pair selection + ROI definition — Back-propagation Ez (full WLS) ─────────────
#
# BP_MANUAL_PAIRS       — list of (run_a, run_b) int tuples (same numbering as Kirchhoff).
#
# BP_COMPARE_TO         — None  → use pairs as written
#                         int   → override run_a for every pair to this run number.
#
# BP_MANUAL_ROIS        — per-pair ROI (z_min_m, z_max_m, x_min_m, x_max_m).
#                         Depth/radial in gprMax physical coordinates (metres).
#
# BP_SHOW_DIAG          — True: plot 5-panel cross-spectrum diagnostic per pair.
# BP_FORCE_DZ_ZERO      — True: fit only Δx + φ₀.
# BP_ROI_THRESH         — envelope threshold for auto-ROI.
# BP_FOCUS_IDX_OFFSET   — snapshot index offset (must match snapshot cell).
# BP_KZ_BAND_FAC        — default kz band factor (overridden per-pair by MANUAL_KZ_BAND_FAC_BP).
# BP_KX_BAND_FAC        — kx band factor.
# BP_WLS_AMP_THR        — amplitude gate for WLS mask.
# MANUAL_KZ_BAND_FAC_BP — per-pair override of BP_KZ_BAND_FAC.
# ────────────────────────────────────────────────────────────────────────────────────

BP_MANUAL_PAIRS = [
    # (1,  2), (2,  3), ...   # uncomment for consecutive pairs
    (1,  3),
    (4,  8),
    (9,  20),
    (21, 38),
]

BP_COMPARE_TO    = 1      # all pairs vs prof_1 (first injection measurement)
BP_FORCE_DZ_ZERO = False
BP_ROI_THRESH    = 0.5
BP_SHOW_DIAG     = True
BP_FOCUS_IDX_OFFSET = 28    # must match value in the snapshot cell

# Tight ROIs centred on the focus peak.
# BP focus is localised (~1-2 m depth x ~1 m radial); wider ROIs dilute the
# cross-spectrum with background noise. Adjust after inspecting mig_bp_comparison.
BP_MANUAL_ROIS = {
    (1,  3):  (71, 72, 4.5, 5.5),
    (4,  8):  (70, 79, 4.5, 6.5),
    (9,  20): (70, 78, 5.0, 7.0),
    (21, 38): (70, 79, 4.5, 6.5),
}

BP_KZ_BAND_FAC = 0.5     # default; overridden per-pair by MANUAL_KZ_BAND_FAC_BP
BP_KX_BAND_FAC = 1.5     # kx band: |kx| < BP_KX_BAND_FAC * kz_c_bp
# Lowered from 0.20: BP cross-spectrum amplitude is nearly flat (focused blob
# ~ delta function in space), so a high threshold excludes most of k-space.
BP_WLS_AMP_THR = 0.10
MANUAL_KZ_BAND_FAC_BP = {
    (9,  20): 0.25,
    (21, 38): 0.15,
}

# ────────────────────────────────────────────────────────────────────────────────────

bp_wls_out = OUT_DIR / 'roi_phase' / 'backprop_wls'
bp_wls_out.mkdir(parents=True, exist_ok=True)

kz_c_bp = 2.0 * np.pi * f0_mig / (v / 2)    # half-velocity for gprMax snapshots

def _load_bp_frame_wls(run_num, focus_offset):
    """Load the focus-frame Ez array for integer run number."""
    slug = f'prof_{run_num}'
    snap_files = completed.get(slug)
    if not snap_files:
        return None
    import pyvista
    snap_times_ns = t_start_ns + np.arange(len(snap_files)) * snap_step * dt_ns_bp
    idx = min(int(np.argmin(np.abs(snap_times_ns - t_focus_ns))) + focus_offset,
              len(snap_files) - 1)
    mesh   = pyvista.read(str(snap_files[idx]))
    nx_c   = mesh.dimensions[0] - 1
    ny_c   = mesh.dimensions[1] - 1
    dx_m   = float(mesh.spacing[0])
    e_data = np.array(mesh['E-field'])
    ez     = e_data[:, 2].reshape(ny_c, nx_c).T      # (nx_c, ny_c): rows=depth, cols=radial
    depth_axis  = np.linspace(0, nx_c * dx_m, nx_c)
    radial_axis = np.linspace(0, ny_c * dx_m, ny_c)
    return ez, depth_axis, radial_axis, dx_m

def _phys_to_pix_bp_wls(depth_axis, radial_axis, d_min, d_max, r_min, r_max):
    """Physical ROI → pixel indices. depth_axis is INCREASING (row 0 = 0 m)."""
    rows = np.where((depth_axis  >= d_min) & (depth_axis  <= d_max))[0]
    cols = np.where((radial_axis >= r_min) & (radial_axis <= r_max))[0]
    if rows.size == 0 or cols.size == 0:
        raise ValueError(
            f'ROI ({d_min}–{d_max} m depth, {r_min}–{r_max} m radial) '
            f'outside grid depth=[0,{depth_axis[-1]:.1f}] radial=[0,{radial_axis[-1]:.1f}]'
        )
    return int(rows[0]), int(rows[-1]) + 1, int(cols[0]), int(cols[-1]) + 1

bp_wls_results = {}
for run_a, run_b in BP_MANUAL_PAIRS:
    ref_run = BP_COMPARE_TO if BP_COMPARE_TO is not None else run_a
    res_a = _load_bp_frame_wls(ref_run, BP_FOCUS_IDX_OFFSET)
    res_b = _load_bp_frame_wls(run_b,   BP_FOCUS_IDX_OFFSET)
    if res_a is None or res_b is None:
        missing = ref_run if res_a is None else run_b
        print(f'[{ref_run}→{run_b}] prof_{missing} not in completed — skipping.')
        continue

    ez_a, depth_ax, radial_ax, dx_m = res_a
    ez_b, _,        _,          _   = res_b

    nd = min(ez_a.shape[0], ez_b.shape[0])
    nr = min(ez_a.shape[1], ez_b.shape[1])
    ez_a, ez_b = ez_a[:nd, :nr], ez_b[:nd, :nr]
    depth_ax   = depth_ax[:nd]
    radial_ax  = radial_ax[:nr]
    diff = ez_b - ez_a
    env  = _monogenic_envelope(diff)

    roi_spec = BP_MANUAL_ROIS.get((run_a, run_b)) or BP_MANUAL_ROIS.get((ref_run, run_b))
    if roi_spec is not None:
        d0, d1, r0, r1 = _phys_to_pix_bp_wls(depth_ax, radial_ax, *roi_spec)
        roi_source = 'manual'
    else:
        d0, d1, r0, r1 = _roi_from_envelope(env, BP_ROI_THRESH)
        roi_source = f'auto (thresh={BP_ROI_THRESH})'

    d_roi_lo = float(depth_ax[d0]);   d_roi_hi = float(depth_ax[d1 - 1])
    r_roi_lo = float(radial_ax[r0]);  r_roi_hi = float(radial_ax[r1 - 1])

    # ── Difference + envelope figure ──────────────────────────────────────────────────────────────────
    extent_bp = [0, float(radial_ax[-1]), 0, float(depth_ax[-1])]
    vmax_d = np.percentile(np.abs(diff), 99.9)
    vmax_e = np.percentile(env, 99.9)

    fig, (ax_d, ax_e) = plt.subplots(1, 2, figsize=(14, 5))
    im_d = ax_d.imshow(diff, aspect='auto', cmap='RdBu_r',
                        extent=extent_bp, origin='lower', vmin=-vmax_d, vmax=vmax_d)
    plt.colorbar(im_d, ax=ax_d, label='ΔEz [V/m]')
    ax_d.invert_yaxis()
    ax_d.add_patch(Rectangle((r_roi_lo, d_roi_lo),
                              width=r_roi_hi - r_roi_lo, height=d_roi_hi - d_roi_lo,
                              lw=1.5, edgecolor='yellow', facecolor='none'))
    ax_d.xaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_d.yaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_d.grid(True, color='white', lw=0.3, alpha=0.4)
    ax_d.set_title(f'Back-prop ΔEz: prof_{run_b} − prof_{ref_run}  [{roi_source}]')
    ax_d.set_xlabel('Radial distance (m)'); ax_d.set_ylabel('Depth (m)')
    ax_d.set_ylim(62, 85)

    im_e = ax_e.imshow(env, aspect='auto', cmap='inferno',
                        extent=extent_bp, origin='lower', vmin=0, vmax=vmax_e)
    plt.colorbar(im_e, ax=ax_e, label='Monogenic envelope [a.u.]')
    ax_e.invert_yaxis()
    ax_e.add_patch(Rectangle((r_roi_lo, d_roi_lo),
                              width=r_roi_hi - r_roi_lo, height=d_roi_hi - d_roi_lo,
                              lw=1.5, edgecolor='cyan', facecolor='none'))
    ax_e.xaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_e.yaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_e.grid(True, color='white', lw=0.3, alpha=0.4)
    ax_e.set_title(f'Monogenic envelope  [{roi_source}]')
    ax_e.set_xlabel('Radial distance (m)'); ax_e.set_ylabel('Depth (m)')
    ax_e.set_ylim(62, 85)

    plt.tight_layout()
    fig.savefig(bp_wls_out / f'backprop_roi_{ref_run}_to_{run_b}.png', dpi=150)
    plt.show(); plt.close(fig)

    # ── WLS phase-plane fit (zero-padded for denser kx/kz sampling) ───────────────────────────
    base_crop = ez_a[d0:d1, r0:r1]
    mon_crop  = ez_b[d0:d1, r0:r1]
    Nz, Nx    = base_crop.shape
    pad_fac   = 10
    Nz_pad, Nx_pad = Nz * pad_fac, Nx * pad_fac

    dz_bp = dx_bp = dx_m    # uniform gprMax grid: same spacing in both axes
    kz_ax = np.fft.fftfreq(Nz_pad, d=dz_bp) * 2 * np.pi
    kx_ax = np.fft.fftfreq(Nx_pad, d=dx_bp) * 2 * np.pi
    KZ, KX = np.meshgrid(kz_ax, kx_ax, indexing='ij')
    taper  = np.outer(tukey(Nz, alpha=0.15), tukey(Nx, alpha=0.15))
    XS     = (np.fft.fft2(base_crop * taper, s=(Nz_pad, Nx_pad)) *
               np.conj(np.fft.fft2(mon_crop * taper, s=(Nz_pad, Nx_pad))))
    w      = np.abs(XS);  phi = np.angle(XS)
    _kz_fac = (MANUAL_KZ_BAND_FAC_BP.get((run_a, run_b))
               or MANUAL_KZ_BAND_FAC_BP.get((ref_run, run_b))
               or BP_KZ_BAND_FAC)
    band   = (np.abs(KZ) < _kz_fac * kz_c_bp) & (np.abs(KX) < BP_KX_BAND_FAC * kz_c_bp)
    mask   = (w > BP_WLS_AMP_THR * w.max()) & band & ((np.abs(KZ) + np.abs(KX)) > 0)

    n_mask = int(mask.sum())
    if n_mask < 3:
        print(f'[{ref_run}→{run_b}] WARNING: only {n_mask} pixels pass WLS mask '
              f'(crop {Nz}×{Nx} px, padded {Nz_pad}×{Nx_pad}).  ROI may be too small.')
        dz_est = dx_est = phi_0 = 0.0
    else:
        W = w[mask]
        if BP_FORCE_DZ_ZERO:
            A = np.column_stack([KX[mask], np.ones(n_mask)])
        else:
            A = np.column_stack([KZ[mask], KX[mask], np.ones(n_mask)])
        c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
        if BP_FORCE_DZ_ZERO:
            dz_est, dx_est, phi_0 = 0.0, float(c[0]), float(c[1])
        else:
            dz_est, dx_est, phi_0 = float(c[0]), float(c[1]), float(c[2])

    # ── Diagnostic: cross-spectrum phase + fitted plane ──────────────────────────────────
    if BP_SHOW_DIAG and n_mask >= 3:
        phi_shift    = np.fft.fftshift(phi)
        w_shift      = np.fft.fftshift(w)
        kz_disp      = np.fft.fftshift(kz_ax)
        kx_disp      = np.fft.fftshift(kx_ax)
        fitted       = KX * dx_est + KZ * dz_est + phi_0
        fitted_shift = np.fft.fftshift(fitted)

        # 1-D kz profile
        phi_1d  = np.zeros(Nz_pad)
        w_1d    = np.zeros(Nz_pad)
        band_kx = (kx_ax > 0) & (kx_ax < BP_KX_BAND_FAC * kz_c_bp)  # positive kx only
        for i_row in range(Nz_pad):
            sel = band_kx & (w[i_row, :] > BP_WLS_AMP_THR * w.max())
            if sel.sum() > 0:
                phi_1d[i_row] = np.average(phi[i_row, :][sel], weights=w[i_row, :][sel])
                w_1d[i_row]   = w[i_row, :][sel].sum()
        kz_1d_s  = np.fft.fftshift(kz_ax)
        phi_1d_s = np.fft.fftshift(phi_1d)
        w_1d_s   = np.fft.fftshift(w_1d)

        # 1-D kx profile
        phi_1d_kx   = np.zeros(Nx_pad)
        w_1d_kx     = np.zeros(Nx_pad)
        band_kz_fit = np.abs(kz_ax) < _kz_fac * kz_c_bp
        for j_col in range(Nx_pad):
            sel_kz = band_kz_fit & (w[:, j_col] > BP_WLS_AMP_THR * w.max())
            if sel_kz.sum() > 0:
                phi_1d_kx[j_col] = np.average(phi[:, j_col][sel_kz],
                                               weights=w[:, j_col][sel_kz])
                w_1d_kx[j_col]   = w[:, j_col][sel_kz].sum()
        kx_1d_s     = np.fft.fftshift(kx_ax)
        phi_1d_kx_s = np.fft.fftshift(phi_1d_kx)
        w_1d_kx_s   = np.fft.fftshift(w_1d_kx)

        fig_d, axes_d = plt.subplots(1, 5, figsize=(28, 4))
        band_shift = np.fft.fftshift(band)
        phi_masked = np.where(band_shift & (np.fft.fftshift(w) > BP_WLS_AMP_THR * w.max()),
                              phi_shift, np.nan)

        # Panel 1: cross-spectrum phase (full kx/kz shown; WLS uses kx>=0 only)
        im_ph = axes_d[0].imshow(phi_masked, aspect='auto', cmap='RdBu_r',
                                  extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]],
                                  vmin=-np.pi, vmax=np.pi, origin='upper')
        plt.colorbar(im_ph, ax=axes_d[0], label='phase [rad]')
        axes_d[0].set_title(f'Cross-spectrum phase  (mask: {n_mask} px)')
        axes_d[0].set_xlabel('kx [rad/m]'); axes_d[0].set_ylabel('kz [rad/m]')
        axes_d[0].set_xlim(kx_disp[0] / 3, kx_disp[-1] / 3)
        axes_d[0].set_ylim(kz_disp[-1] / 3, kz_disp[0] / 3)

        # Panel 2: cross-spectrum energy with WLS amplitude threshold contour
        w_plot = np.where(band_shift, w_shift, np.nan)
        im_en = axes_d[1].imshow(w_plot, aspect='auto', cmap='inferno',
                                  extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]],
                                  origin='upper')
        plt.colorbar(im_en, ax=axes_d[1], label='|XS| [a.u.]')
        axes_d[1].contour(kx_disp, kz_disp, w_shift,
                          levels=[BP_WLS_AMP_THR * w.max()], colors='cyan', linewidths=0.8)
        axes_d[1].set_title(f'Energy  (thr={BP_WLS_AMP_THR:.2f}×max — cyan contour)')
        axes_d[1].set_xlabel('kx [rad/m]'); axes_d[1].set_ylabel('kz [rad/m]')
        axes_d[1].set_xlim(kx_disp[0] / 3, kx_disp[-1] / 3)
        axes_d[1].set_ylim(kz_disp[-1] / 3, kz_disp[0] / 3)

        # Panel 3: fitted plane
        fitted_masked = np.where(band_shift, fitted_shift, np.nan)
        im_fit = axes_d[2].imshow(fitted_masked, aspect='auto', cmap='RdBu_r',
                                   extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]],
                                   vmin=-np.pi, vmax=np.pi, origin='upper')
        plt.colorbar(im_fit, ax=axes_d[2], label='phase [rad]')
        axes_d[2].set_title(f'Fitted plane  Δz={dz_est:+.4f} m  Δx={dx_est:+.4f} m  φ₀={phi_0:+.3f} rad')
        axes_d[2].set_xlabel('kx [rad/m]'); axes_d[2].set_ylabel('kz [rad/m]')
        axes_d[2].set_xlim(kx_disp[0] / 3, kx_disp[-1] / 3)
        axes_d[2].set_ylim(kz_disp[-1] / 3, kz_disp[0] / 3)

        # Panel 4: 1-D kz slice — measured vs fitted
        in_band = np.abs(kz_1d_s) < _kz_fac * kz_c_bp
        sc_kz = axes_d[3].scatter(kz_1d_s[in_band & (w_1d_s > 0)],
                          phi_1d_s[in_band & (w_1d_s > 0)],
                          c=w_1d_s[in_band & (w_1d_s > 0)],
                          cmap='viridis', s=20, label='measured (weighted)')
        plt.colorbar(sc_kz, ax=axes_d[3], label='weight [a.u.]')
        kz_fit_ax = kz_1d_s[in_band]
        axes_d[3].plot(kz_fit_ax, kz_fit_ax * dz_est + phi_0, 'r-', lw=1.5, label='fitted slope')
        axes_d[3].axhline(0, color='k', lw=0.5, ls='--')
        axes_d[3].set_xlabel('kz [rad/m]'); axes_d[3].set_ylabel('phase [rad]')
        axes_d[3].set_title('1-D kz phase profile (kx>0 only, both kz±)')
        axes_d[3].legend(fontsize=8)

        # Panel 5: 1-D kx slice — measured vs fitted
        in_band_kx = (kx_1d_s > 0) & (kx_1d_s < BP_KX_BAND_FAC * kz_c_bp)  # positive kx only
        sc_kx = axes_d[4].scatter(kx_1d_s[in_band_kx & (w_1d_kx_s > 0)],
                          phi_1d_kx_s[in_band_kx & (w_1d_kx_s > 0)],
                          c=w_1d_kx_s[in_band_kx & (w_1d_kx_s > 0)],
                          cmap='viridis', s=20, label='measured (weighted)')
        plt.colorbar(sc_kx, ax=axes_d[4], label='weight [a.u.]')
        kx_fit_ax = kx_1d_s[in_band_kx]
        axes_d[4].plot(kx_fit_ax, kx_fit_ax * dx_est + phi_0, 'r-', lw=1.5, label='fitted slope')
        axes_d[4].axhline(0, color='k', lw=0.5, ls='--')
        axes_d[4].set_xlabel('kx [rad/m]'); axes_d[4].set_ylabel('phase [rad]')
        axes_d[4].set_title('1-D kx phase profile (positive kx only, kz± collapsed)')
        axes_d[4].legend(fontsize=8)

        plt.suptitle(f'Diagnostic (back-prop): prof_{ref_run} → prof_{run_b}', y=1.02)
        plt.tight_layout()
        fig_d.savefig(bp_wls_out / f'backprop_diag_{ref_run}_to_{run_b}.png',
                      dpi=150, bbox_inches='tight')
        plt.show(); plt.close(fig_d)

    bp_wls_results[(ref_run, run_b)] = {
        'dx': dx_est, 'dz': dz_est, 'phi_0': phi_0,
        'roi_m':  (d_roi_lo, d_roi_hi, r_roi_lo, r_roi_hi),
        'roi_px': (d0, d1, r0, r1),
        'n_mask': n_mask, 'source': roi_source,
    }
    print(f'prof_{ref_run}→{run_b}  [{roi_source}]:  '
          f'Δx={dx_est:+.4f} m  Δz={dz_est:+.4f} m  φ₀={phi_0:+.4f} rad  '
          f'mask={n_mask} px  crop={Nz}×{Nx} px  '
          f'ROI depth=[{d_roi_lo:.1f},{d_roi_hi:.1f}] m  '
          f'radial=[{r_roi_lo:.2f},{r_roi_hi:.2f}] m')

# ── Summary plot ─────────────────────────────────────────────────────────────────────────────────────
if bp_wls_results:
    pair_labels = [f'{a}→{b}' for a, b in bp_wls_results]
    dx_vals = [bp_wls_results[k]['dx'] for k in bp_wls_results]
    dz_vals = [bp_wls_results[k]['dz'] for k in bp_wls_results]

    fig2, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
    from matplotlib.patches import Patch as _Patch
    _PHASE_DEFS = [('Pushing', 1, 3, '#aed6f1'), ('Chasing', 4, 8, '#a9dfbf'),
                   ('Waiting', 9, 20, '#f9e79f'), ('Pulling', 21, 38, '#f1948a')]
    _rn_col = {n: c for _nm, lo, hi, c in _PHASE_DEFS for n in range(lo, hi + 1)}
    _pkeys  = list(bp_wls_results.keys())
    for _bax in (ax1, ax2):
        for _i, _k in enumerate(_pkeys):
            _c = _rn_col.get(_k[1])
            if _c:
                _bax.axvspan(_i - 0.5, _i + 0.5, color=_c, alpha=0.3, zorder=0, lw=0)
    _ph_hdl = [_Patch(facecolor=c, alpha=0.6, label=nm, edgecolor='grey', lw=0.5)
               for nm, lo, hi, c in _PHASE_DEFS
               if any(_rn_col.get(_k[1]) == c for _k in _pkeys)]
    ax1.legend(handles=_ph_hdl, loc='best', fontsize=8, framealpha=0.7)
    ax1.plot(dx_vals, 'o-', color='steelblue')
    ax1.axhline(0, color='k', lw=0.6, ls='--')
    ax1.set_ylabel('Δx (radial) [m]')
    mode_str = f'vs prof_{BP_COMPARE_TO}' if BP_COMPARE_TO is not None else 'stage pairs'
    ax1.set_title(f'WLS fluid-front shift — Back-propagation ({mode_str})')
    ax2.plot(dz_vals, '^-', color='seagreen')
    ax2.axhline(0, color='k', lw=0.6, ls='--')
    ax2.set_ylabel('Δz (depth) [m]')
    ax2.set_xticks(range(len(pair_labels)))
    ax2.set_xticklabels(pair_labels, rotation=30, ha='right')
    plt.tight_layout()
    fig2.savefig(bp_wls_out / 'backprop_wls_summary.png', dpi=150)
    plt.show(); plt.close(fig2)

print(f'\nDone — {len(bp_wls_results)} pair(s).  Output: {bp_wls_out}')


In [ ]:
# ── Manual pair selection + ROI definition — Back-propagation Ez focus frames ────────
#
# MANUAL_PAIRS_BP  — list of (slug_a, slug_b) string tuples from the `completed` dict.
#                    e.g. ('prof_1', 'prof_4').
#
# MANUAL_ROIS_BP   — per-pair ROI in gprMax physical coordinates (metres):
#                      (slug_a, slug_b): (depth_min_m, depth_max_m, radial_min_m, radial_max_m)
#                    depth_min/max refer to the gprMax x-axis (borehole source positions,
#                    0 → 86 m; sources sit between ~62 m and 85 m in this dataset).
#                    radial_min/max refer to the gprMax y-axis (0 → ~14 m from borehole).
#                    Set a pair's value to None to auto-detect from the monogenic envelope.
#
# FOCUS_IDX_OFFSET_BP — snapshot index offset, must match the value in the snapshot cell.
# BP_FORCE_DZ_ZERO    — True: fit only Δx + φ₀ (fluid front is lateral only).
# BP_ROI_THRESH       — envelope threshold for auto-detected pairs (0–1).
# ─────────────────────────────────────────────────────────────────────────────────────

MANUAL_PAIRS_BP = [
    ('prof_1',  'prof_2'),
    ('prof_2',  'prof_3'),
    ('prof_3',  'prof_4'),
    ('prof_4', 'prof_5'),
    ('prof_5', 'prof_7'),
    ('prof_7', 'prof_8'),
    ('prof_8', 'prof_9'),
    ('prof_9', 'prof_10')
]

MANUAL_ROIS_BP = {
    ('prof_1',  'prof_2'): (70, 78, 5.0, 6.5),
    ('prof_2',  'prof_3'): (70, 78, 5.0, 6.5),
    ('prof_3',  'prof_4'): (70, 78, 5.0, 6.5),
    ('prof_4', 'prof_5'): (70, 78, 5.0, 6.5),
    ('prof_5', 'prof_7'): (70, 78, 5.0, 6.5),
    ('prof_7', 'prof_8'): (70, 78, 5.0, 6.5),
    ('prof_8', 'prof_9'): (70, 78, 5.0, 6.5),
    ('prof_9', 'prof_10'): (70, 78, 5.0, 6.5)
}

FOCUS_IDX_OFFSET_BP = 26    # must match offset used in the snapshot cell above
BP_FORCE_DZ_ZERO    = True
BP_ROI_THRESH       = 0.5

# ─────────────────────────────────────────────────────────────────────────────────────

bp_roi_out = OUT_DIR / 'roi_phase' / 'backprop'
bp_roi_out.mkdir(parents=True, exist_ok=True)

def _load_bp_ez(slug):
    """Load the focus-frame Ez array for a given backprop slug.
    Returns (ez, t_actual_ns, depth_axis_m, radial_axis_m, dx_m).
    """
    snap_files = completed.get(slug)
    if not snap_files:
        return None
    import pyvista
    snap_times_ns = t_start_ns + np.arange(len(snap_files)) * snap_step * dt_ns_bp
    idx = min(int(np.argmin(np.abs(snap_times_ns - t_focus_ns))) + FOCUS_IDX_OFFSET_BP,
              len(snap_files) - 1)
    mesh   = pyvista.read(str(snap_files[idx]))
    nx_c   = mesh.dimensions[0] - 1
    ny_c   = mesh.dimensions[1] - 1
    dx_m   = float(mesh.spacing[0])
    e_data = np.array(mesh['E-field'])
    ez     = e_data[:, 2].reshape(ny_c, nx_c).T      # (nx_c, ny_c): rows=depth, cols=radial
    depth_axis  = np.linspace(0, nx_c * dx_m, nx_c)  # gprMax x: 0 → domain_x [m]
    radial_axis = np.linspace(0, ny_c * dx_m, ny_c)  # gprMax y: 0 → domain_y [m]
    return ez, snap_times_ns[idx], depth_axis, radial_axis, dx_m

def _phys_to_pix_bp(depth_axis, radial_axis, d_min, d_max, r_min, r_max):
    """Convert physical-coordinate ROI to pixel indices for the backprop Ez array.
    depth_axis is INCREASING (row 0 = 0 m, last row = domain_x ≈ 86 m).
    Returns (d0, d1, r0, r1) as exclusive-end integer indices.
    """
    row_mask = (depth_axis  >= d_min) & (depth_axis  <= d_max)
    col_mask = (radial_axis >= r_min) & (radial_axis <= r_max)
    rows = np.where(row_mask)[0]
    cols = np.where(col_mask)[0]
    if rows.size == 0 or cols.size == 0:
        raise ValueError(
            f'ROI ({d_min}–{d_max} m depth, {r_min}–{r_max} m radial) '
            f'is outside grid  depth=[0, {depth_axis[-1]:.1f}] m  '
            f'radial=[0, {radial_axis[-1]:.1f}] m'
        )
    return int(rows[0]), int(rows[-1]) + 1, int(cols[0]), int(cols[-1]) + 1

bp_results = {}
for slug_a, slug_b in MANUAL_PAIRS_BP:
    res_a = _load_bp_ez(slug_a)
    res_b = _load_bp_ez(slug_b)
    if res_a is None or res_b is None:
        missing = slug_a if res_a is None else slug_b
        print(f'[{slug_a}→{slug_b}] {missing} not in completed dict — skipping.')
        continue

    ez_a, t_a, depth_ax, radial_ax, dx_m = res_a
    ez_b, t_b, _,        _,         _    = res_b

    # Align shapes (should be identical, but guard against edge cases)
    nd = min(ez_a.shape[0], ez_b.shape[0])
    nr = min(ez_a.shape[1], ez_b.shape[1])
    ez_a, ez_b   = ez_a[:nd, :nr], ez_b[:nd, :nr]
    depth_ax     = depth_ax[:nd]
    radial_ax    = radial_ax[:nr]
    diff = ez_b - ez_a
    env  = _monogenic_envelope(diff)

    # Resolve ROI
    roi_spec = MANUAL_ROIS_BP.get((slug_a, slug_b), None)
    if roi_spec is not None:
        d_min, d_max, r_min, r_max = roi_spec
        d0, d1, r0, r1 = _phys_to_pix_bp(depth_ax, radial_ax, d_min, d_max, r_min, r_max)
        roi_source = 'manual'
    else:
        d0, d1, r0, r1 = _roi_from_envelope(env, BP_ROI_THRESH)
        roi_source = f'auto (thresh={BP_ROI_THRESH})'

    d_roi_lo = float(depth_ax[d0])
    d_roi_hi = float(depth_ax[d1 - 1])
    r_roi_lo = float(radial_ax[r0])
    r_roi_hi = float(radial_ax[r1 - 1])

    # extent convention: [x_min, x_max, y_min, y_max] with origin='lower' + invert_yaxis
    # → x = radial, y = depth with 0 at top after invert
    extent = [0, float(radial_ax[-1]), 0, float(depth_ax[-1])]
    vmax_d = np.percentile(np.abs(diff), 98)
    vmax_e = np.percentile(env, 98)

    fig, (ax_d, ax_e) = plt.subplots(1, 2, figsize=(14, 5))

    im_d = ax_d.imshow(diff, aspect='auto', cmap='RdBu_r',
                        extent=extent, origin='lower',
                        vmin=-vmax_d, vmax=vmax_d)
    plt.colorbar(im_d, ax=ax_d, label='ΔEz [V/m]')
    ax_d.invert_yaxis()
    ax_d.add_patch(Rectangle((r_roi_lo, d_roi_lo),
                              width=r_roi_hi - r_roi_lo,
                              height=d_roi_hi - d_roi_lo,
                              lw=1.5, edgecolor='yellow', facecolor='none'))
    ax_d.xaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_d.yaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_d.grid(True, color='white', linewidth=0.3, alpha=0.4)
    ax_d.set_title(f'Back-prop ΔEz: {slug_b} − {slug_a}  [{roi_source}]')
    ax_d.set_xlabel('Radial distance (m)')
    ax_d.set_ylabel('Depth (m)')

    im_e = ax_e.imshow(env, aspect='auto', cmap='inferno',
                        extent=extent, origin='lower',
                        vmin=0, vmax=vmax_e)
    plt.colorbar(im_e, ax=ax_e, label='Monogenic envelope [a.u.]')
    ax_e.invert_yaxis()
    ax_e.add_patch(Rectangle((r_roi_lo, d_roi_lo),
                              width=r_roi_hi - r_roi_lo,
                              height=d_roi_hi - d_roi_lo,
                              lw=1.5, edgecolor='cyan', facecolor='none'))
    ax_e.xaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_e.yaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_e.grid(True, color='white', linewidth=0.3, alpha=0.4)
    ax_e.set_title(f'Monogenic envelope  [{roi_source}]')
    ax_e.set_xlabel('Radial distance (m)')
    ax_e.set_ylabel('Depth (m)')

    plt.tight_layout()
    fname = f'bp_roi_{slug_a}_to_{slug_b}.png'
    fig.savefig(bp_roi_out / fname, dpi=150)
    plt.show()
    plt.close(fig)

    # WLS phase-plane fit — uniform grid so dz_g = dx_g = dx_m
    # kz_c uses half-velocity (eps_r * 4 → v_half = v/2) since gprMax runs at v/2
    kz_c_bp = 2.0 * np.pi * f0_mig / (v / 2)
    base_crop = ez_a[d0:d1, r0:r1]
    mon_crop  = ez_b[d0:d1, r0:r1]
    dz_est, dx_est, phi_0 = _estimate_shift_2d(
        base_crop, mon_crop, dx_m, dx_m, kz_c_bp, force_dz_zero=False)

    bp_results[(slug_a, slug_b)] = {
        'dx': dx_est, 'dz': dz_est, 'phi_0': phi_0,
        'roi_m':  (d_roi_lo, d_roi_hi, r_roi_lo, r_roi_hi),
        'roi_px': (d0, d1, r0, r1),
        'source': roi_source,
    }
    print(f'{slug_a}→{slug_b}  [{roi_source}]:  '
          f'Δx={dx_est:+.4f} m  Δz={dz_est:+.4f} m  φ₀={phi_0:+.4f} rad  '
          f'ROI depth=[{d_roi_lo:.1f}, {d_roi_hi:.1f}] m  '
          f'radial=[{r_roi_lo:.2f}, {r_roi_hi:.2f}] m  '
          f'size={d1-d0}×{r1-r0} px')

# ── Summary plot ─────────────────────────────────────────────────────────────────────
if bp_results:
    pair_labels = [f'{a.split("_")[1]}→{b.split("_")[1]}' for a, b in bp_results]
    dx_vals  = [bp_results[k]['dx']    for k in bp_results]
    phi_vals = [bp_results[k]['phi_0'] for k in bp_results]

    fig2, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
    dz_vals = [bp_results[k]['dz'] for k in bp_results]
    from matplotlib.patches import Patch as _Patch
    _PHASE_DEFS = [('Pushing', 1, 3, '#aed6f1'), ('Chasing', 4, 8, '#a9dfbf'),
                   ('Waiting', 9, 20, '#f9e79f'), ('Pulling', 21, 38, '#f1948a')]
    _rn_col = {n: c for _nm, lo, hi, c in _PHASE_DEFS for n in range(lo, hi + 1)}
    _pkeys  = list(bp_results.keys())
    for _bax in (ax1, ax2, ax3):
        for _i, _k in enumerate(_pkeys):
            _c = _rn_col.get(int(_k[1].split('_')[1]))
            if _c:
                _bax.axvspan(_i - 0.5, _i + 0.5, color=_c, alpha=0.3, zorder=0, lw=0)
    _ph_hdl = [_Patch(facecolor=c, alpha=0.6, label=nm, edgecolor='grey', lw=0.5)
               for nm, lo, hi, c in _PHASE_DEFS
               if any(_rn_col.get(int(_k[1].split('_')[1])) == c for _k in _pkeys)]
    ax1.legend(handles=_ph_hdl, loc='best', fontsize=8, framealpha=0.7)
    ax1.plot(dx_vals, 'o-', color='steelblue')
    ax1.axhline(0, color='k', lw=0.6, ls='--')
    ax1.set_ylabel('Δx (radial) [m]')
    ax1.set_title('Localised WLS fluid-front shift — Back-propagation Ez')
    ax2.plot(dz_vals, '^-', color='seagreen')
    ax2.axhline(0, color='k', lw=0.6, ls='--')
    ax2.set_ylabel('Δz (depth) [m]')
    ax3.plot(np.degrees(phi_vals), 's-', color='darkorange')
    ax3.axhline(0, color='k', lw=0.6, ls='--')
    ax3.set_ylabel('φ₀ [°]')
    ax3.set_xlabel('Pair')
    ax3.set_xticks(range(len(pair_labels)))
    ax3.set_xticklabels(pair_labels, rotation=30, ha='right')
    plt.tight_layout()
    fig2.savefig(bp_roi_out / 'bp_dx_summary.png', dpi=150)
    plt.show()
    plt.close(fig2)

print(f'\nDone — {len(bp_results)} pair(s) analysed.  Output: {bp_roi_out}')

# Fluid Front Movement Estimate:

## 1\. Consecutive Profiles ($1 \rightarrow 2, 2 \rightarrow 3, \dots, 37 \rightarrow 38$)

Pushing phase: profiles 1-3

Chasing phase: profiles 4-8

Waiting phase: profiles 9-20

Pulling phase: profiles 21-38

In [ ]:
# ── Strategy 1: Consecutive Profiles (n → n+1) ───────────────────────────────────
# Pro: δx ≪ λ/4 ⇒ phase always in [−π, +π], high waveform correlation.
# Con: integrating 37 independent noise terms ⇒ random-walk drift ∝ √N.
# Summary shows CUMULATIVE displacement (sum of incremental δx values).
# ─────────────────────────────────────────────────────────────────────────────────────

S1_ROI          = (70, 79, 4.5, 7.5)   # [z_min, z_max, r_min, r_max] m
S1_METHOD       = 'gazdag'
S1_FORCE_DZ_ZERO = False

pairs_s1   = [(n, n + 1) for n in range(1, 38)]
results_s1 = {}

for _ra, _rb in pairs_s1:
    _a = _load_img(S1_METHOD, _ra);  _b = _load_img(S1_METHOD, _rb)
    if _a is None or _b is None:
        print(f'[{_ra}->{_rb}] missing .npy — skipped')
        continue
    _n = min(_a.shape[0], _b.shape[0])
    _a, _b = _a[:_n], _b[:_n]
    _z0, _z1, _x0, _x1 = _phys_to_pix(depth[:_n], x_img, *S1_ROI)
    _dz, _dx, _phi = _estimate_shift_2d(
        _a[_z0:_z1, _x0:_x1], _b[_z0:_z1, _x0:_x1],
        dz_g, dx_g, kz_c, force_dz_zero=S1_FORCE_DZ_ZERO)
    results_s1[(_ra, _rb)] = {'dx': _dx, 'dz': _dz, 'phi_0': _phi}
    print(f'[{_ra}->{_rb}]  dx={_dx:+.4f} m  dz={_dz:+.4f} m  phi0={_phi:+.4f} rad')

# ── Summary: cumulative (integrated) displacement ─────────────────────────────────
if results_s1:
    from matplotlib.patches import Patch as _Patch
    _PHASE_DEFS = [('Pushing', 1, 3, '#aed6f1'), ('Chasing', 4, 8, '#a9dfbf'),
                   ('Waiting', 9, 20, '#f9e79f'), ('Pulling', 21, 38, '#f1948a')]
    _rn_col = {n: c for _nm, lo, hi, c in _PHASE_DEFS for n in range(lo, hi + 1)}
    _pkeys = list(results_s1.keys())
    dx_cum = np.cumsum([results_s1[k]['dx'] for k in _pkeys])
    dz_cum = np.cumsum([results_s1[k]['dz'] for k in _pkeys])
    phi_v  = [results_s1[k]['phi_0'] for k in _pkeys]
    labels = [f'{a}->{b}' for a, b in _pkeys]

    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
    for _bax in (ax1, ax2, ax3):
        for _i, _k in enumerate(_pkeys):
            _c = _rn_col.get(_k[1])
            if _c:
                _bax.axvspan(_i - 0.5, _i + 0.5, color=_c, alpha=0.3, zorder=0, lw=0)
    _ph_hdl = [_Patch(facecolor=c, alpha=0.6, label=nm, edgecolor='grey', lw=0.5)
               for nm, lo, hi, c in _PHASE_DEFS if any(_rn_col.get(_k[1]) == c for _k in _pkeys)]
    ax1.plot(dx_cum, 'o-', color='steelblue')
    ax1.axhline(0, color='k', lw=0.6, ls='--')
    ax1.set_ylabel('Cumulative Δx (radial) [m]')
    ax1.set_title(f'Strategy 1: Consecutive  [{S1_METHOD}]  (σ_drift ∝ √N)')
    ax1.legend(handles=_ph_hdl, loc='best', fontsize=8)
    ax2.plot(dz_cum, '^-', color='seagreen')
    ax2.axhline(0, color='k', lw=0.6, ls='--')
    ax2.set_ylabel('Cumulative Δz (depth) [m]')
    ax3.plot(np.degrees(phi_v), 's-', color='darkorange')
    ax3.axhline(0, color='k', lw=0.6, ls='--')
    ax3.set_ylabel('φ₀ [°]')
    ax3.set_xlabel('Pair')
    ax3.set_xticks(range(len(labels)))
    ax3.set_xticklabels(labels, rotation=45, ha='right', fontsize=6)
    plt.tight_layout()
    fig.savefig(OUT_DIR / 'roi_phase' / 'strat1_consecutive_summary.png', dpi=150)
    save_fig(fig, 'strat1_consecutive_summary', study='FieldData_Study', prefix='FD_')
    plt.show();  plt.close(fig)


## 2\. Fixed Global Baseline ($1 \rightarrow 2, 1 \rightarrow 3, \dots, 1 \rightarrow 38$)

In [ ]:
# ── Strategy 2: Fixed Global Baseline (1 → n for all n) ─────────────────────────
# Pro: independent estimates ⇒ zero error accumulation.
# Con: phase wrapping when net shift > λ/4; waveform decorrelation in late profiles.
# Summary shows DIRECT displacement relative to the pre-injection baseline (Profile 1).
# ─────────────────────────────────────────────────────────────────────────────────────

S2_ROI          = (70, 79, 4.5, 7.5)
S2_METHOD       = 'gazdag'
S2_FORCE_DZ_ZERO = False

pairs_s2   = [(1, n) for n in range(2, 39)]
results_s2 = {}

for _ra, _rb in pairs_s2:
    _a = _load_img(S2_METHOD, _ra);  _b = _load_img(S2_METHOD, _rb)
    if _a is None or _b is None:
        print(f'[{_ra}->{_rb}] missing .npy — skipped')
        continue
    _n = min(_a.shape[0], _b.shape[0])
    _a, _b = _a[:_n], _b[:_n]
    _z0, _z1, _x0, _x1 = _phys_to_pix(depth[:_n], x_img, *S2_ROI)
    _dz, _dx, _phi = _estimate_shift_2d(
        _a[_z0:_z1, _x0:_x1], _b[_z0:_z1, _x0:_x1],
        dz_g, dx_g, kz_c, force_dz_zero=S2_FORCE_DZ_ZERO)
    results_s2[(_ra, _rb)] = {'dx': _dx, 'dz': _dz, 'phi_0': _phi}
    print(f'[{_ra}->{_rb}]  dx={_dx:+.4f} m  dz={_dz:+.4f} m  phi0={_phi:+.4f} rad')

# ── Summary: direct displacement from profile 1 ───────────────────────────────────
if results_s2:
    from matplotlib.patches import Patch as _Patch
    _PHASE_DEFS = [('Pushing', 1, 3, '#aed6f1'), ('Chasing', 4, 8, '#a9dfbf'),
                   ('Waiting', 9, 20, '#f9e79f'), ('Pulling', 21, 38, '#f1948a')]
    _rn_col = {n: c for _nm, lo, hi, c in _PHASE_DEFS for n in range(lo, hi + 1)}
    _pkeys = list(results_s2.keys())
    dx_v   = [results_s2[k]['dx'] for k in _pkeys]
    dz_v   = [results_s2[k]['dz'] for k in _pkeys]
    phi_v  = [results_s2[k]['phi_0'] for k in _pkeys]
    labels = [f'1->{b}' for _, b in _pkeys]

    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
    for _bax in (ax1, ax2, ax3):
        for _i, _k in enumerate(_pkeys):
            _c = _rn_col.get(_k[1])
            if _c:
                _bax.axvspan(_i - 0.5, _i + 0.5, color=_c, alpha=0.3, zorder=0, lw=0)
    _ph_hdl = [_Patch(facecolor=c, alpha=0.6, label=nm, edgecolor='grey', lw=0.5)
               for nm, lo, hi, c in _PHASE_DEFS if any(_rn_col.get(_k[1]) == c for _k in _pkeys)]
    ax1.plot(dx_v, 'o-', color='steelblue')
    ax1.axhline(0, color='k', lw=0.6, ls='--')
    ax1.set_ylabel('Δx from baseline (radial) [m]')
    ax1.set_title(f'Strategy 2: Fixed Baseline (1→n)  [{S2_METHOD}]  (wrapping risk at large shifts)')
    ax1.legend(handles=_ph_hdl, loc='best', fontsize=8)
    ax2.plot(dz_v, '^-', color='seagreen')
    ax2.axhline(0, color='k', lw=0.6, ls='--')
    ax2.set_ylabel('Δz from baseline (depth) [m]')
    ax3.plot(np.degrees(phi_v), 's-', color='darkorange')
    ax3.axhline(0, color='k', lw=0.6, ls='--')
    ax3.set_ylabel('φ₀ [°]')
    ax3.set_xlabel('Profile')
    ax3.set_xticks(range(len(labels)))
    ax3.set_xticklabels(labels, rotation=45, ha='right', fontsize=6)
    plt.tight_layout()
    fig.savefig(OUT_DIR / 'roi_phase' / 'strat2_baseline_summary.png', dpi=150)
    save_fig(fig, 'strat2_baseline_summary', study='FieldData_Study', prefix='FD_')
    plt.show();  plt.close(fig)


## 3\. Stage-Anchored Tracking ($1 \rightarrow 4$ \[Push\], $5 \rightarrow 9$ \[Chase\], $10 \rightarrow 20$ \[Wait\], etc.)

In [ ]:
# ── Strategy 3: Stage-Anchored Hybrid — RECOMMENDED ─────────────────────────────────
# Reference reset at the start of each hydraulic stage:
#   Pushing  (profs 2–4 vs 1)    — sub-wavelength increments, zero wrapping
#   Chasing  (profs 6–9 vs 5)    — controlled drift, stage-isolated physics
#   Waiting  (profs 11–20 vs 10) — diffusion-dominated; ref reset prevents
#                                   decorrelation bleed-through from push phase
#   Pulling  (profs 22–38 vs 21) — symmetric to push, separate drift budget
# Kinematic chaining: boundary pairs (1→5, 5→10, 10→21) bridge stage offsets
# so all intra-stage tracks are stitched into one global trajectory X_total(t).
# ─────────────────────────────────────────────────────────────────────────────────────

S3_ROI          = (70, 79, 4.5, 7.5)
S3_METHOD       = 'gazdag'
S3_FORCE_DZ_ZERO = False

# Intra-stage pairs (local reference → profile)
_push_pairs  = [(1,  n) for n in range(2, 5)]    # ref=1: profs 2,3,4
_chase_pairs = [(5,  n) for n in range(6, 10)]   # ref=5: profs 6,7,8,9
_wait_pairs  = [(10, n) for n in range(11, 21)]  # ref=10: profs 11–20
_pull_pairs  = [(21, n) for n in range(22, 39)]  # ref=21: profs 22–38
# Inter-stage boundary pairs (bridge stage transitions)
_boundary_pairs = [(1, 5), (5, 10), (10, 21)]

results_s3 = {}
for _ra, _rb in _push_pairs + _chase_pairs + _wait_pairs + _pull_pairs + _boundary_pairs:
    _a = _load_img(S3_METHOD, _ra);  _b = _load_img(S3_METHOD, _rb)
    if _a is None or _b is None:
        print(f'[{_ra}->{_rb}] missing .npy — skipped')
        continue
    _n = min(_a.shape[0], _b.shape[0])
    _a, _b = _a[:_n], _b[:_n]
    _z0, _z1, _x0, _x1 = _phys_to_pix(depth[:_n], x_img, *S3_ROI)
    _dz, _dx, _phi = _estimate_shift_2d(
        _a[_z0:_z1, _x0:_x1], _b[_z0:_z1, _x0:_x1],
        dz_g, dx_g, kz_c, force_dz_zero=S3_FORCE_DZ_ZERO)
    results_s3[(_ra, _rb)] = {'dx': _dx, 'dz': _dz, 'phi_0': _phi}
    tag = ' [boundary]' if (_ra, _rb) in _boundary_pairs else ''
    print(f'[{_ra}->{_rb}]{tag}  dx={_dx:+.4f} m  dz={_dz:+.4f} m  phi0={_phi:+.4f} rad')

# ── Kinematic chaining: add boundary offsets to intra-stage tracks ──────────────
def _bnd(key, comp):
    return results_s3.get(key, {}).get(comp, 0.0)

# Cumulative boundary offsets: push → chase → wait → pull
_off_dx = [0.0,
            _bnd((1, 5), 'dx'),
            _bnd((1, 5), 'dx') + _bnd((5, 10), 'dx'),
            _bnd((1, 5), 'dx') + _bnd((5, 10), 'dx') + _bnd((10, 21), 'dx')]
_off_dz = [0.0,
            _bnd((1, 5), 'dz'),
            _bnd((1, 5), 'dz') + _bnd((5, 10), 'dz'),
            _bnd((1, 5), 'dz') + _bnd((5, 10), 'dz') + _bnd((10, 21), 'dz')]

global_prof, global_dx, global_dz, global_phi, global_col = [], [], [], [], []
from matplotlib.patches import Patch as _Patch
_PHASE_DEFS = [('Pushing', 1, 3, '#aed6f1'), ('Chasing', 4, 8, '#a9dfbf'),
               ('Waiting', 9, 20, '#f9e79f'), ('Pulling', 21, 38, '#f1948a')]
_rn_col = {n: c for _nm, lo, hi, c in _PHASE_DEFS for n in range(lo, hi + 1)}

for _stage_pairs, _odx, _odz in zip(
        [_push_pairs, _chase_pairs, _wait_pairs, _pull_pairs],
        _off_dx, _off_dz):
    for _k in _stage_pairs:
        if _k not in results_s3:
            continue
        global_prof.append(_k[1])
        global_dx.append(_odx + results_s3[_k]['dx'])
        global_dz.append(_odz + results_s3[_k]['dz'])
        global_phi.append(results_s3[_k]['phi_0'])
        global_col.append(_rn_col.get(_k[1], '#cccccc'))

# ── Summary: chained global trajectory ────────────────────────────────────────────
if global_dx:
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
    for _bax in (ax1, ax2, ax3):
        for _i, _c in enumerate(global_col):
            _bax.axvspan(_i - 0.5, _i + 0.5, color=_c, alpha=0.3, zorder=0, lw=0)
        # Dotted vertical lines at stage-boundary transitions
        for _b_prof in (5, 10, 21):
            _bi = [_j for _j, p in enumerate(global_prof) if p == _b_prof]
            if _bi:
                _bax.axvline(_bi[0] - 0.5, color='k', lw=1.0, ls=':', zorder=1)
    _ph_hdl = [_Patch(facecolor=c, alpha=0.6, label=nm, edgecolor='grey', lw=0.5)
               for nm, lo, hi, c in _PHASE_DEFS]
    ax1.plot(global_dx, 'o-', color='steelblue')
    ax1.axhline(0, color='k', lw=0.6, ls='--')
    ax1.set_ylabel('Global Δx (radial) [m]')
    ax1.set_title(f'Strategy 3: Stage-Anchored Hybrid  [{S3_METHOD}]  (chained global trajectory)')
    ax1.legend(handles=_ph_hdl, loc='best', fontsize=8)
    ax2.plot(global_dz, '^-', color='seagreen')
    ax2.axhline(0, color='k', lw=0.6, ls='--')
    ax2.set_ylabel('Global Δz (depth) [m]')
    ax3.plot(np.degrees(global_phi), 's-', color='darkorange')
    ax3.axhline(0, color='k', lw=0.6, ls='--')
    ax3.set_ylabel('φ₀ [°]')
    ax3.set_xlabel('Profile number')
    ax3.set_xticks(range(len(global_prof)))
    ax3.set_xticklabels(global_prof, fontsize=7)
    plt.tight_layout()
    fig.savefig(OUT_DIR / 'roi_phase' / 'strat3_stage_anchored_summary.png', dpi=150)
    save_fig(fig, 'strat3_stage_anchored_summary', study='FieldData_Study', prefix='FD_')
    plt.show();  plt.close(fig)


## 3b\. Stage-Anchored (Coarse) \u2014 Total Stage Displacements

A coarser variant of strategy 3: instead of comparing every profile within a stage
to its stage reference, we take a **single large WLS step per stage** (ref \u2192 last
profile of that stage). This maximises cross-spectrum SNR \u2014 larger \u0394x means a
stronger, less ambiguous phase ramp \u2014 while the stage-boundary reset keeps the
reference waveform close enough to prevent severe decorrelation.

| Step | Pair | Physics |
|---|---|---|
| Push total | 1 \u2192 4 | Full mechanical injection displacement |
| Chase total | 5 \u2192 9 | Net tracer migration during chasing |
| Wait total | 10 \u2192 20 | Net diffusion / dispersion spread |
| Pull total | 21 \u2192 38 | Full withdrawal displacement |

Three **boundary pairs** (1\u21925, 5\u219210, 10\u219221) bridge the stage transitions.
Together these seven estimates reconstruct the **global trajectory** X\u209c\u1d52\u1d57\u2090\u2097(t).


In [ ]:
# ── Strategy 3b: Stage-Anchored (Coarse) — Total Stage Displacements ─────────────
# 4 stage-end pairs + 3 boundary pairs = 7 WLS estimates → global trajectory.
# Pairs are deliberately large (sub-wavelength PER STAGE rather than per profile)
# to boost cross-spectrum SNR while bounding wrapping to the intra-stage max shift.
# ─────────────────────────────────────────────────────────────────────────────────────

# ROI per pair ? stage-end pairs have explicit ROIs;
# boundary pairs inherit the destination stage ROI.
S3B_PAIR_ROIS = {
    (1,  4):  (71, 78, 5.0, 6.5),   # Push total
    (5,  9):  (70, 78, 5.0, 7.0),   # Chase total
    (10, 20): (72, 78, 5.0, 6.5),   # Wait total
    (21, 38): (70, 77, 5.0, 6.0),   # Pull total
    (1,  5):  (70, 78, 5.0, 7.0),   # boundary -> Chase
    (5,  10): (72, 78, 5.0, 6.5),   # boundary -> Wait
    (10, 21): (70, 77, 5.0, 6.0),   # boundary -> Pull
}
S3B_METHOD        = 'gazdag'
S3B_FORCE_DZ_ZERO = False

_stage_end_s3b  = [(1, 4), (5, 9), (10, 20), (21, 38)]
_boundary_s3b   = [(1, 5), (5, 10), (10, 21)]

results_s3b = {}
for _ra, _rb in _stage_end_s3b + _boundary_s3b:
    _a = _load_img(S3B_METHOD, _ra);  _b = _load_img(S3B_METHOD, _rb)
    if _a is None or _b is None:
        print(f'[{_ra}->{_rb}] missing .npy — skipped')
        continue
    _n = min(_a.shape[0], _b.shape[0])
    _a, _b = _a[:_n], _b[:_n]
    _z0, _z1, _x0, _x1 = _phys_to_pix(depth[:_n], x_img, *S3B_PAIR_ROIS[(_ra, _rb)])
    _dz, _dx, _phi = _estimate_shift_2d(
        _a[_z0:_z1, _x0:_x1], _b[_z0:_z1, _x0:_x1],
        dz_g, dx_g, kz_c, force_dz_zero=S3B_FORCE_DZ_ZERO)
    results_s3b[(_ra, _rb)] = {'dx': _dx, 'dz': _dz, 'phi_0': _phi}
    tag = ' [boundary]' if (_ra, _rb) in _boundary_s3b else ' [stage end]'
    print(f'[{_ra}->{_rb}]{tag}  dx={_dx:+.4f} m  dz={_dz:+.4f} m  phi0={_phi:+.4f} rad')

# ── Kinematic chaining: cumulative boundary offsets ───────────────────────────────
def _g(key, comp):
    return results_s3b.get(key, {}).get(comp, 0.0)

_odx = [0.0,
         _g((1,5),'dx'),
         _g((1,5),'dx') + _g((5,10),'dx'),
         _g((1,5),'dx') + _g((5,10),'dx') + _g((10,21),'dx')]
_odz = [0.0,
         _g((1,5),'dz'),
         _g((1,5),'dz') + _g((5,10),'dz'),
         _g((1,5),'dz') + _g((5,10),'dz') + _g((10,21),'dz')]

# 7-point trajectory: 4 stage-end points + 3 boundary points, sorted by profile
_traj = []
for (ref, end), odx, odz in zip(_stage_end_s3b, _odx, _odz):
    _traj.append((end, odx + _g((ref,end),'dx'), odz + _g((ref,end),'dz'), False))
for (ra, rb), odx, odz in zip(_boundary_s3b, _odx[1:], _odz[1:]):
    _traj.append((rb, odx, odz, True))
_traj.sort(key=lambda t: t[0])

print('\nGlobal trajectory (profile | Δx | Δz | type):')
for _p, _dx, _dz, _ib in _traj:
    print(f'  prof {_p:>2}  Δx={_dx:+.4f} m  Δz={_dz:+.4f} m  {"boundary" if _ib else "stage end"}')

# ── Summary plot ──────────────────────────────────────────────────────────────────
if _traj:
    from matplotlib.patches import Patch as _Patch
    import matplotlib.lines as _mlines
    _PHASE_DEFS = [('Pushing', 1, 3, '#aed6f1'), ('Chasing', 4, 8, '#a9dfbf'),
                   ('Waiting', 9, 20, '#f9e79f'), ('Pulling', 21, 38, '#f1948a')]
    _rn_col = {n: c for _nm, lo, hi, c in _PHASE_DEFS for n in range(lo, hi + 1)}

    _profs  = [t[0] for t in _traj]
    _gdx    = [t[1] for t in _traj]
    _gdz    = [t[2] for t in _traj]
    _is_bnd = [t[3] for t in _traj]
    _cols   = [_rn_col.get(p, '#cccccc') for p in _profs]
    _xi_end = [i for i, b in enumerate(_is_bnd) if not b]
    _xi_bnd = [i for i, b in enumerate(_is_bnd) if b]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    for _bax in (ax1, ax2):
        for _i, _c in enumerate(_cols):
            _bax.axvspan(_i - 0.5, _i + 0.5, color=_c, alpha=0.3, zorder=0, lw=0)

    _ph_hdl = [_Patch(facecolor=c, alpha=0.6, label=nm, edgecolor='grey', lw=0.5)
               for nm, lo, hi, c in _PHASE_DEFS]
    _leg_end = _mlines.Line2D([], [], color='steelblue', marker='o', ls='-', label='stage end')
    _leg_bnd = _mlines.Line2D([], [], color='steelblue', marker='D', ls='none', ms=7, label='boundary')

    ax1.plot(_gdx, '-', color='steelblue', lw=1.2, zorder=2)
    ax1.scatter(_xi_end, [_gdx[i] for i in _xi_end], s=60, color='steelblue', marker='o', zorder=3)
    ax1.scatter(_xi_bnd, [_gdx[i] for i in _xi_bnd], s=60, color='steelblue', marker='D', zorder=3)
    ax1.axhline(0, color='k', lw=0.6, ls='--')
    ax1.set_ylabel('Global Δx (radial) [m]')
    ax1.set_title(f'Strategy 3b: Stage-Anchored Coarse  [{S3B_METHOD}]  (7-point global trajectory)')
    ax1.legend(handles=_ph_hdl + [_leg_end, _leg_bnd], loc='best', fontsize=8)

    ax2.plot(_gdz, '-', color='seagreen', lw=1.2, zorder=2)
    ax2.scatter(_xi_end, [_gdz[i] for i in _xi_end], s=60, color='seagreen', marker='o', zorder=3)
    ax2.scatter(_xi_bnd, [_gdz[i] for i in _xi_bnd], s=60, color='seagreen', marker='D', zorder=3)
    ax2.axhline(0, color='k', lw=0.6, ls='--')
    ax2.set_ylabel('Global Δz (depth) [m]')
    ax2.set_xlabel('Profile number')
    ax2.set_xticks(range(len(_profs)))
    ax2.set_xticklabels(_profs)
    plt.tight_layout()
    fig.savefig(OUT_DIR / 'roi_phase' / 'strat3b_coarse_summary.png', dpi=150)
    save_fig(fig, 'strat3b_coarse_summary', study='FieldData_Study', prefix='FD_')
    plt.show();  plt.close(fig)


# Thesis Figure Compilations (Chapter 7 -- Hypothesis 3, field data)

The three figure sets below compile the field-data results for
`07_hypothesis3_complex.typ`. They reuse the migrated `.npy` products
(`OUT_DIR/migrated/`), the back-propagation snapshots (`completed`, built in
the "Save focus-frame snapshots" cell above), and the monogenic-envelope /
WLS phase-plane helpers defined in the "Manual pair selection + ROI
definition" cells above. Run this section after the full notebook has been
executed at least once (so the cached `.npy` / `.vti` products exist) --
none of it re-runs Kirchhoff, Gazdag, or gprMax.

Stage boundaries (Push 1->4, Chase 5->9, Wait 10->20, Pull 21->38) and the
fixed ROI (depth 70-79 m, radial 4.5-7.5 m) match the ones already used in
`sec:hyp3-fd-setup` / `sec:hyp3-fd-phaseplane` of the thesis text.

In [ ]:
# ── Thesis figure compilations: shared setup ────────────────────────────────
ROI_PHYS = (70, 79, 4.5, 7.5)   # fixed ROI, matches sec:hyp3-fd-phaseplane
PHASE_STAGES = [
    ('Pushing', 1, 4),
    ('Chasing', 5, 9),
    ('Waiting', 10, 20),
    ('Pulling', 21, 38),
]

thesis_fig_dir = OUT_DIR / 'thesis_figures'
thesis_fig_dir.mkdir(exist_ok=True)


def _load_processed_Dt(run):
    """Recomputes the processed B-scan (Dt) for one profile (cheap: I/O + filtering)."""
    data, _ = load_mala(str(DATA / _prof_name(run)), return_object=False)
    n = min(data.shape[1], n_traces)
    d_bp = filter_data(data[:, :n], fq=(0.02, 0.2), sfreq=sf, btype='bandpass')
    d_dc, _ = remove_mean(d_bp, 299, 517)
    d_aligned, _, _ = align_traces(d_dc, ref_aligned[:, :n], upsample=5, normalize=True, align_reference=False)
    d_svd, _ = remove_svd(d_aligned, low_s=0, high_s=1)
    d_gain, _ = linear_gain(d_svd, t)
    return d_gain[:rad_cut, :].T   # (n, rad_cut)


def _wls_diag(base_crop, mon_crop, dz_g_, dx_g_, kz_cent, kz_fac, kx_fac,
              wls_thr, pad_fac=10, wls_pow=1):
    """Zero-padded WLS cross-spectrum phase-plane fit; returns fit + everything
    needed to draw the 5-panel diagnostic (mirrors the SHOW_DIAG blocks above)."""
    Nz, Nx = base_crop.shape
    Nz_pad, Nx_pad = Nz * pad_fac, Nx * pad_fac
    kz_ax = np.fft.fftfreq(Nz_pad, d=dz_g_) * 2 * np.pi
    kx_ax = np.fft.fftfreq(Nx_pad, d=dx_g_) * 2 * np.pi
    KZ, KX = np.meshgrid(kz_ax, kx_ax, indexing='ij')
    taper = np.outer(tukey(Nz, alpha=0.15), tukey(Nx, alpha=0.15))
    XS = (np.fft.fft2(base_crop * taper, s=(Nz_pad, Nx_pad)) *
          np.conj(np.fft.fft2(mon_crop * taper, s=(Nz_pad, Nx_pad))))
    w = np.abs(XS)
    phi = np.angle(XS)
    band = (np.abs(KZ) < kz_fac * kz_cent) & (np.abs(KX) < kx_fac * kz_cent)
    mask = (w > wls_thr * w.max()) & band & ((np.abs(KZ) + np.abs(KX)) > 0)
    n_mask = int(mask.sum())
    if n_mask < 3:
        dz_est = dx_est = phi_0 = 0.0
    else:
        W = w[mask] ** wls_pow
        A = np.column_stack([KZ[mask], KX[mask], np.ones(n_mask)])
        c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
        dz_est, dx_est, phi_0 = float(c[0]), float(c[1]), float(c[2])

    fitted = KX * dx_est + KZ * dz_est + phi_0

    phi_shift, w_shift = np.fft.fftshift(phi), np.fft.fftshift(w)
    kz_disp, kx_disp = np.fft.fftshift(kz_ax), np.fft.fftshift(kx_ax)
    band_shift = np.fft.fftshift(band)
    fitted_shift = np.fft.fftshift(fitted)
    phi_masked = np.where(band_shift & (np.fft.fftshift(w) > wls_thr * w.max()), phi_shift, np.nan)
    w_plot = np.where(band_shift, w_shift, np.nan)
    fitted_masked = np.where(band_shift, fitted_shift, np.nan)

    # 1-D kz slice: subtract kx contribution, average over the kx band
    phi_kz_resid = phi - KX * dx_est
    phi_1d = np.zeros(Nz_pad); w_1d = np.zeros(Nz_pad)
    band_kx = np.abs(kx_ax) < kx_fac * kz_cent
    for i in range(Nz_pad):
        sel = band_kx & (w[i, :] > wls_thr * w.max())
        if sel.sum() > 0:
            phi_1d[i] = np.average(phi_kz_resid[i, :][sel], weights=w[i, :][sel])
            w_1d[i] = w[i, :][sel].sum()
    kz_1d_s, phi_1d_s, w_1d_s = np.fft.fftshift(kz_ax), np.fft.fftshift(phi_1d), np.fft.fftshift(w_1d)

    # 1-D kx slice: subtract kz contribution, average over the kz band
    phi_kx_resid = phi - KZ * dz_est
    phi_1d_kx = np.zeros(Nx_pad); w_1d_kx = np.zeros(Nx_pad)
    band_kz_fit = np.abs(kz_ax) < kz_fac * kz_cent
    for j in range(Nx_pad):
        sel = band_kz_fit & (w[:, j] > wls_thr * w.max())
        if sel.sum() > 0:
            phi_1d_kx[j] = np.average(phi_kx_resid[:, j][sel], weights=w[:, j][sel])
            w_1d_kx[j] = w[:, j][sel].sum()
    kx_1d_s, phi_1d_kx_s, w_1d_kx_s = np.fft.fftshift(kx_ax), np.fft.fftshift(phi_1d_kx), np.fft.fftshift(w_1d_kx)

    return dict(
        dz_est=dz_est, dx_est=dx_est, phi_0=phi_0, n_mask=n_mask,
        kz_disp=kz_disp, kx_disp=kx_disp, phi_masked=phi_masked, w_plot=w_plot,
        w_max=w.max(), wls_thr=wls_thr, fitted_masked=fitted_masked,
        kz_1d_s=kz_1d_s, phi_1d_s=phi_1d_s, w_1d_s=w_1d_s,
        kx_1d_s=kx_1d_s, phi_1d_kx_s=phi_1d_kx_s, w_1d_kx_s=w_1d_kx_s,
        kz_fac=kz_fac, kx_fac=kx_fac, kz_cent=kz_cent,
    )


print('Thesis-figure helpers ready.')


## Figure 1: Processed profiles + migrated / back-propagated counterparts

5 rows (profiles 1, 3, 8, 20, 38) x 4 columns (processed B-scan,
Kirchhoff-BP, Gazdag, back-propagation $E_z$ focus frame).

In [ ]:
# ── Figure 1: profile / migration / back-propagation compilation grid ──────
FIG1_PROFILES = [1, 3, 8, 20, 38]
FIG1_DEPTH_RANGE = (62, 86)
FIG1_RADIAL_RANGE = (0, float(x_img[-1]))
FIG1_BP_NEAR_MASK_M = 1.0
FIG1_SC = 1.2

fig1_col_titles = ['Processed B-scan', 'Kirchhoff-BP migration', 'Gazdag migration',
                    'Back-propagation ($E_z$ focus frame)']

fig1, fig1_axes = plt.subplots(len(FIG1_PROFILES), 4, figsize=(17, 3.6 * len(FIG1_PROFILES)),
                                sharex=True, sharey=True)

for row, run in enumerate(FIG1_PROFILES):
    Dt_row = _load_processed_Dt(run)
    n = Dt_row.shape[0]
    z_common = depth[:n]
    lim = max(FIG1_SC * np.max(np.abs(Dt_row)), 1.0)
    ax = fig1_axes[row, 0]
    ax.imshow(Dt_row, aspect='auto', cmap='seismic',
              extent=[x_img[0], x_img[-1], z_common[-1], z_common[0]],
              vmin=-lim, vmax=lim, origin='upper')
    ax.invert_yaxis()
    ax.set_ylabel(f'Profile {run}\nDepth (m)')

    for col, method in [(1, 'kirchhoff_bp'), (2, 'gazdag')]:
        img = _load_img(method, run)
        ax = fig1_axes[row, col]
        if img is None:
            ax.text(0.5, 0.5, 'missing', ha='center', va='center', transform=ax.transAxes)
            continue
        n2 = img.shape[0]
        z2 = depth[:n2]
        lim2 = max(FIG1_SC * np.max(np.abs(img)), 1.0)
        ax.imshow(img, aspect='auto', cmap='seismic',
                  extent=[x_img[0], x_img[-1], z2[-1], z2[0]],
                  vmin=-lim2, vmax=lim2, origin='upper')
        ax.invert_yaxis()

    ax = fig1_axes[row, 3]
    res = _load_bp_frame_wls(run, BP_FOCUS_IDX_OFFSET)
    if res is None:
        ax.text(0.5, 0.5, 'missing', ha='center', va='center', transform=ax.transAxes)
    else:
        ez, depth_axis, radial_axis, dx_m = res
        mask_px = max(1, round(FIG1_BP_NEAR_MASK_M / dx_m))
        ez_disp = ez.copy(); ez_disp[:, :mask_px] = 0.0
        clim = np.percentile(np.abs(ez_disp), 100) if ez_disp.any() else 1.0
        ax.imshow(ez_disp, aspect='auto', cmap='seismic',
                  extent=[0, float(radial_axis[-1]), 0, float(depth_axis[-1])],
                  vmin=-clim, vmax=clim, origin='lower')
        ax.invert_yaxis()

    for col in range(4):
        fig1_axes[row, col].set_ylim(FIG1_DEPTH_RANGE[1], FIG1_DEPTH_RANGE[0])
        fig1_axes[row, col].set_xlim(*FIG1_RADIAL_RANGE)
        fig1_axes[row, col].xaxis.set_major_locator(ticker.MultipleLocator(2))
        fig1_axes[row, col].yaxis.set_major_locator(ticker.MultipleLocator(5))
        if row == len(FIG1_PROFILES) - 1:
            fig1_axes[row, col].set_xlabel('Radial distance (m)')

for col, title in enumerate(fig1_col_titles):
    fig1_axes[0, col].set_title(title)

fig1.suptitle('Processed profiles and their migrated / back-propagated counterparts', y=1.005)
plt.tight_layout()
save_fig(fig1, 'profile_migration_grid', study='FieldData_Study', prefix='FD_', category='Compilations')
plt.show(); plt.close(fig1)


## Figure 2: Per-stage difference + monogenic envelope

One figure per stage (Pushing, Chasing, Waiting, Pulling); each has 3 rows
(Kirchhoff-BP, Gazdag, Back-propagation) x 2 columns (difference + ROI,
monogenic envelope + ROI).

In [ ]:
# ── Figure 2: per-stage difference + monogenic-envelope compilations ───────
FIG2_METHOD_ROWS = [('kirchhoff_bp', 'Kirchhoff-BP'), ('gazdag', 'Gazdag'),
                     ('backprop', 'Back-propagation')]


def _fig2_pair_arrays(method, run_a, run_b):
    if method == 'backprop':
        res_a = _load_bp_frame_wls(run_a, BP_FOCUS_IDX_OFFSET)
        res_b = _load_bp_frame_wls(run_b, BP_FOCUS_IDX_OFFSET)
        if res_a is None or res_b is None:
            return None
        ez_a, depth_axis, radial_axis, dx_m = res_a
        ez_b, _, _, _ = res_b
        nd = min(ez_a.shape[0], ez_b.shape[0]); nr = min(ez_a.shape[1], ez_b.shape[1])
        ez_a, ez_b = ez_a[:nd, :nr], ez_b[:nd, :nr]
        depth_axis, radial_axis = depth_axis[:nd], radial_axis[:nr]
        diff = ez_b - ez_a
        env = _monogenic_envelope(diff)
        d0, d1, r0, r1 = _phys_to_pix(depth_axis, radial_axis, *ROI_PHYS)
        return diff, env, depth_axis, radial_axis, (d0, d1, r0, r1), 'lower'
    else:
        img_a = _load_img(method, run_a); img_b = _load_img(method, run_b)
        if img_a is None or img_b is None:
            return None
        n = min(img_a.shape[0], img_b.shape[0])
        img_a, img_b = img_a[:n], img_b[:n]
        z_common = depth[:n]
        diff = img_b - img_a
        env = _monogenic_envelope(diff)
        z0, z1, x0, x1 = _phys_to_pix(z_common, x_img, *ROI_PHYS)
        return diff, env, z_common, x_img, (z0, z1, x0, x1), 'upper'


def _fig2_plot_stage(stage_name, run_a, run_b):
    fig, axes = plt.subplots(3, 2, figsize=(12, 12), sharex=True, sharey=True)
    z_min, z_max, x_min, x_max = ROI_PHYS

    for row, (method, label) in enumerate(FIG2_METHOD_ROWS):
        result = _fig2_pair_arrays(method, run_a, run_b)
        ax_d, ax_e = axes[row, 0], axes[row, 1]
        if result is None:
            for ax in (ax_d, ax_e):
                ax.text(0.5, 0.5, 'missing data', ha='center', va='center', transform=ax.transAxes)
            continue
        diff, env, z_axis, x_axis, roi_px, origin = result
        z0, z1, x0, x1 = roi_px
        z_roi_lo, z_roi_hi = float(z_axis[z0]), float(z_axis[z1 - 1])
        roi_bottom, roi_h = min(z_roi_lo, z_roi_hi), abs(z_roi_hi - z_roi_lo)
        extent = ([x_axis[0], x_axis[-1], z_axis[-1], z_axis[0]] if origin == 'upper'
                  else [x_axis[0], x_axis[-1], z_axis[0], z_axis[-1]])

        vmax_d = np.percentile(np.abs(diff), 98)
        vmax_e = np.percentile(env, 98)

        im_d = ax_d.imshow(diff, aspect='auto', cmap='RdBu_r', extent=extent,
                            origin=origin, vmin=-vmax_d, vmax=vmax_d)
        plt.colorbar(im_d, ax=ax_d, label='Δ amplitude [a.u.]', fraction=0.046, pad=0.04)
        ax_d.add_patch(Rectangle((x_min, roi_bottom), x_max - x_min, roi_h,
                                  lw=1.5, edgecolor='yellow', facecolor='none'))

        im_e = ax_e.imshow(env, aspect='auto', cmap='inferno', extent=extent,
                            origin=origin, vmin=0, vmax=vmax_e)
        plt.colorbar(im_e, ax=ax_e, label='Monogenic envelope [a.u.]', fraction=0.046, pad=0.04)
        ax_e.add_patch(Rectangle((x_min, roi_bottom), x_max - x_min, roi_h,
                                  lw=1.5, edgecolor='cyan', facecolor='none'))

        for ax in (ax_d, ax_e):
            ax.invert_yaxis()
            ax.set_ylim(86, 62)
            ax.set_xlim(0, float(x_img[-1]))
            ax.xaxis.set_major_locator(ticker.MultipleLocator(2))
            ax.yaxis.set_major_locator(ticker.MultipleLocator(5))
            ax.grid(True, color='grey', lw=0.3, alpha=0.4)
        ax_d.set_ylabel(f'{label}\nDepth (m)')

    axes[0, 0].set_title(f'Difference: prof_{run_b} − prof_{run_a}')
    axes[0, 1].set_title('Monogenic envelope')
    for ax in axes[-1]:
        ax.set_xlabel('Radial distance (m)')

    fig.suptitle(f'{stage_name} stage (profiles {run_a}→{run_b}): difference and monogenic '
                 f'envelope, ROI depth {ROI_PHYS[0]}–{ROI_PHYS[1]} m, radial {ROI_PHYS[2]}–{ROI_PHYS[3]} m', y=1.01)
    plt.tight_layout()
    return fig


for stage_name, run_a, run_b in PHASE_STAGES:
    fig2 = _fig2_plot_stage(stage_name, run_a, run_b)
    save_fig(fig2, f'stage_diff_envelope_{stage_name.lower()}',
             study='FieldData_Study', prefix='FD_', category='Compilations')
    plt.show(); plt.close(fig2)


## Figure 3: Displacement-estimate (WLS phase-plane) diagnostics per method

One figure per migration method (Kirchhoff-BP, Gazdag, Back-propagation);
each has 4 rows (Pushing, Chasing, Waiting, Pulling) x 5 columns
(cross-spectrum phase, cross-spectrum energy + WLS contour, fitted plane,
1-D $k_z$ slice, 1-D $k_x$ slice).

In [ ]:
# ── Figure 3: displacement-estimate (WLS phase-plane) diagnostics per method ──
FIG3_KZ_BAND_FAC = 0.5
FIG3_KX_BAND_FAC = 2.0
FIG3_WLS_AMP_THR = 0.20
FIG3_BP_KX_BAND_FAC = 1.5
FIG3_BP_WLS_AMP_THR = 0.10
FIG3_PAD_FAC = 10


def _fig3_get_crop(method, run_a, run_b):
    if method == 'backprop':
        res_a = _load_bp_frame_wls(run_a, BP_FOCUS_IDX_OFFSET)
        res_b = _load_bp_frame_wls(run_b, BP_FOCUS_IDX_OFFSET)
        if res_a is None or res_b is None:
            return None
        ez_a, depth_axis, radial_axis, dx_m = res_a
        ez_b, _, _, _ = res_b
        nd = min(ez_a.shape[0], ez_b.shape[0]); nr = min(ez_a.shape[1], ez_b.shape[1])
        ez_a, ez_b = ez_a[:nd, :nr], ez_b[:nd, :nr]
        depth_axis, radial_axis = depth_axis[:nd], radial_axis[:nr]
        d0, d1, r0, r1 = _phys_to_pix(depth_axis, radial_axis, *ROI_PHYS)
        return (ez_a[d0:d1, r0:r1], ez_b[d0:d1, r0:r1], dx_m, dx_m, kz_c_bp,
                FIG3_BP_KX_BAND_FAC, FIG3_BP_WLS_AMP_THR)
    else:
        img_a = _load_img(method, run_a); img_b = _load_img(method, run_b)
        if img_a is None or img_b is None:
            return None
        n = min(img_a.shape[0], img_b.shape[0])
        img_a, img_b = img_a[:n], img_b[:n]
        z_common = depth[:n]
        z0, z1, x0, x1 = _phys_to_pix(z_common, x_img, *ROI_PHYS)
        return (img_a[z0:z1, x0:x1], img_b[z0:z1, x0:x1], dz_g, dx_g, kz_c,
                FIG3_KX_BAND_FAC, FIG3_WLS_AMP_THR)


def _fig3_plot_panel_row(axes_row, diag, stage_name, run_a, run_b):
    kz_disp, kx_disp = diag['kz_disp'], diag['kx_disp']
    kz_fac, kx_fac, kz_cent = diag['kz_fac'], diag['kx_fac'], diag['kz_cent']

    ax = axes_row[0]
    im = ax.imshow(diag['phi_masked'], aspect='auto', cmap='RdBu_r',
                    extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]],
                    vmin=-np.pi, vmax=np.pi, origin='upper')
    plt.colorbar(im, ax=ax, label='phase [rad]', fraction=0.046, pad=0.04)
    ax.set_title(f'Cross-spectrum phase ({diag["n_mask"]} px)')
    ax.set_xlim(kx_disp[0] / 3, kx_disp[-1] / 3); ax.set_ylim(kz_disp[-1] / 3, kz_disp[0] / 3)

    ax = axes_row[1]
    im = ax.imshow(diag['w_plot'], aspect='auto', cmap='inferno',
                    extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]], origin='upper')
    plt.colorbar(im, ax=ax, label='|XS| [a.u.]', fraction=0.046, pad=0.04)
    ax.contour(kx_disp, kz_disp, np.nan_to_num(diag['w_plot']),
               levels=[diag['wls_thr'] * diag['w_max']], colors='cyan', linewidths=0.8)
    ax.set_title(f'Energy (thr={diag["wls_thr"]:.2f}×max)')
    ax.set_xlim(kx_disp[0] / 3, kx_disp[-1] / 3); ax.set_ylim(kz_disp[-1] / 3, kz_disp[0] / 3)

    ax = axes_row[2]
    im = ax.imshow(diag['fitted_masked'], aspect='auto', cmap='RdBu_r',
                    extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]],
                    vmin=-np.pi, vmax=np.pi, origin='upper')
    plt.colorbar(im, ax=ax, label='phase [rad]', fraction=0.046, pad=0.04)
    ax.set_title(f'Fitted plane  Δz={diag["dz_est"]:+.3f} m  Δx={diag["dx_est"]:+.3f} m')
    ax.set_xlim(kx_disp[0] / 3, kx_disp[-1] / 3); ax.set_ylim(kz_disp[-1] / 3, kz_disp[0] / 3)

    ax = axes_row[3]
    kz_1d_s, phi_1d_s, w_1d_s = diag['kz_1d_s'], diag['phi_1d_s'], diag['w_1d_s']
    in_band = np.abs(kz_1d_s) < kz_fac * kz_cent
    sel = in_band & (w_1d_s > 0)
    sc = ax.scatter(kz_1d_s[sel], phi_1d_s[sel], c=w_1d_s[sel], cmap='viridis', s=14)
    plt.colorbar(sc, ax=ax, label='weight', fraction=0.046, pad=0.04)
    kz_fit = kz_1d_s[in_band]
    ax.plot(kz_fit, kz_fit * diag['dz_est'] + diag['phi_0'], 'r-', lw=1.3)
    ax.axhline(0, color='k', lw=0.5, ls='--')
    ax.set_title('1-D $k_z$ slice: measured vs fitted')
    ax.set_xlabel('$k_z$ [rad/m]'); ax.set_ylabel('phase [rad]')

    ax = axes_row[4]
    kx_1d_s, phi_1d_kx_s, w_1d_kx_s = diag['kx_1d_s'], diag['phi_1d_kx_s'], diag['w_1d_kx_s']
    in_band_kx = np.abs(kx_1d_s) < kx_fac * kz_cent
    sel = in_band_kx & (w_1d_kx_s > 0)
    sc = ax.scatter(kx_1d_s[sel], phi_1d_kx_s[sel], c=w_1d_kx_s[sel], cmap='viridis', s=14)
    plt.colorbar(sc, ax=ax, label='weight', fraction=0.046, pad=0.04)
    kx_fit = kx_1d_s[in_band_kx]
    ax.plot(kx_fit, kx_fit * diag['dx_est'] + diag['phi_0'], 'r-', lw=1.3)
    ax.axhline(0, color='k', lw=0.5, ls='--')
    ax.set_title('1-D $k_x$ slice: measured vs fitted')
    ax.set_xlabel('$k_x$ [rad/m]'); ax.set_ylabel('phase [rad]')

    axes_row[0].set_ylabel(f'{stage_name}\n(prof {run_a}$\\to${run_b})\n$k_z$ [rad/m]')


def _fig3_plot_method(method, label):
    fig, axes = plt.subplots(4, 5, figsize=(24, 16))
    for row, (stage_name, run_a, run_b) in enumerate(PHASE_STAGES):
        crop = _fig3_get_crop(method, run_a, run_b)
        if crop is None:
            for ax in axes[row]:
                ax.text(0.5, 0.5, 'missing data', ha='center', va='center', transform=ax.transAxes)
            continue
        base_crop, mon_crop, dz_gv, dx_gv, kz_cent, kx_fac, wls_thr = crop
        diag = _wls_diag(base_crop, mon_crop, dz_gv, dx_gv, kz_cent,
                          FIG3_KZ_BAND_FAC, kx_fac, wls_thr, pad_fac=FIG3_PAD_FAC)
        _fig3_plot_panel_row(axes[row], diag, stage_name, run_a, run_b)
        print(f'  [{label}] {stage_name} ({run_a}->{run_b}): '
              f'dz={diag["dz_est"]:+.4f} m  dx={diag["dx_est"]:+.4f} m  n_mask={diag["n_mask"]}')

    fig.suptitle(f'{label}: WLS cross-spectrum phase-plane diagnostics per stage '
                 f'(ROI depth {ROI_PHYS[0]}-{ROI_PHYS[1]} m, radial {ROI_PHYS[2]}-{ROI_PHYS[3]} m)', y=1.01)
    plt.tight_layout()
    return fig


for method, label in [('kirchhoff_bp', 'Kirchhoff-BP'), ('gazdag', 'Gazdag'),
                       ('backprop', 'Back-propagation')]:
    print(f'--- {label} ---')
    fig3 = _fig3_plot_method(method, label)
    save_fig(fig3, f'displacement_diagnostics_{method}',
             study='FieldData_Study', prefix='FD_', category='Compilations')
    plt.show(); plt.close(fig3)
